# tieuluan2 · RNN và LSTM

Notebook là nơi đọc và sửa toàn bộ implementation dưới đây: dữ liệu, mô hình, loss, optimizer, train/đánh giá. Không cần mở file `.py` để xem thuật toán. Các bản `.py` cũ giữ nguyên để bảo toàn backend/artifact.

**Cách đọc:** khởi tạo → code từng hàm/lớp → thực hành theo dataset → kết quả. `%%inline_module` chỉ chạy code có ngay trong cell ở namespace riêng, không nạp implementation từ `.py`. Mỗi framework dùng kernel WSL riêng; code của cả hai vẫn hiện trong notebook. Mặc định không train/TEST lại.

In [ ]:
PROJECT_NAME = 'tieuluan2'
from pathlib import Path
import sys, os, types, __future__
from IPython.core.magic import register_cell_magic
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "ASSIGN_REORGANIZATION_EXECUTION_PLAN.md").exists())
sys.path.insert(0, str(REPO / "_shared"))
from assignment_runtime import activate
ACTIVE_FRAMEWORK = "keras" if ".venv-keras" in sys.executable else "torch"
_context = activate(PROJECT_NAME, ACTIVE_FRAMEWORK)
LOGICAL_ROOT = Path(_context["logical_root"])
RUN_EXPERIMENTS = False
RUN_TRAINING = False
RUN_TEST = False
# Kết quả của lượt chạy mới nằm ngoài notebooks/, không ghi vào artifact gốc.
import assignment_runtime as _paths
_paths._WRITE_CONTEXT["output"] = REPO / PROJECT_NAME / "artifacts/notebook_runs/inline_v1"
_paths._WRITE_CONTEXT["overlay"] = False
INLINE_MODULES = {}

def _notebook_package(name):
    if not name:
        return None
    if name not in INLINE_MODULES:
        parent, _, child = name.rpartition(".")
        package = _notebook_package(parent)
        module = types.ModuleType(name)
        module.__package__ = parent
        module.__path__ = []  # Không tìm implementation khác trong file .py.
        module.__file__ = str(LOGICAL_ROOT / (name.replace(".", "/") + "/__init__.py"))
        module.__dict__.update(ACTIVE_FRAMEWORK=ACTIVE_FRAMEWORK, RUN_TRAINING=False, RUN_TEST=False)
        INLINE_MODULES[name] = module
        sys.modules[name] = module
        if package is not None:
            setattr(package, child, module)
    return INLINE_MODULES[name]

@register_cell_magic
def inline_module(line, cell):
    """Chạy đúng code trong cell ở namespace riêng; không đọc code từ .py."""
    name, relative, scope = line.split()
    if scope != "both" and scope != ACTIVE_FRAMEWORK:
        return
    parent, _, child = name.rpartition(".")
    _notebook_package(parent)
    module = _notebook_package(name)
    module.__file__ = str(LOGICAL_ROOT / relative)
    module.__package__ = name if relative.endswith("/__init__.py") else parent
    compiled = compile(cell, "<notebook>/" + name, "exec", flags=__future__.annotations.compiler_flag, dont_inherit=True)
    exec(compiled, module.__dict__)
    module.__notebook_source__ = True

print({"bài": PROJECT_NAME, "kernel": ACTIVE_FRAMEWORK, "train": RUN_TRAINING, "TEST": RUN_TEST})


### `src.common.contracts`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
"""Offline Phase 22 specification guards, not a training or TEST executor.

jsonschema/referencing are already pinned by the shared uv lock. No network fetch
is registered: unknown external schema URIs fail closed.
"""

from __future__ import annotations

import csv

import hashlib

import json

import math

from datetime import datetime

from functools import lru_cache

from itertools import pairwise

from pathlib import Path

from jsonschema import Draft202012Validator, FormatChecker

from referencing import Registry, Resource

ROOT = Path(__file__).resolve().parents[2]

SCHEMAS = ROOT / "src/common/schemas/v3"

PROTOCOL = ROOT / "experiments/configs/v3/protocol_v3.json"

REQUIREMENT_IDS = {f"REQ-{number:02}" for number in range(1, 55)}



#### ContractError

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
class ContractError(ValueError):
    """A structural or semantic specification violation."""



#### read_json

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8-sig"))



#### canonical_sha256

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def canonical_sha256(value) -> str:
    encoded = json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(encoded.encode("utf-8")).hexdigest()



#### load_schemas

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
@lru_cache(maxsize=1)
def load_schemas() -> dict:
    schemas = {
        path.name.removesuffix(".schema.json"): read_json(path) for path in SCHEMAS.glob("*.schema.json")
    }
    for schema in schemas.values():
        Draft202012Validator.check_schema(schema)
    return schemas



#### _finite

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _finite(value):
    if isinstance(value, float) and not math.isfinite(value):
        raise ContractError("All numeric values must be finite; undefined metrics use null with a reason")
    if isinstance(value, dict):
        for child in value.values():
            _finite(child)
    elif isinstance(value, list):
        for child in value:
            _finite(child)



#### _backend

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _backend(document):
    allowed = {"scratch": {"numpy"}, "keras": {"tensorflow", "torch"}, "pytorch": {"torch"}}
    if document["backend"] not in allowed[document["implementation"]]:
        raise ContractError("implementation/backend mismatch")



#### _dataset_family

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _dataset_family(document):
    protocol = read_json(PROTOCOL)
    dataset = next((row for row in protocol["datasets"] if row["dataset_id"] == document["dataset_id"]), None)
    if dataset is None or dataset["chapter"] != protocol["families"][document["family"]]["chapter"]:
        raise ContractError("Model family/dataset mismatch")



#### _time

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _time(value):
    return datetime.fromisoformat(value)



#### _model

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _model(document):
    _backend(document)
    _dataset_family(document)
    policy = read_json(PROTOCOL)
    layers = document["layers"]
    eligible = [
        layer
        for layer in layers
        if layer["kind"] in policy["layer_policy"]["counted_kinds"] and layer["parameter_count"] > 0
    ]
    expected = policy["families"][document["family"]]
    if len(eligible) < expected["minimum_learnable_layers"]:
        raise ContractError("Too few learnable layers; pooling/normalization/time steps do not count")
    if [layer["kind"] for layer in eligible] != expected["blocks"]:
        raise ContractError("Learnable layer composition differs from the locked reference architecture")
    if sum(layer["parameter_count"] for layer in layers) != document["parameter_count"]:
        raise ContractError("Total parameter count does not match layers")
    functions = {function["id"]: function for function in document["functions"]}
    if len(functions) != len(document["functions"]) or len({layer["id"] for layer in layers}) != len(layers):
        raise ContractError("Duplicate function/layer IDs")
    if any(name not in functions for name in document["composition"]):
        raise ContractError("Composition references a missing function")
    for layer in layers:
        if layer["function_id"] not in functions:
            raise ContractError("Layer references a missing function")
    if document["family"] in {"rnn", "lstm"} and not document["recurrent_unroll"]:
        raise ContractError("Recurrent model needs a time/layer unroll explanation")



#### _split

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _split(document):
    partitions = document["partitions"]
    if {part["name"] for part in partitions} != {"train", "validation", "test"}:
        raise ContractError("Each partition must occur exactly once")
    if document["method"] == "temporal":
        for part in partitions:
            if (
                not part["target_start"]
                or not part["target_end"]
                or _time(part["target_start"]) > _time(part["target_end"])
            ):
                raise ContractError("Temporal partition needs a valid target time range")
        by_name = {part["name"]: part for part in partitions}
        times = [
            (_time(by_name[name]["target_start"]), _time(by_name[name]["target_end"]))
            for name in ("train", "validation", "test")
        ]
        if any(left[1] >= right[0] for left, right in pairwise(times)):
            raise ContractError("Temporal target time ranges overlap")



#### _run

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _run(document):
    _backend(document)
    _dataset_family(document)
    if document["track"] == "matched_subset" and not document["subset_sha256"]:
        raise ContractError("Matched-subset run needs its selected train ID hash")
    if document["split"] == "test" and not document["test_seal_id"]:
        raise ContractError("TEST metrics require a test seal ID")
    if _time(document["started_at"]) > _time(document["finished_at"]):
        raise ContractError("Run finish precedes start")
    efficiency = document["efficiency"]
    if efficiency["inference_p95_ms"] < efficiency["inference_p50_ms"]:
        raise ContractError("Inference p95 cannot be below p50")
    if document["device"] == "cpu" and efficiency["peak_vram_bytes"] != 0:
        raise ContractError("CPU run cannot claim GPU allocator memory")
    for metric in document["metrics"]:
        if (metric["value"] is None) != bool(metric["undefined_reason"]):
            raise ContractError("Undefined metric must use null and an explicit reason")



#### _dataset

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _dataset(document):
    if document["status"] == "ready":
        if document["source"]["license_status"] != "approved":
            raise ContractError("Ready dataset requires approved license/access")
        if document["logical_size"]["records"] is None:
            raise ContractError("Ready dataset requires measured logical size")
        if document["kind"] != "task_view" and not document["files"]:
            raise ContractError("Ready dataset needs file hashes")
    if document["kind"] == "task_view" and not document["parent_dataset_ids"]:
        raise ContractError("Task view must reference source datasets")



#### _seal

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _seal(document):
    expected = 1 if document["state"] in {"evaluating", "evaluated", "failed_sealed"} else 0
    if document["evaluation_count"] != expected:
        raise ContractError("Seal state/evaluation count mismatch")
    if len(set(document["approved_run_ids"])) != len(document["approved_run_ids"]):
        raise ContractError("Seal approved run IDs must be unique")



#### _artifact

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _artifact(document):
    loaders = {
        "npz": "numpy_no_pickle",
        "keras": "keras_safe_mode",
        "torch_state_dict": "torch_weights_only",
        "legacy_trusted_joblib": "legacy_allowlist",
    }
    expected = loaders.get(document["serialization"], "data_only")
    if document["load_policy"] != expected:
        raise ContractError("Artifact serialization/load policy mismatch")
    if document["kind"] == "checkpoint" and not document["cpu_loadable"]:
        raise ContractError("Deployable checkpoint must support CPU-safe load")



#### _api

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _api(document):
    datasets = {row["dataset_id"]: row for row in read_json(PROTOCOL)["datasets"]}
    dataset = datasets.get(document["dataset_id"])
    if not dataset or dataset["chapter"] != document["chapter"]:
        raise ContractError("Request dataset/chapter mismatch")
    family = document["family"]
    chapter_families = {2: {"mlp"}, 3: {"cnn"}, 4: {"rnn", "lstm"}, 5: {"kg_rag"}, 6: {"multi_agent"}}
    if family not in chapter_families[document["chapter"]]:
        raise ContractError("Request family/chapter mismatch")
    expected = (
        ["B0", "B1", "B2", "B3", "B4"]
        if family == "kg_rag"
        else ["A0", "A1", "A2"]
        if family == "multi_agent"
        else ["scratch", "keras", "pytorch"]
    )
    if document["implementations"] != expected:
        raise ContractError("Request implementations do not match family")
    kind = document["input"]["kind"]
    if kind != {2: "tabular", 3: "image", 4: "sequence", 5: "question", 6: "question"}[document["chapter"]]:
        raise ContractError("Request payload kind does not match chapter")
    if kind == "sequence" and len({len(row) for row in document["input"]["values"]}) != 1:
        raise ContractError("Sequence rows must have a consistent feature width")



#### _job

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _job(document):
    results = document["results"]
    expected = (
        {"B0", "B1", "B2", "B3", "B4"}
        if document["family"] == "kg_rag"
        else {"A0", "A1", "A2"}
        if document["family"] == "multi_agent"
        else {"scratch", "keras", "pytorch"}
    )
    names = [result["implementation"] for result in results]
    if len(names) != len(set(names)) or any(name not in expected for name in names):
        raise ContractError("Duplicate or inapplicable result slot")
    if document["status"] in {"succeeded", "partial_failed", "failed"} and set(names) != expected:
        raise ContractError("Terminal job must report every implementation, including unavailable slots")
    all_success = len(results) == len(expected) and all(result["status"] == "succeeded" for result in results)
    if any(result["input_sha256"] != document["input_sha256"] for result in results):
        raise ContractError("Job result input hash differs from the shared input hash")
    groups = {result["comparison_group_sha256"] for result in results}
    if document["comparison_valid"] and (len(groups) != 1 or None in groups):
        raise ContractError("Models do not belong to the same comparison group")
    if document["comparison_valid"] and (not all_success or document["comparison_blockers"]):
        raise ContractError("Incomplete or blocked comparison cannot be valid")
    for result in results:
        if result["status"] == "succeeded" and (
            result["prediction"] is None
            or result["error"] is not None
            or result["actual_device"] == "not_run"
            or not result["artifact_sha256"]
        ):
            raise ContractError("Successful result needs prediction, actual device and artifact hash")
        if result["status"] != "succeeded" and (result["error"] is None or result["prediction"] is not None):
            raise ContractError("Failed result needs explicit error, not a fabricated prediction")
        prediction = result["prediction"]
        if (
            prediction
            and prediction["kind"] == "classification"
            and not math.isclose(sum(row["value"] for row in prediction["probabilities"]), 1.0, abs_tol=1e-5)
        ):
            raise ContractError("Classification probabilities must sum to one")



#### _agent

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _agent(document):
    seen = set()
    agents = set()
    for node in document["nodes"]:
        if node["node_id"] in seen or not set(node["parent_ids"]).issubset(seen):
            raise ContractError("Agent trace must be acyclic and topologically ordered")
        seen.add(node["node_id"])
        agents.add(node["agent"])
    required = {
        "A0": {"direct"},
        "A1": {"router", "specialist"},
        "A2": {"router", "specialist", "verifier", "reporter"},
    }
    if not agents.issubset(required[document["variant"]]):
        raise ContractError("Agent trace uses a role outside its benchmark variant")
    if document["status"] == "succeeded" and not required[document["variant"]].issubset(agents):
        raise ContractError("Agent trace omits required roles")



#### _agent_task

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _agent_task(document):
    dataset = next(
        (row for row in read_json(PROTOCOL)["datasets"] if row["dataset_id"] == document["dataset_id"]), None
    )
    if not dataset or dataset["chapter"] != 6:
        raise ContractError("Agent task must use a chapter6 task-view dataset")



#### validate_document

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def validate_document(name: str, document: dict, *, publish: bool = False) -> None:
    schema_name, _, definition = name.partition("#")
    schemas = load_schemas()
    if schema_name not in schemas or (definition and definition not in schemas[schema_name].get("$defs", {})):
        raise ContractError(f"Unknown schema: {name}")
    _finite(document)
    registry = Registry().with_resources(
        (schema["$id"], Resource.from_contents(schema)) for schema in schemas.values()
    )
    schema = (
        {"$ref": schemas[schema_name]["$id"] + f"#/$defs/{definition}"}
        if definition
        else schemas[schema_name]
    )
    validator = Draft202012Validator(schema, registry=registry, format_checker=FormatChecker())
    errors = sorted(validator.iter_errors(document), key=lambda error: error.json_path)
    if errors:
        raise ContractError("; ".join(f"{error.json_path}: {error.message}" for error in errors[:5]))
    if publish and document.get("evidence_kind") == "synthetic_fixture":
        raise ContractError("Cannot publish synthetic fixture as measured evidence")
    checks = {
        "dataset": _dataset,
        "split": _split,
        "model": _model,
        "run": _run,
        "seal": _seal,
        "artifact": _artifact,
        "api": _api,
        "api#job": _job,
        "api#agent_task": _agent_task,
        "agent": _agent,
    }
    if name in checks:
        checks[name](document)



#### check_seal_transition

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def check_seal_transition(seal: dict, target: str, *, config_sha256: str) -> None:
    """Validate a proposed transition; do not change files or authorize TEST execution."""
    validate_document("seal", seal)
    if seal["state"] in {"evaluated", "failed_sealed"}:
        raise ContractError("TEST is sealed; do not reopen it")
    if config_sha256 != seal["locked_config_sha256"]:
        raise ContractError("TEST config hash changed")
    allowed = {"draft": {"locked"}, "locked": {"evaluating"}, "evaluating": {"evaluated", "failed_sealed"}}
    if target not in allowed.get(seal["state"], set()):
        raise ContractError("Invalid TEST seal transition")



#### _refs

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def _refs(value):
    if isinstance(value, dict):
        if "$ref" in value:
            yield value["$ref"]
        for child in value.values():
            yield from _refs(child)
    elif isinstance(value, list):
        for child in value:
            yield from _refs(child)



#### audit_specification

In [ ]:
%%inline_module src.common.contracts src/common/contracts.py both
def audit_specification(root: Path) -> dict:
    """Audit frozen design documents; never read dataset contents or execute training."""
    errors = []
    result = {"errors": errors, "requirements": 0, "datasets": 0}
    required = [
        "docs/srs_v3.md",
        "docs/traceability_v3.csv",
        "docs/report_style_v3.md",
        "docs/architecture_v3.mmd",
        "docs/risk_register_v3.csv",
        "docs/adr/0001-v3-sources-and-runtime.md",
        "docs/adr/0002-v3-comparison-and-seals.md",
        "docs/adr/0003-v3-api-and-serving.md",
        "docs/adr/0004-v3-formula-minimal-report.md",
        "experiments/configs/v3/protocol_v3.json",
        "report/outline_v3.json",
        "docs/contracts/v3/openapi.json",
    ]
    for relative in required:
        if not (root / relative).is_file():
            errors.append(f"Missing required specification: {relative}")
    if errors:
        return result
    with (root / "docs/traceability_v3.csv").open(encoding="utf-8-sig", newline="") as handle:
        rows = list(csv.DictReader(handle))
    result["requirements"] = len(rows)
    if {row["requirement_id"] for row in rows} != REQUIREMENT_IDS or len(rows) != len(REQUIREMENT_IDS):
        errors.append("Requirement IDs are missing, duplicated or unknown")
    for row in rows:
        if not all(
            row.get(key)
            for key in (
                "origin",
                "requirement",
                "phases",
                "spec_artifact",
                "planned_artifact",
                "acceptance_test",
                "implementation_status",
            )
        ):
            errors.append(f"Incomplete traceability: {row.get('requirement_id')}")
        if not (root / row["spec_artifact"]).is_file():
            errors.append(f"Missing specification artifact for {row['requirement_id']}")
        if any(not token.isdigit() or not 20 <= int(token) <= 35 for token in row["phases"].split(";")):
            errors.append(f"Invalid phase mapping for {row['requirement_id']}")
    protocol = read_json(root / "experiments/configs/v3/protocol_v3.json")
    outline = read_json(root / "report/outline_v3.json")
    datasets = protocol["datasets"]
    result["datasets"] = len(datasets)
    if len(datasets) != 18 or len({row["dataset_id"] for row in datasets}) != 18:
        errors.append("Expected 18 unique dataset/benchmark/view IDs")
    if [row["chapter"] for row in outline["chapters"]] != list(range(1, 7)):
        errors.append("Outline must contain exactly six chapters")
    for chapter in outline["chapters"]:
        expected = {row["dataset_id"] for row in datasets if row["chapter"] == chapter["chapter"]}
        if len(expected) != 3 or set(chapter["dataset_ids"]) != expected:
            errors.append(f"Dataset/outline mismatch: chapter {chapter['chapter']}")
        deploy = [section for section in chapter["sections"] if section["kind"] == "deployment"]
        if chapter["chapter"] >= 2 and (len(deploy) != 1 or deploy[0] != chapter["sections"][-1]):
            errors.append("Deployment must be the last short section, never its own chapter")
    if protocol["ch5"]["new_test_evaluation_allowed"] or protocol["ch5"]["test_run_count"] != 1:
        errors.append("Ch5 TEST must remain sealed at run_count=1")
    if protocol["comparison"]["training_runs_required"] != 4 * 3 * 3 * len(protocol["seeds"]["stability"]):
        errors.append("Neural run matrix count mismatch")
    api_path = root / "docs/contracts/v3/openapi.json"
    api = read_json(api_path)
    if api.get("openapi") != "3.1.1":
        errors.append("OpenAPI version must be explicit")
    for reference in _refs(api):
        relative, _, fragment = reference.partition("#")
        target = (api_path.parent / relative).resolve() if relative else api_path.resolve()
        try:
            target.relative_to(root.resolve())
            content = read_json(target)
            for token in filter(None, fragment.split("/")):
                content = content[token.replace("~1", "/").replace("~0", "~")]
        except (ValueError, KeyError, OSError) as error:
            errors.append(f"Unresolved local API reference {reference}: {error}")
    return result



### `src.common.artifact_core`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.common.artifact_core src/common/artifact_core.py both
"""Data-only, hash-verified checkpoints and complete provenance (no pickle)."""

from __future__ import annotations

import hashlib

import io

import json

import re

import zipfile

from pathlib import Path

import numpy as np

from src.common.contracts import canonical_sha256



#### sha256_file

In [ ]:
%%inline_module src.common.artifact_core src/common/artifact_core.py both
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()



#### write_once

In [ ]:
%%inline_module src.common.artifact_core src/common/artifact_core.py both
def write_once(path, document):
    path = Path(path)
    encoded = (
        json.dumps(document, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False) + "\n"
    ).encode()
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with path.open("xb") as stream:
            stream.write(encoded)
    except FileExistsError:
        if path.read_bytes() != encoded:
            raise ValueError(f"Immutable artifact differs: {path.name}") from None
    return {"path": path, "sha256": hashlib.sha256(encoded).hexdigest()}



#### ArtifactStore

In [ ]:
%%inline_module src.common.artifact_core src/common/artifact_core.py both
class ArtifactStore:
    PROVENANCE_KEYS = (
        "dataset_sha256",
        "split_sha256",
        "subset_sha256",
        "preprocessing_sha256",
        "adapter_sha256",
        "config_sha256",
        "code_sha256",
        "environment_sha256",
        "model_contract_sha256",
    )

    def __init__(self, root, max_bytes=128 * 1024**2):
        self.root = Path(root).resolve()
        self.root.mkdir(parents=True, exist_ok=True)
        self.max_bytes = max_bytes

    def _path(self, name, suffix):
        if not re.fullmatch(r"[a-z][a-z0-9_-]{0,95}", name):
            raise ValueError("Invalid checkpoint ID")
        path = (self.root / (name + suffix)).resolve()
        if path.parent != self.root:
            raise ValueError("Checkpoint path escapes root")
        return path

    def _provenance(self, provenance):
        if set(provenance) != set(self.PROVENANCE_KEYS) or any(
            not isinstance(v, str) or not re.fullmatch(r"[a-f0-9]{64}", v) for v in provenance.values()
        ):
            raise ValueError("Missing/invalid artifact provenance")

    def save_checkpoint(self, name, arrays, provenance):
        self._provenance(provenance)
        path = self._path(name, ".npz")
        metadata_path = self._path(name, ".json")
        if path.exists() or metadata_path.exists():
            raise ValueError("Checkpoint already exists")
        if not arrays or any(
            not re.fullmatch(r"[a-z][a-z0-9_/-]{0,127}", k)
            or v.dtype.kind not in "fiu"
            or not np.isfinite(v).all()
            for k, v in arrays.items()
        ):
            raise ValueError("Only finite named numeric arrays may be saved")
        if sum(v.nbytes for v in arrays.values()) > self.max_bytes:
            raise ValueError("Checkpoint size budget exceeded")
        buffer = io.BytesIO()
        np.savez(buffer, **arrays)
        payload = buffer.getvalue()
        if len(payload) > self.max_bytes:
            raise ValueError("Serialized size budget exceeded")
        metadata = {
            "checkpoint_version": "phase25.1",
            "serialization": "npz",
            "load_policy": "numpy_no_pickle",
            "cpu_loadable": True,
            "sha256": hashlib.sha256(payload).hexdigest(),
            "size_bytes": len(payload),
            "provenance": dict(provenance),
            "arrays": {k: {"shape": list(v.shape), "dtype": str(v.dtype)} for k, v in arrays.items()},
        }
        with path.open("xb") as stream:
            stream.write(payload)
        write_once(metadata_path, metadata)
        return metadata

    def load_checkpoint(self, name, expected_provenance, *, expected_manifest_sha256=None):
        self._provenance(expected_provenance)
        if (
            expected_manifest_sha256 is not None
            and sha256_file(self._path(name, ".json")) != expected_manifest_sha256
        ):
            raise ValueError("Checkpoint manifest hash mismatch")
        metadata = json.loads(self._path(name, ".json").read_text(encoding="utf-8"))
        path = self._path(name, ".npz")
        if (
            metadata["provenance"] != expected_provenance
            or metadata.get("checkpoint_version") != "phase25.1"
            or metadata.get("load_policy") != "numpy_no_pickle"
        ):
            raise ValueError("Checkpoint provenance/load policy mismatch")
        if (
            path.stat().st_size != metadata["size_bytes"]
            or path.stat().st_size > self.max_bytes
            or sha256_file(path) != metadata["sha256"]
        ):
            raise ValueError("Checkpoint integrity mismatch")
        payload = path.read_bytes()
        if hashlib.sha256(payload).hexdigest() != metadata["sha256"]:
            raise ValueError("Checkpoint changed while loading")
        with zipfile.ZipFile(io.BytesIO(payload)) as packed:
            if sum(item.file_size for item in packed.infolist()) > self.max_bytes:
                raise ValueError("Expanded checkpoint exceeds size budget")
        with np.load(io.BytesIO(payload), allow_pickle=False) as archive:
            arrays = {k: archive[k] for k in archive.files}
        if set(arrays) != set(metadata["arrays"]) or any(
            list(v.shape) != metadata["arrays"][k]["shape"]
            or str(v.dtype) != metadata["arrays"][k]["dtype"]
            or v.dtype.kind not in "fiu"
            or not np.isfinite(v).all()
            for k, v in arrays.items()
        ):
            raise ValueError("Checkpoint tensor metadata mismatch")
        return arrays



#### code_fingerprint

In [ ]:
%%inline_module src.common.artifact_core src/common/artifact_core.py both
def code_fingerprint(root, paths):
    root = Path(root)
    return canonical_sha256({str(path): sha256_file(root / path) for path in sorted(paths)})



### `src.common.acquisition`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.common.acquisition src/common/acquisition.py both
"""Bounded HTTPS acquisition and archive hashing; never extracts or unpickles data."""

from __future__ import annotations

import hashlib

import os

import shutil

import stat

import tarfile

import uuid

import zipfile

from pathlib import Path, PurePosixPath

from urllib.parse import urljoin, urlsplit

import requests

RESERVE_BYTES = 5 * 1024**3



#### AcquisitionError

In [ ]:
%%inline_module src.common.acquisition src/common/acquisition.py both
class AcquisitionError(ValueError):
    """Untrusted input, failed integrity check, or insufficient storage."""



#### file_digest

In [ ]:
%%inline_module src.common.acquisition src/common/acquisition.py both
def file_digest(path, algorithm="sha256"):
    digest = hashlib.new(algorithm)
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024**2), b""):
            digest.update(chunk)
    return digest.hexdigest()



#### safe_member_name

In [ ]:
%%inline_module src.common.acquisition src/common/acquisition.py both
def safe_member_name(name):
    path = PurePosixPath(name)
    if not name or "\\" in name or "\x00" in name or ":" in name or path.is_absolute() or ".." in path.parts:
        raise AcquisitionError(f"Unsafe archive member: {name!r}")
    return path.as_posix()



#### storage_required

In [ ]:
%%inline_module src.common.acquisition src/common/acquisition.py both
def storage_required(download_bytes, expanded_bytes, reserve_bytes=RESERVE_BYTES):
    if min(download_bytes, expanded_bytes, reserve_bytes) < 0:
        raise ValueError("Storage estimates must be nonnegative")
    return download_bytes + expanded_bytes + reserve_bytes



#### _url_allowed

In [ ]:
%%inline_module src.common.acquisition src/common/acquisition.py both
def _url_allowed(url, allowed_hosts):
    parsed = urlsplit(url)
    if (
        parsed.scheme != "https"
        or parsed.hostname not in allowed_hosts
        or parsed.username
        or parsed.password
        or parsed.port not in (None, 443)
    ):
        raise AcquisitionError("Only credential-free HTTPS on approved source hosts is allowed")



#### download_file

In [ ]:
%%inline_module src.common.acquisition src/common/acquisition.py both
def download_file(
    url,
    path,
    *,
    max_bytes,
    allowed_hosts,
    expected_sha256=None,
    expected_md5=None,
    opener=None,
    reserve_bytes=RESERVE_BYTES,
):
    """Never overwrites an existing file; trusted hashes make repeat calls offline."""
    _url_allowed(url, allowed_hosts)
    path = Path(path)
    if path.is_symlink():
        raise AcquisitionError("Download destination cannot be a symlink")
    if path.exists():
        if not expected_sha256 or file_digest(path) != expected_sha256:
            raise AcquisitionError(f"Existing file lacks a matching trusted SHA-256: {path}")
        return {"action": "reused", "sha256": expected_sha256, "size_bytes": path.stat().st_size}
    if max_bytes <= 0:
        raise AcquisitionError("Download must have a positive size bound")
    path.parent.mkdir(parents=True, exist_ok=True)
    if shutil.disk_usage(path.parent).free < storage_required(max_bytes, 0, reserve_bytes):
        raise AcquisitionError("Insufficient host free space including safety reserve")
    opener = opener or requests.get
    response = None
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.part")
    try:
        current = url
        for _ in range(6):
            _url_allowed(current, allowed_hosts)
            response = opener(
                current,
                stream=True,
                timeout=(20, 120),
                allow_redirects=False,
                headers={
                    "User-Agent": "AI-Term-Paper-V3 academic dataset acquisition",
                    "Accept-Encoding": "identity",
                },
            )
            if response.status_code not in (301, 302, 303, 307, 308):
                break
            location = response.headers.get("Location")
            response.close()
            if not location:
                raise AcquisitionError("Redirect lacks Location")
            current = urljoin(current, location)
        else:
            raise AcquisitionError("Too many redirects")
        if response.status_code != 200:
            raise AcquisitionError(f"Source returned HTTP {response.status_code}")
        length = response.headers.get("Content-Length")
        if length and int(length) > max_bytes:
            raise AcquisitionError("Source exceeds configured byte budget")
        sha256, md5, count = hashlib.sha256(), hashlib.md5(), 0
        with temporary.open("xb") as handle:
            for chunk in response.iter_content(1024**2):
                if not chunk:
                    continue
                count += len(chunk)
                if count > max_bytes or shutil.disk_usage(path.parent).free < reserve_bytes + len(chunk):
                    raise AcquisitionError("Stream exceeded byte/storage budget")
                handle.write(chunk)
                sha256.update(chunk)
                md5.update(chunk)
            handle.flush()
            os.fsync(handle.fileno())
        if count == 0 or (length and count != int(length)):
            raise AcquisitionError("Empty or truncated response")
        if expected_sha256 and sha256.hexdigest() != expected_sha256:
            raise AcquisitionError("Downloaded SHA-256 mismatch")
        if expected_md5 and md5.hexdigest() != expected_md5:
            raise AcquisitionError("Official MD5 mismatch")
        # Hard-link installation is atomic AND no-clobber, unlike os.replace.
        os.link(temporary, path)
        return {
            "action": "downloaded",
            "sha256": sha256.hexdigest(),
            "md5": md5.hexdigest(),
            "size_bytes": count,
            "url": url,
            "final_url": current,
            "etag": response.headers.get("ETag"),
            "last_modified": response.headers.get("Last-Modified"),
        }
    finally:
        if response is not None:
            response.close()
        temporary.unlink(missing_ok=True)



#### _hash_stream

In [ ]:
%%inline_module src.common.acquisition src/common/acquisition.py both
def _hash_stream(handle, expected_size):
    digest, count = hashlib.sha256(), 0
    for chunk in iter(lambda: handle.read(1024**2), b""):
        count += len(chunk)
        if count > expected_size:
            raise AcquisitionError("Archive member exceeds declared size")
        digest.update(chunk)
    if count != expected_size:
        raise AcquisitionError("Truncated archive member")
    return digest.hexdigest()



#### archive_members

In [ ]:
%%inline_module src.common.acquisition src/common/acquisition.py both
def archive_members(path, *, max_expanded_bytes, max_members=100000):
    """Hash every regular member without extraction; reject links and unsafe paths."""
    seen, total, count = set(), 0, 0

    def check(name, size):
        nonlocal total, count
        normalized = safe_member_name(name)
        key = normalized.casefold()
        total += size
        count += 1
        if key in seen or count > max_members or total > max_expanded_bytes or size < 0:
            raise AcquisitionError("Duplicate archive path or member/expanded-size budget exceeded")
        seen.add(key)
        return normalized

    if zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as archive:
            for info in archive.infolist():
                name = check(info.filename, info.file_size)
                mode = info.external_attr >> 16
                kind = stat.S_IFMT(mode)
                if kind not in (0, stat.S_IFREG, stat.S_IFDIR) or info.flag_bits & 1:
                    raise AcquisitionError("Links/special/encrypted ZIP members are not accepted")
                if info.is_dir():
                    continue
                with archive.open(info) as handle:
                    digest = _hash_stream(handle, info.file_size)
                yield {"name": name, "size_bytes": info.file_size, "sha256": digest}
    else:
        with tarfile.open(path, "r:*") as archive:
            for info in archive:
                name = check(info.name, info.size)
                if info.isdir():
                    continue
                if not info.isfile():
                    raise AcquisitionError("Only regular files/directories accepted in TAR")
                with archive.extractfile(info) as handle:
                    digest = _hash_stream(handle, info.size)
                yield {"name": name, "size_bytes": info.size, "sha256": digest}



#### download_ranged_file

In [ ]:
%%inline_module src.common.acquisition src/common/acquisition.py both
def download_ranged_file(
    url,
    path,
    *,
    total_bytes,
    allowed_hosts,
    expected_md5,
    expected_sha256=None,
    block_bytes=4 * 1024**2,
    reserve_bytes=RESERVE_BYTES,
    opener=None,
):
    """Bounded four-connection Range transport for slow official large-file servers.

    Ranges are written into ONE private temporary file. Every response must be 206
    with the exact requested Content-Range; final published MD5 is mandatory.
    """
    from concurrent.futures import ThreadPoolExecutor

    _url_allowed(url, allowed_hosts)
    path = Path(path)
    if path.exists() or path.is_symlink():
        return download_file(
            url,
            path,
            max_bytes=total_bytes,
            allowed_hosts=allowed_hosts,
            expected_sha256=expected_sha256,
            opener=opener,
            reserve_bytes=reserve_bytes,
        )
    if total_bytes <= 0 or block_bytes <= 0 or not expected_md5:
        raise AcquisitionError("Range transport needs positive bounds and official MD5")
    path.parent.mkdir(parents=True, exist_ok=True)
    if shutil.disk_usage(path.parent).free < storage_required(total_bytes, 0, reserve_bytes):
        raise AcquisitionError("Insufficient storage for range download")
    opener = opener or requests.get
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.part")

    def block(start):
        end = min(start + block_bytes, total_bytes) - 1
        with opener(
            url,
            stream=True,
            timeout=(20, 60),
            allow_redirects=False,
            headers={
                "Range": f"bytes={start}-{end}",
                "Accept-Encoding": "identity",
                "User-Agent": "AI-Term-Paper-V3 academic range download",
            },
        ) as response:
            if (
                response.status_code != 206
                or response.headers.get("Content-Range") != f"bytes {start}-{end}/{total_bytes}"
            ):
                raise AcquisitionError("Source did not honor the exact bounded range")
            count = 0
            with temporary.open("r+b") as handle:
                handle.seek(start)
                for chunk in response.iter_content(65536):
                    count += len(chunk)
                    if count > end - start + 1 or shutil.disk_usage(path.parent).free < reserve_bytes + len(
                        chunk
                    ):
                        raise AcquisitionError("Range response exceeds byte/storage bound")
                    handle.write(chunk)
            if count != end - start + 1:
                raise AcquisitionError("Truncated range response")
        return count

    try:
        with temporary.open("xb") as handle:
            handle.truncate(total_bytes)
        with ThreadPoolExecutor(max_workers=4) as pool:
            for _ in pool.map(block, range(0, total_bytes, block_bytes)):
                pass
        digest = file_digest(temporary)
        if file_digest(temporary, "md5") != expected_md5 or (expected_sha256 and digest != expected_sha256):
            raise AcquisitionError("Full-file range checksum mismatch")
        os.link(temporary, path)
        return {
            "action": "downloaded",
            "sha256": digest,
            "md5": expected_md5,
            "size_bytes": total_bytes,
            "url": url,
            "final_url": url,
            "transport": "verified_exact_ranges_4_connections",
        }
    finally:
        temporary.unlink(missing_ok=True)



### `src.common.eda_artifacts`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.common.eda_artifacts src/common/eda_artifacts.py both
"""Reproducible EDA artifacts. Raw values stay separate from transformations."""

from __future__ import annotations

import gzip

import json

import math

from datetime import datetime

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from src.common.acquisition import file_digest

from src.common.contracts import ROOT, read_json

BASE = ROOT / "artifacts/manifests/phase24"

CARDS = ROOT / "report/references/dataset_cards/v3_eda"

FIGURES = ROOT / "artifacts/figures/phase24"

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "svg.hashsalt": "ai_v3_phase24_42"})



#### clean

In [ ]:
%%inline_module src.common.eda_artifacts src/common/eda_artifacts.py both
def clean(value):
    if value is pd.NA or value is pd.NaT:
        return None
    if isinstance(value, dict):
        return {str(k): clean(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, np.ndarray)):
        return [clean(v) for v in value]
    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()
    if isinstance(value, np.generic):
        return clean(value.item())
    if value is pd.NA or value is pd.NaT or isinstance(value, float) and not math.isfinite(value):
        return None
    return value



#### write_json

In [ ]:
%%inline_module src.common.eda_artifacts src/common/eda_artifacts.py both
def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(clean(value), ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )



#### write_rows

In [ ]:
%%inline_module src.common.eda_artifacts src/common/eda_artifacts.py both
def write_rows(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as target, gzip.GzipFile(filename="", mode="wb", fileobj=target, mtime=0) as stream:
        for row in rows:
            stream.write(
                (json.dumps(clean(row), ensure_ascii=False, sort_keys=True, allow_nan=False) + "\n").encode()
            )



#### read_rows

In [ ]:
%%inline_module src.common.eda_artifacts src/common/eda_artifacts.py both
def read_rows(path):
    with gzip.open(path, "rt", encoding="utf-8") as stream:
        return [json.loads(line) for line in stream]



#### reference

In [ ]:
%%inline_module src.common.eda_artifacts src/common/eda_artifacts.py both
def reference(path):
    return {
        "path": path.relative_to(ROOT).as_posix(),
        "sha256": file_digest(path),
        "size_bytes": path.stat().st_size,
    }



#### describe_frame

In [ ]:
%%inline_module src.common.eda_artifacts src/common/eda_artifacts.py both
def describe_frame(frame, ids, train_ids):
    train = frame.loc[np.isin(ids, train_ids)]
    numeric = {}
    categories = {}
    for name in frame:
        if pd.api.types.is_numeric_dtype(frame[name]):
            full = frame[name].replace([np.inf, -np.inf], np.nan)
            q1, q3 = train[name].quantile([0.25, 0.75])
            lower, upper = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
            numeric[name] = {
                "min": full.min(),
                "median": full.median(),
                "max": full.max(),
                "iqr_lower_train": lower,
                "iqr_upper_train": upper,
                "outlier_count_full_source": int(((full < lower) | (full > upper)).sum()),
            }
        else:
            categories[name] = frame[name].fillna("__missing__").astype(str).value_counts().to_dict()
    return clean(
        {
            "records": len(frame),
            "columns": list(frame.columns),
            "missing": frame.isna().sum().to_dict(),
            "numeric": numeric,
            "categories": categories,
            "frame_memory_bytes": int(frame.memory_usage(deep=True).sum()),
            "outlier_policy": "1.5 IQR bounds from train only; descriptive flags, no automatic row deletion",
        }
    )



#### save_figure

In [ ]:
%%inline_module src.common.eda_artifacts src/common/eda_artifacts.py both
def save_figure(fig, name):
    FIGURES.mkdir(parents=True, exist_ok=True)
    result = []
    for suffix in ("png", "svg"):
        path = FIGURES / f"{name}.{suffix}"
        metadata = (
            {"Date": None, "Creator": "AI V3 Phase24"} if suffix == "svg" else {"Software": "AI V3 Phase24"}
        )
        fig.savefig(path, dpi=300, bbox_inches="tight", metadata=metadata)
        result.append(reference(path))
    plt.close(fig)
    return result



#### distribution_figure

In [ ]:
%%inline_module src.common.eda_artifacts src/common/eda_artifacts.py both
def distribution_figure(dataset_id, distribution):
    names, values = list(distribution), list(distribution.values())
    fig, ax = plt.subplots(figsize=(7, max(3, len(names) * 0.3)))
    ax.barh(list(map(str, names)), values, color="#326A9B")
    ax.invert_yaxis()
    ax.set_xlabel(
        "Số mẫu (thống kê mô tả, không phải metric mô hình)"
        if "wikidata" not in dataset_id
        else "Số claim trong từng entity snapshot"
    )
    titles = {
        "ch1_iris_history": "Iris: số mẫu theo loài",
        "ch1_mnist_history": "MNIST: số ảnh theo chữ số",
        "ch1_imagenet_history": "ImageNet: chỉ kiểm kê metadata local, không có ảnh",
        "ch2_cdc_diabetes": "CDC Diabetes: phân bố nhãn nhị phân",
        "ch2_bank_marketing": "Bank Marketing: phân bố nhãn đăng ký",
        "ch3_cifar10": "CIFAR-10: phân bố lớp",
        "ch3_eurosat": "EuroSAT: phân bố lớp phủ đất",
        "ch3_idrid": "IDRiD: phân bố DR grade",
        "ch5_finqa": "FinQA: split đã khóa từ Phase12",
        "ch5_medquad_diabetes": "MedQuAD: 247 query diabetes đã khóa",
        "ch5_wikidata_subgraphs": "Wikidata: phân bố 2.056 claim",
    }
    ax.set_title(titles.get(dataset_id, dataset_id.replace("_", " ")))
    ax.spines[["top", "right"]].set_visible(False)
    return save_figure(fig, dataset_id + "_distribution")



#### render_card

In [ ]:
%%inline_module src.common.eda_artifacts src/common/eda_artifacts.py both
def render_card(spec, source, measured):
    """Keep inherited/deferred evidence unmistakably separate from measured training data."""
    dataset = spec["dataset_id"]
    scope = measured["scope"]
    lines = [
        f"# {dataset} — dữ liệu và kiểm tra Phase 24",
        "",
        f"Phạm vi: **{scope}**. Không huấn luyện hay đánh giá TEST trong Phase 24.",
        "",
        "## Nguồn, kích cỡ và cấu trúc",
        "",
        f"Nguồn: [{source['source']['citation']}]({source['source']['url'] or spec['source_url'] or '#'})",
        f"Phiên bản: {source['source']['version']}",
        "",
        f"Giấy phép/phạm vi: {source['source']['license_note']}",
        "",
        (
            f"Raw: {sum(f['size_bytes'] for f in source['files']):,} byte; "
            f"{source['logical_size']['records']} {source['logical_size']['unit']}."
        ),
        "",
        "Số đo và schema phần dùng trong V3:",
        "",
        "```json",
        json.dumps(clean(measured["size_and_schema"]), ensure_ascii=False, indent=2),
        "```",
        "",
        "## Mẫu input/output",
        "",
        measured["sample_note"],
        "",
        "```json",
        json.dumps(clean(measured["samples"]), ensure_ascii=False, indent=2),
        "```",
        "",
        "## Phân bố và chất lượng dữ liệu",
        "",
        "```json",
        json.dumps(clean(measured["distribution_summary"]), ensure_ascii=False, indent=2),
        "```",
        "",
        "## Cách chia tập và tiền xử lý",
        "",
        measured["split_note"],
        "",
        "## Nhận xét và giới hạn",
        "",
    ]
    lines += [f"- {note}" for note in measured["comments"]]
    if spec["parent_dataset_ids"]:
        lines += ["", "Nguồn và điều khoản của từng parent:", ""]
        for parent in spec["parent_dataset_ids"]:
            parent_source = read_json(ROOT / f"artifacts/manifests/datasets/{parent}.json")["contract"][
                "source"
            ]
            lines += [f"- [{parent}]({parent}.md): [{parent_source['citation']}]({parent_source['url']})."]
    lines += [
        "",
        "## Bằng chứng tái lập",
        "",
        f"Chi tiết: `artifacts/manifests/phase24/{dataset}/eda.json`.",
        "Nguồn Phase23, raw và TEST Ch5 giữ nguyên; số đo mới không ghi ngược vào protocol.",
    ]
    for ref in measured.get("figures", []):
        if ref["path"].endswith(".png"):
            lines += ["", f"![{dataset}]({('../../../../' + ref['path'])})"]
    CARDS.mkdir(parents=True, exist_ok=True)
    path = CARDS / f"{dataset}.md"
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return reference(path)



### `src.common.source_inspection`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.common.source_inspection src/common/source_inspection.py both
"""Source-level counts and schema validation only. No EDA, splits or TEST evaluation."""

from __future__ import annotations

import io

import re

import tarfile

import zipfile

from pathlib import Path

import numpy as np

import pandas as pd

from PIL import Image

from src.common.acquisition import AcquisitionError



#### require

In [ ]:
%%inline_module src.common.source_inspection src/common/source_inspection.py both
def require(condition, message):
    if not condition:
        raise AcquisitionError(message)



#### normalize_column

In [ ]:
%%inline_module src.common.source_inspection src/common/source_inspection.py both
def normalize_column(name):
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", str(name).strip())
    name = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    return name if name and name[0].isalpha() else "v_" + name



#### column_contracts

In [ ]:
%%inline_module src.common.source_inspection src/common/source_inspection.py both
def column_contracts(frame, target, ids=frozenset(), times=frozenset()):
    result = []
    for raw in frame.columns:
        name = normalize_column(raw)
        series = frame[raw]
        dtype = (
            "integer"
            if pd.api.types.is_integer_dtype(series)
            else ("number" if pd.api.types.is_numeric_dtype(series) else "string")
        )
        role = (
            "target"
            if name == target
            else ("id" if name in ids else ("time" if name in times else "feature"))
        )
        result.append({"name": name, "dtype": dtype, "role": role})
    require(len({row["name"] for row in result}) == len(result), "Column alias collision")
    return result



#### _table

In [ ]:
%%inline_module src.common.source_inspection src/common/source_inspection.py both
def _table(frame, *, rows, target, ids=frozenset(), exclude=frozenset(), aliases=None):
    require(len(frame) == rows, f"Unexpected record count: {len(frame)} != {rows}")
    columns = column_contracts(frame, target, ids)
    require(any(row["role"] == "target" for row in columns), f"Missing target {target}")
    return {
        "records": len(frame),
        "features": sum(row["role"] == "feature" and row["name"] not in exclude for row in columns),
        "unit": "raw rows",
        "columns": columns,
        "observed_columns": list(frame.columns),
        "raw_to_contract_aliases": aliases or {raw: normalize_column(raw) for raw in frame.columns},
        "excluded_feature_columns": sorted(ids | exclude),
        "checks": ["row_count", "column_schema"],
    }



#### inspect_source

In [ ]:
%%inline_module src.common.source_inspection src/common/source_inspection.py both
def inspect_source(dataset_id, path):
    path = Path(path)
    if dataset_id == "ch1_iris_history":
        with zipfile.ZipFile(path) as archive:
            frame = pd.read_csv(
                archive.open("iris.data"),
                header=None,
                names=["sepal_length", "sepal_width", "petal_length", "petal_width", "species"],
            )
        require(frame.shape == (150, 5) and frame.species.nunique() == 3, "Invalid Iris schema")
        return _table(frame, rows=150, target="species")
    if dataset_id == "ch1_mnist_history":
        with np.load(path, allow_pickle=False) as data:
            require(set(data.files) == {"x_train", "y_train", "x_test", "y_test"}, "MNIST keys changed")
            for split, count in (("train", 60000), ("test", 10000)):
                require(data[f"x_{split}"].shape == (count, 28, 28), "Invalid MNIST images")
                require(data[f"x_{split}"].dtype == np.uint8, "Invalid MNIST pixel dtype")
                require(
                    data[f"y_{split}"].shape == (count,)
                    and set(np.unique(data[f"y_{split}"])) == set(range(10)),
                    "Invalid MNIST labels",
                )
        return {
            "records": 70000,
            "features": 784,
            "unit": "images (official 60000 train + 10000 test)",
            "columns": [],
            "checks": ["npz_no_pickle", "shape", "dtype", "label_range"],
            "native_shape": [28, 28, 1],
        }
    if dataset_id == "ch1_imagenet_history":
        text = path.read_text(encoding="utf-8")
        require("ILSVRC2012" in text and "1000" in text, "Not the ILSVRC2012 page")
        return {
            "records": 1,
            "features": None,
            "unit": "official metadata webpage, ZERO local images",
            "columns": [],
            "checks": ["official_page_identity"],
            "published_not_measured": {"categories": 1000, "training_images_approx": 1200000},
        }
    if dataset_id == "ch2_cdc_diabetes":
        frame = pd.read_csv(path)
        require(
            frame.shape == (253680, 23) and "Diabetes_binary" in frame and "ID" in frame,
            "UCI CDC schema changed; do not substitute a three-class or balanced variant",
        )
        require(set(frame.Diabetes_binary.unique()) == {0, 1}, "CDC binary labels changed")
        return _table(frame, rows=253680, target="diabetes_binary", ids={"id"})
    if dataset_id == "ch2_ames_housing":
        frame = pd.read_csv(path, sep="\t")
        require(
            frame.shape == (2930, 82) and "SalePrice" in frame,
            "Require original Ames, not processed 2929 rows",
        )
        require((frame.SalePrice > 0).all(), "Ames invalid target")
        return _table(frame, rows=2930, target="sale_price", ids={"pid", "order"})
    if dataset_id == "ch2_bank_marketing":
        with zipfile.ZipFile(path) as outer, zipfile.ZipFile(io.BytesIO(outer.read("bank.zip"))) as inner:
            frame = pd.read_csv(inner.open("bank-full.csv"), sep=";")
        require(frame.shape == (45211, 17) and set(frame.y.unique()) == {"yes", "no"}, "Wrong Bank variant")
        frame = frame.rename(columns={"y": "subscription"})
        result = _table(frame, rows=45211, target="subscription", exclude={"duration"})
        result["selected_member"] = "bank.zip!bank-full.csv"
        result["observed_columns"] = ["y" if name == "subscription" else name for name in frame.columns]
        result["raw_to_contract_aliases"].pop("subscription")
        result["raw_to_contract_aliases"]["y"] = "subscription"
        return result
    if dataset_id == "ch3_eurosat":
        classes, shapes, images = set(), set(), 0
        with zipfile.ZipFile(path) as archive:
            for name in archive.namelist():
                if not name.lower().endswith(".jpg"):
                    continue
                with archive.open(name) as handle, Image.open(handle) as picture:
                    picture.load()
                    shapes.add((*picture.size, picture.mode))
                classes.add(Path(name).parent.name)
                images += 1
        require(
            images == 27000 and len(classes) == 10 and shapes == {(64, 64, "RGB")},
            "EuroSAT RGB schema mismatch",
        )
        return {
            "records": images,
            "features": 64 * 64 * 3,
            "unit": "RGB image patches",
            "columns": [],
            "native_shape": [64, 64, 3],
            "class_names": sorted(classes),
            "checks": ["all_images_decode", "image_count", "class_count", "native_shape"],
        }
    if dataset_id == "ch3_cifar10":
        counts = {}
        with tarfile.open(path, "r:gz") as archive:
            for member in archive:
                if not re.fullmatch(r"cifar-10-batches-bin/(data_batch_[1-5]|test_batch)\.bin", member.name):
                    continue
                with archive.extractfile(member) as handle:
                    array = np.frombuffer(handle.read(), dtype=np.uint8)
                require(array.size == 10000 * 3073, "CIFAR binary record length mismatch")
                require(
                    set(np.unique(array.reshape(10000, 3073)[:, 0])) == set(range(10)), "CIFAR labels invalid"
                )
                counts[member.name] = 10000
        require(len(counts) == 6, "CIFAR missing official batches")
        return {
            "records": sum(counts.values()),
            "features": 3072,
            "unit": "images (50000 train + 10000 test)",
            "columns": [],
            "native_shape": [32, 32, 3],
            "batch_counts": counts,
            "checks": ["binary_not_pickle", "record_length", "batch_count", "label_range"],
        }
    if dataset_id == "ch4_gold_pink_sheet":
        return _gold(path)
    if dataset_id == "ch4_bis_house_prices":
        return _bis(path)
    if dataset_id == "ch4_household_power":
        with zipfile.ZipFile(path) as archive:
            name = "household_power_consumption.txt"
            records, headers = 0, None
            with io.TextIOWrapper(archive.open(name), encoding="utf-8") as handle:
                headers = handle.readline().strip().split(";")
                for line in handle:
                    require(len(line.rstrip().split(";")) == 9, "Power malformed record")
                    records += 1
        require(
            records == 2075259 and len(headers) == 9 and "Global_active_power" in headers,
            "Power release mismatch",
        )
        return {
            "records": records,
            "features": 7,
            "unit": "raw minute rows, NOT hourly training windows",
            "columns": [
                {
                    "name": normalize_column(c),
                    "dtype": "string" if i < 2 else "number",
                    "role": "time" if i < 2 else ("target" if i == 2 else "feature"),
                }
                for i, c in enumerate(headers)
            ],
            "observed_columns": headers,
            "checks": ["raw_line_count", "row_width", "column_schema"],
            "deferred": "Phase24 causal hourly aggregation and missingness handling; raw '?' retained.",
        }
    raise AcquisitionError(f"No source inspector: {dataset_id}")



#### _gold

In [ ]:
%%inline_module src.common.source_inspection src/common/source_inspection.py both
def _gold(path):
    import openpyxl

    workbook = openpyxl.load_workbook(path, read_only=True, data_only=True)
    try:
        require("Monthly Prices" in workbook.sheetnames, f"Gold sheets changed: {workbook.sheetnames}")
        rows = list(workbook["Monthly Prices"].values)
        header_index = next(i for i, row in enumerate(rows) if "Gold" in row)
        headers = rows[header_index]
        indices = {name: headers.index(name) for name in ("Gold", "Silver", "Platinum", "Crude oil, average")}
        records = [row for row in rows[header_index + 1 :] if re.fullmatch(r"\d{4}M\d{2}", str(row[0]))]
        require(len(records) > 600, "Gold history unexpectedly short")
        require(
            all(isinstance(row[indices["Gold"]], (int, float)) for row in records),
            "Gold target missing/nonnumeric",
        )
        return {
            "records": len(records),
            "features": 4,
            "unit": "monthly observations, latest revised vintage",
            "columns": [{"name": "month", "dtype": "datetime", "role": "time"}]
            + [
                {
                    "name": {"Gold": "gold_usd_per_troy_ounce", "Crude oil, average": "crude_oil"}.get(
                        name, normalize_column(name)
                    ),
                    "dtype": "number",
                    "role": "target" if name == "Gold" else "feature",
                }
                for name in indices
            ],
            "observed_columns": [str(name) for name in headers],
            "raw_to_contract_aliases": {"Gold": "gold_usd_per_troy_ounce", "Crude oil, average": "crude_oil"},
            "time_range": [str(records[0][0]), str(records[-1][0])],
            "selected_column_indices": indices,
            "checks": ["sheet_schema", "required_commodity_columns", "numeric_gold", "history_length"],
            "deferred": "Phase24 uses gold plus explanatory commodities with one-period lag; no windows fitted here.",
        }
    finally:
        workbook.close()



#### _bis

In [ ]:
%%inline_module src.common.source_inspection src/common/source_inspection.py both
def _bis(path):
    with zipfile.ZipFile(path) as archive:
        names = [name for name in archive.namelist() if name.lower().endswith(".csv")]
        require(len(names) == 1, f"BIS archive schema changed: {names}")
        frame = pd.read_csv(archive.open(names[0]), low_memory=False)
    time_columns = [name for name in frame if re.fullmatch(r"\d{4}-Q[1-4]", name)]
    require(
        len(time_columns) >= 80 and "FREQ" in frame and "REF_AREA" in frame,
        f"BIS unexpected columns: {list(frame)[:25]}",
    )
    observations = int(frame[time_columns].apply(pd.to_numeric, errors="coerce").notna().sum().sum())
    return {
        "records": observations,
        "features": None,
        "unit": "nonmissing quarterly series-observations in entire raw bulk (not selected nominal panel)",
        "columns": [
            {"name": "ref_area", "dtype": "category", "role": "group"},
            {"name": "quarter", "dtype": "datetime", "role": "time"},
            {"name": "nominal_house_price_index", "dtype": "number", "role": "target"},
        ],
        "raw_series_rows": len(frame),
        "raw_series_dimensions": {
            str(name): sorted(str(v) for v in frame[name].dropna().unique())
            for name in frame.columns[:10]
            if frame[name].nunique() < 100
        },
        "time_range": [time_columns[0], time_columns[-1]],
        "observed_columns": list(frame.columns),
        "selected_member": names[0],
        "checks": ["wide_csv_schema", "quarter_columns", "nonmissing_observation_count"],
        "deferred": "Phase24 freezes nominal national series and coverage >=80; no selection by TEST performance.",
    }



### `src.common.eda_sources`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.common.eda_sources src/common/eda_sources.py both
"""Read frozen sources without extraction or mutation; deterministic V3 aliases."""

from __future__ import annotations

import hashlib

import io

import re

import tarfile

import zipfile

import numpy as np

import pandas as pd

from PIL import Image

from src.common.contracts import ROOT, read_json

from src.common.source_inspection import normalize_column, require



#### source

In [ ]:
%%inline_module src.common.eda_sources src/common/eda_sources.py both
def source(dataset_id):
    return read_json(ROOT / f"artifacts/manifests/datasets/{dataset_id}.json")



#### load_tabular

In [ ]:
%%inline_module src.common.eda_sources src/common/eda_sources.py both
def load_tabular(dataset_id):
    envelope = source(dataset_id)
    path = ROOT / envelope["contract"]["files"][0]["path"]
    if dataset_id == "ch2_cdc_diabetes":
        raw = pd.read_csv(path)
        raw.columns = [normalize_column(c) for c in raw]
        target = raw.pop("diabetes_binary")
        raw = raw.drop(columns="id")
        properties = None
    elif dataset_id == "ch2_ames_housing":
        raw = pd.read_csv(path, sep="\t")
        raw.columns = [normalize_column(c) for c in raw]
        target = raw.pop("sale_price")
        properties = raw.pid.astype(str).tolist()
        raw = raw.drop(columns=["pid", "order"])
    elif dataset_id == "ch2_bank_marketing":
        with zipfile.ZipFile(path) as outer, zipfile.ZipFile(io.BytesIO(outer.read("bank.zip"))) as inner:
            raw = pd.read_csv(inner.open("bank-full.csv"), sep=";")
        target = raw.pop("y").map({"no": 0, "yes": 1})
        raw = raw.drop(columns="duration")
        properties = None
    else:
        raise ValueError(f"Not a Ch2 source: {dataset_id}")
    require(target.notna().all(), "Missing source target")
    ids = [f"row_{i:07d}" for i in range(len(raw))]
    # SHA-256 includes named, typed scalar features, never labels or excluded IDs.
    hashes = [
        hashlib.sha256(line.encode()).hexdigest()
        for line in raw.to_json(orient="records", lines=True).splitlines()
    ]
    return raw, target.reset_index(drop=True), ids, hashes, properties



#### gold_features

In [ ]:
%%inline_module src.common.eda_sources src/common/eda_sources.py both
def gold_features(raw):
    result = pd.DataFrame({"time": raw.time, "series": "GOLD", "target": raw.gold, "gold": raw.gold})
    for name in ("silver", "platinum", "crude_oil"):
        result[name + "_lag1"] = raw[name].shift(1)
    return result, ["gold", "silver_lag1", "platinum_lag1", "crude_oil_lag1"]



#### select_bis_panel

In [ ]:
%%inline_module src.common.eda_sources src/common/eda_sources.py both
def select_bis_panel(raw):
    """BIS coded VALUE=N, UNIT_MEASURE=628, exclude aggregates, >=80 observations.

    Eligibility uses coverage/identifiers, never price performance or TEST metrics.
    No mixing nominal/real or index/YoY; preserve missing targets in a regular grid.
    """
    time_columns = [c for c in raw if re.fullmatch(r"\d{4}-Q[1-4]", c)]
    selected = raw.loc[(raw.FREQ == "Q") & (raw.VALUE == "N") & (raw.UNIT_MEASURE.astype(str) == "628")]
    selected = selected.loc[
        selected.REF_AREA.str.fullmatch(r"[A-Z]{2}") & ~selected.REF_AREA.isin(["XM", "XW"])
    ]
    require(not selected.REF_AREA.duplicated().any(), "Multiple nominal national series per economy")
    panels, retained, excluded = [], [], []
    coverage = {}
    for _, row in selected.sort_values("REF_AREA").iterrows():
        values = pd.to_numeric(row[time_columns], errors="coerce")
        sid = f"Q.{row.REF_AREA}.N.628"
        coverage[sid] = int(values.notna().sum())
        if coverage[sid] < 80:
            excluded.append({"series": sid, "reason": "fewer_than_80_nonmissing_observations"})
            continue
        first, last = np.flatnonzero(values.notna().to_numpy())[[0, -1]]
        values = values.iloc[first : last + 1]
        times = pd.PeriodIndex(values.index.str.replace("-Q", "Q"), freq="Q").to_timestamp()
        panel = pd.DataFrame({"time": times, "series": sid, "target": values.to_numpy(dtype=float)})
        # A window ends at its last observed quarter; no future revised quarter is an input.
        panel["house_index"] = panel.target
        panel["index_change"] = panel.target.diff()
        panel["economy"] = row.REF_AREA
        panels.append(panel)
        retained.append(sid)
    require(bool(panels), "No admissible nominal BIS series")
    panel = pd.concat(panels, ignore_index=True)
    # Freeze a calendar by coverage, not values: avoid one century-long series
    # setting the global cutoff while most economies did not yet publish data.
    simultaneous = panel.groupby("time").target.count()
    common_start = simultaneous.index[simultaneous >= int(np.ceil(0.8 * len(retained)))][0]
    panel = panel.loc[panel.time >= common_start].reset_index(drop=True)
    for economy in sorted(panel.economy.unique()):
        panel["country_" + economy] = (panel.economy == economy).astype(float)
    features = ["house_index", "index_change"] + [c for c in panel if c.startswith("country_")]
    panel = panel.drop(columns="economy")
    return (
        panel,
        features,
        {
            "retained_series": retained,
            "excluded_series": excluded,
            "coverage": coverage,
            "common_history_start": common_start.isoformat(),
            "history_rule": "earliest quarter with >=80% retained economies observed; schema/coverage only",
            "unit": "nominal index, 2010=100",
            "selection": "Q, N, 628; national economies only; >=80 nonmissing; no performance filter",
        },
    )



#### hourly_power

In [ ]:
%%inline_module src.common.eda_sources src/common/eda_sources.py both
def hourly_power(raw):
    times = pd.to_datetime(raw.Date + " " + raw.Time, format="%d/%m/%Y %H:%M:%S")
    values = raw.drop(columns=["Date", "Time"]).apply(pd.to_numeric, errors="coerce")
    values.index = times
    require(not values.index.duplicated().any(), "Duplicate power minute timestamp")
    observed = values.resample("h").count()
    hourly = values.resample("h").mean().where(observed >= 45)
    result = pd.DataFrame(
        {"time": hourly.index, "series": "HOUSEHOLD", "target": hourly.Global_active_power.to_numpy()}
    )
    features = []
    for name in hourly:
        # Target remains untouched. Only input sensor aggregates use bounded causal ffill.
        result[name] = hourly[name].ffill(limit=6).to_numpy()
        result[name + "_missing"] = hourly[name].isna().astype(float).to_numpy()
        features.extend([name, name + "_missing"])
    return result, features



#### load_series

In [ ]:
%%inline_module src.common.eda_sources src/common/eda_sources.py both
def load_series(dataset_id):
    path = ROOT / source(dataset_id)["contract"]["files"][0]["path"]
    if dataset_id == "ch4_gold_pink_sheet":
        import openpyxl

        book = openpyxl.load_workbook(path, read_only=True, data_only=True)
        try:
            rows = list(book["Monthly Prices"].values)
        finally:
            book.close()
        header = next(i for i, row in enumerate(rows) if "Gold" in row)
        positions = {
            name: rows[header].index(name) for name in ("Gold", "Silver", "Platinum", "Crude oil, average")
        }
        records = [row for row in rows[header + 1 :] if re.fullmatch(r"\d{4}M\d{2}", str(row[0]))]
        raw = pd.DataFrame({"time": pd.to_datetime([r[0] for r in records], format="%YM%m")})
        for old, new in zip(positions, ("gold", "silver", "platinum", "crude_oil"), strict=True):
            raw[new] = [r[positions[old]] for r in records]
        frame, features = gold_features(raw)
        return (
            frame,
            features,
            {"publication_lag": "explanatory commodities lagged one month; revised vintage, not realtime"},
        )
    with zipfile.ZipFile(path) as archive:
        if dataset_id == "ch4_bis_house_prices":
            name = next(n for n in archive.namelist() if n.endswith(".csv"))
            return select_bis_panel(pd.read_csv(archive.open(name), low_memory=False))
        require(dataset_id == "ch4_household_power", "Unknown time-series source")
        raw = pd.read_csv(
            archive.open("household_power_consumption.txt"), sep=";", na_values="?", low_memory=False
        )
    frame, features = hourly_power(raw)
    return (
        frame,
        features,
        {
            "raw_minute_rows": len(raw),
            "observed_minute_threshold": 45,
            "input_ffill_hours": 6,
            "target_imputation": "never",
            "sensor_units": {
                "Global_active_power": "kW",
                "Global_reactive_power": "kVAR",
                "Voltage": "V",
                "Global_intensity": "A",
                "Sub_metering_1/2/3": "Wh/min observations, hourly mean not summed kWh",
            },
        },
    )



#### _image_record

In [ ]:
%%inline_module src.common.eda_sources src/common/eda_sources.py both
def _image_record(image, *, key, label, member, official, resolution, **extra):
    require(image.mode == "RGB", "Require RGB source images")
    native_hash = hashlib.sha256(np.asarray(image).tobytes()).hexdigest()
    resized = image.resize(tuple(resolution), Image.Resampling.BILINEAR)
    model_pixels = np.asarray(resized)
    return {
        "id": key,
        "target": str(label),
        "member": member,
        "official_split": official,
        "pixel_sha256": native_hash,
        "model_pixel_sha256": hashlib.sha256(np.asarray(resized).tobytes()).hexdigest(),
        "width": image.width,
        "height": image.height,
        "mean_intensity": float(model_pixels.mean()),
        "std_intensity": float(model_pixels.std()),
        "black_fraction": float((model_pixels.max(axis=2) == 0).mean()),
        **extra,
    }



#### parse_cifar_classes

In [ ]:
%%inline_module src.common.eda_sources src/common/eda_sources.py both
def parse_cifar_classes(text):
    names = [line.strip() for line in text.splitlines() if line.strip()]
    require(len(names) == len(set(names)) == 10, "Require ten unique CIFAR class names")
    return names



#### load_images

In [ ]:
%%inline_module src.common.eda_sources src/common/eda_sources.py both
def load_images(dataset_id, resolution):
    path = ROOT / source(dataset_id)["contract"]["files"][0]["path"]
    rows = []
    if dataset_id == "ch3_cifar10":
        with tarfile.open(path, "r:gz") as archive:
            with archive.extractfile("cifar-10-batches-bin/batches.meta.txt") as stream:
                class_names = parse_cifar_classes(stream.read().decode("utf-8"))
            for member in archive:
                if not re.fullmatch(r"cifar-10-batches-bin/(data_batch_[1-5]|test_batch)\.bin", member.name):
                    continue
                with archive.extractfile(member) as stream:
                    records = np.frombuffer(stream.read(), dtype=np.uint8).reshape(-1, 3073)
                official = "test" if "test_batch" in member.name else "train"
                for i, record in enumerate(records):
                    pixels = record[1:].reshape(3, 32, 32).transpose(1, 2, 0)
                    image = Image.fromarray(pixels)
                    rows.append(
                        _image_record(
                            image,
                            key=f"{member.name.split('/')[-1]}/{i:05d}",
                            label=record[0],
                            member=member.name,
                            official=official,
                            resolution=resolution,
                            record_offset=i,
                            class_name=class_names[int(record[0])],
                        )
                    )
        return pd.DataFrame(rows)
    index = None
    if dataset_id == "ch3_idrid":
        import yaml

        registry = yaml.safe_load((ROOT / "data/registry.yaml").read_text())
        index_path = next(p for p in registry["provenance_evidence_paths"] if p.endswith("image_index.json"))
        index = read_json(ROOT / index_path)
        members = {row["member"]: key for key, row in index["image_members"].items()}
    else:
        require(dataset_id == "ch3_eurosat", "Unknown CNN source")
    with zipfile.ZipFile(path) as archive:
        for name in sorted(archive.namelist()):
            if not name.lower().endswith(".jpg"):
                continue
            key = members[name] if index else name
            label = index["labels"][key]["grade"] if index else name.split("/")[-2]
            official = index["labels"][key]["official_split"] if index else None
            with archive.open(name) as stream, Image.open(stream) as image:
                image.load()
                rows.append(
                    _image_record(
                        image, key=key, label=label, member=name, official=official, resolution=resolution
                    )
                )
    return pd.DataFrame(rows)



#### image_for_row

In [ ]:
%%inline_module src.common.eda_sources src/common/eda_sources.py both
def image_for_row(dataset_id, row, resolution=None):
    path = ROOT / source(dataset_id)["contract"]["files"][0]["path"]
    if dataset_id == "ch3_cifar10":
        with tarfile.open(path, "r:gz") as archive, archive.extractfile(row["member"]) as stream:
            stream.read(int(row["record_offset"]) * 3073)
            record = np.frombuffer(stream.read(3073), dtype=np.uint8)
        image = Image.fromarray(record[1:].reshape(3, 32, 32).transpose(1, 2, 0))
    else:
        with (
            zipfile.ZipFile(path) as archive,
            archive.open(row["member"]) as stream,
            Image.open(stream) as raw,
        ):
            image = raw.copy()
    return image.resize(tuple(resolution), Image.Resampling.BILINEAR) if resolution else image



### `src.common.tensor_adapters`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.common.tensor_adapters src/common/tensor_adapters.py both
"""Canonical Phase25 tensors. Apply frozen train statistics; never fit here.

Order is numeric input columns followed by categorical input columns/vocabulary.
The ordered names are persisted explicitly, independently of JSON object order.
"""

from __future__ import annotations

import copy

import numpy as np

import pandas as pd



#### TensorAdapter

In [ ]:
%%inline_module src.common.tensor_adapters src/common/tensor_adapters.py both
class TensorAdapter:
    def __init__(self, state):
        self.state = copy.deepcopy(state)
        if "target_transform" in self.state and "name" not in self.state["target_transform"]:
            self.state["target_transform"]["name"] = "standardize"
        state = self.state
        names = state["input_columns"]
        numeric, categorical = state["numeric"], state["categorical"]
        if (
            state.get("fit_scope") != "train_only"
            or len(set(names)) != len(names)
            or set(numeric) & set(categorical)
            or set(names) != set(numeric) | set(categorical)
        ):
            raise ValueError("Invalid train-only feature schema")
        for rule in numeric.values():
            if not all(np.isfinite(rule[k]) for k in ("median", "mean", "std")) or rule["std"] <= 0:
                raise ValueError("Invalid frozen numeric statistics")
        for values in categorical.values():
            if not values or len(set(values)) != len(values) or not all(isinstance(v, str) for v in values):
                raise ValueError("Invalid category vocabulary")
        self.numeric_names = [n for n in names if n in numeric]
        self.categorical_names = [n for n in names if n in categorical]
        self.output_feature_names = self.numeric_names + [
            f"{name}={category}" for name in self.categorical_names for category in categorical[name]
        ]
        if (
            len(set(self.output_feature_names)) != len(self.output_feature_names)
            or len(self.output_feature_names) != state["output_width"]
        ):
            raise ValueError("Feature output width/name mismatch")
        labels = state.get("class_names", {})
        if labels and set(labels) != {str(i) for i in range(len(labels))}:
            raise ValueError("Class IDs must be contiguous from zero")
        self.class_names = [str(labels[str(i)]) for i in range(len(labels))]
        transform = state.get("target_transform", {"name": "identity"})
        if transform["name"] not in {"identity", "standardize"}:
            raise ValueError("Unsupported frozen target transform")
        if transform["name"] == "standardize" and (
            not np.isfinite([transform["mean"], transform["std"]]).all() or transform["std"] <= 0
        ):
            raise ValueError("Invalid target statistics")

    def to_dict(self):
        return {
            "adapter_version": "phase25.1",
            "state": copy.deepcopy(self.state),
            "output_feature_names": list(self.output_feature_names),
            "class_names": list(self.class_names),
        }

    @classmethod
    def from_dict(cls, document):
        adapter = cls(document["state"])
        if (
            document["adapter_version"] != "phase25.1"
            or document["output_feature_names"] != adapter.output_feature_names
            or document["class_names"] != adapter.class_names
        ):
            raise ValueError("Serialized adapter order/mapping mismatch")
        return adapter

    def transform(self, frame):
        if len(frame.columns) != len(set(frame.columns)) or set(frame.columns) != set(
            self.state["input_columns"]
        ):
            raise ValueError("Feature schema mismatch; IDs/targets must be excluded")
        columns = []
        for name in self.numeric_names:
            rule = self.state["numeric"][name]
            value = (
                pd.to_numeric(frame[name], errors="raise")
                .replace([np.inf, -np.inf], np.nan)
                .fillna(rule["median"])
            )
            columns.append(((value.to_numpy(dtype=float) - rule["mean"]) / rule["std"])[:, None])
        for name in self.categorical_names:
            value = frame[name].fillna("__missing__").astype(str).to_numpy()
            columns.append(value[:, None] == np.asarray(self.state["categorical"][name])[None, :])
        result = np.ascontiguousarray(np.concatenate(columns, axis=1), dtype=np.float32)
        if not np.isfinite(result).all():
            raise ValueError("Nonfinite input tensor")
        return result

    def target(self, values, *, inverse=False):
        result = np.asarray(values, dtype=np.float64)
        if not np.isfinite(result).all():
            raise ValueError("Missing/nonfinite labels must not be imputed")
        rule = self.state.get("target_transform", {"name": "identity"})
        if rule["name"] == "standardize":
            result = result * rule["std"] + rule["mean"] if inverse else (result - rule["mean"]) / rule["std"]
        return result if inverse else result.astype(np.float32)

    def inverse_target(self, values):
        return self.target(values, inverse=True)

    def encode_labels(self, values):
        mapping = {name: i for i, name in enumerate(self.class_names)}
        if not mapping or any(value not in mapping for value in values):
            raise ValueError("Unknown class label")
        return np.asarray([mapping[value] for value in values], dtype=np.int64)



#### image_batch

In [ ]:
%%inline_module src.common.tensor_adapters src/common/tensor_adapters.py both
def image_batch(images, implementation):
    images = np.asarray(images)
    if images.ndim != 4 or images.shape[-1] != 3 or images.dtype != np.uint8:
        raise ValueError("Images must be uint8 NHWC RGB; resize/decode through frozen source loader")
    if implementation not in {"scratch", "keras", "pytorch"}:
        raise ValueError("Unknown implementation")
    result = images.astype(np.float32) / 255
    if implementation == "pytorch":
        result = result.transpose(0, 3, 1, 2)
    return np.ascontiguousarray(result)



#### select_rows

In [ ]:
%%inline_module src.common.tensor_adapters src/common/tensor_adapters.py both
def select_rows(source_ids, selected_ids):
    if len(set(source_ids)) != len(source_ids) or len(set(selected_ids)) != len(selected_ids):
        raise ValueError("Duplicate IDs cannot be aligned")
    index = {value: i for i, value in enumerate(source_ids)}
    if any(value not in index for value in selected_ids):
        raise ValueError("Unknown selected ID")
    return [index[value] for value in selected_ids]



#### window_batch

In [ ]:
%%inline_module src.common.tensor_adapters src/common/tensor_adapters.py both
def window_batch(frame, windows, adapter):
    """Only materialize requested windows; source indices must match Phase24 cache."""
    features = adapter.state["input_columns"]
    x, y = [], []
    for row in windows:
        start, end = row["start_index"], row["origin_index"]
        targets = row["target_indices"]
        if start < 0 or end < start or not targets or min(targets) <= end or max(targets) >= len(frame):
            raise ValueError("Noncausal/out-of-range window")
        context = frame.iloc[start : end + 1]
        future = frame.iloc[targets]
        if not (context.series.eq(row["series"]).all() and future.series.eq(row["series"]).all()):
            raise ValueError("Window crosses series")
        x.append(adapter.transform(context[features]))
        y.append(adapter.target(future.target.to_numpy()))
    if not x:
        raise ValueError("Empty window batch")
    return np.stack(x), np.stack(y)



### `src.common.data_access`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.common.data_access src/common/data_access.py both
"""Read-only model data views over Phase24. TEST is deliberately unavailable.

Future TEST evaluation must use a separately sealed evaluator in Phases26–29,
not a boolean override on this development loader.
"""

from __future__ import annotations

import copy

import io

import numpy as np

import pandas as pd

from src.common.artifact_core import sha256_file

from src.common.contracts import PROTOCOL, ROOT, canonical_sha256, read_json

from src.common.eda_artifacts import read_rows

from src.common.eda_sources import image_for_row, load_series, load_tabular

from src.common.tensor_adapters import TensorAdapter, image_batch, select_rows, window_batch



#### DatasetView

In [ ]:
%%inline_module src.common.data_access src/common/data_access.py both
class DatasetView:
    def __init__(self, dataset_id, fold=None):
        catalog = read_json(ROOT / "data/catalog_v3.json")
        entry = next((row for row in catalog["datasets"] if row["dataset_id"] == dataset_id), None)
        if entry is None or entry["chapter"] not in {2, 3, 4}:
            raise ValueError("Not a V3 trainable dataset")
        if fold not in {None, 1, 2, 3} or fold is not None and entry["chapter"] != 4:
            raise ValueError("Invalid rolling fold")
        self.dataset_id = dataset_id
        self.kind = {2: "tabular", 3: "image", 4: "timeseries"}[entry["chapter"]]
        self.spec = next(row for row in read_json(PROTOCOL)["datasets"] if row["dataset_id"] == dataset_id)
        self.base = ROOT / f"artifacts/manifests/phase24/{dataset_id}"
        self.split_dir = ROOT / f"data/splits/v3/{dataset_id}"
        for key in ("source_manifest", "split", "preprocessing"):
            ref = entry[key]
            if sha256_file(ROOT / ref["path"]) != ref["sha256"]:
                raise ValueError(f"Catalog hash mismatch: {key}")
        self.preprocessing = read_json(ROOT / entry["preprocessing"]["path"])
        prep_path = ROOT / entry["preprocessing"]["path"]
        if fold is not None:
            self.split_dir /= f"rolling_fold_{fold}"
            prep_path = self.split_dir / "preprocessing.json"
            self.preprocessing = read_json(prep_path)
        self.adapter = None if self.kind == "image" else TensorAdapter(self.preprocessing)
        labels = self.preprocessing.get("class_names")
        if labels is None and dataset_id == "ch3_eurosat":
            labels = {str(i): name for i, name in enumerate(sorted(self.preprocessing["class_weights"]))}
        if labels is None and self.kind == "image":
            labels = {str(i): f"DR grade {i}" for i in range(self.spec["class_count"])}
        if labels is None and self.spec["task"] == "binary_classification":
            labels = (
                {"0": "no diabetes", "1": "prediabetes or diabetes"}
                if dataset_id == "ch2_cdc_diabetes"
                else {"0": "no", "1": "yes"}
            )
        source_labels = (
            sorted(self.preprocessing["class_weights"]) if dataset_id == "ch3_eurosat" else list(labels or {})
        )
        label_to_id = {key: i for i, key in enumerate(source_labels)}
        if labels and dataset_id != "ch3_eurosat":
            label_to_id = {key: int(key) for key in labels}
        self.adapter_contract = {
            "adapter_version": "phase25.1",
            "dataset_id": dataset_id,
            "fold": fold,
            "kind": self.kind,
            "dtype": "float32",
            "feature_order_policy": "numeric input_columns order; then categorical input_columns/vocabulary order",
            "output_feature_names": self.adapter.output_feature_names
            if self.adapter
            else ["red", "green", "blue"],
            "class_names": labels or {},
            "source_label_to_id": label_to_id,
            "class_weights_by_id": {
                str(label_to_id[k]): v for k, v in self.preprocessing.get("class_weights", {}).items()
            },
            "canonical_layout": "NHWC"
            if self.kind == "image"
            else "NTF"
            if self.kind == "timeseries"
            else "NF",
            "pytorch_image_layout": "NCHW" if self.kind == "image" else None,
            "preprocessing": {
                "path": prep_path.relative_to(ROOT).as_posix(),
                "sha256": sha256_file(prep_path),
            },
        }
        self.provenance = {
            "dataset_sha256": read_json(ROOT / entry["split"]["path"])["dataset_sha256"],
            "split_sha256": entry["split"]["sha256"],
            "subset_sha256": sha256_file(self.split_dir / "matched_train.ids.jsonl.gz"),
            "preprocessing_sha256": sha256_file(prep_path),
            "adapter_sha256": canonical_sha256(self.adapter_contract),
        }
        self._frame = None
        self._targets = None
        self._source_ids = None
        self._index = None
        self._allowed = set(self.ids("train", matched=False)) | set(self.ids("validation"))

    def ids(self, partition, matched=True):
        if partition not in {"train", "validation"}:
            raise ValueError("TEST/quarantine is sealed; development loader accepts only train/validation")
        filename = "matched_train" if partition == "train" and matched else partition
        return read_rows(self.split_dir / f"{filename}.ids.jsonl.gz")

    def _load(self):
        if self._frame is not None:
            return
        if self.kind == "tabular":
            frame, target, ids, _, _ = load_tabular(self.dataset_id)
            self._frame, self._targets, self._source_ids = frame, target.to_numpy(), ids
        elif self.kind == "image":
            self._index = {row["id"]: row for row in read_rows(self.base / "index.jsonl.gz")}
            self._frame = pd.DataFrame()  # decoded lazily; no image tensor cache
        else:
            receipt = read_json(self.base / "series_cache.json")
            path = ROOT / receipt["path"]
            if path.exists():
                if sha256_file(path) != receipt["sha256"]:
                    raise ValueError("Series cache hash mismatch")
                self._frame = pd.read_csv(path, parse_dates=["time"])
            else:
                frame, _, _ = load_series(self.dataset_id)
                frame = frame.sort_values(["series", "time"]).reset_index(drop=True)
                # Match the frozen cache's decimal serialization before scaling.
                self._frame = pd.read_csv(
                    io.StringIO(
                        frame.to_csv(index=False, date_format="%Y-%m-%dT%H:%M:%S", float_format="%.12g")
                    ),
                    parse_dates=["time"],
                )
            self._index = {row["id"]: row for row in read_rows(self.base / "index.jsonl.gz")}

    def batch(self, ids, implementation):
        if implementation not in {"scratch", "keras", "pytorch"}:
            raise ValueError("Unknown implementation")
        if not ids or len(ids) != len(set(ids)) or any(value not in self._allowed for value in ids):
            raise ValueError("Unknown/duplicate/sealed batch IDs")
        self._load()
        if self.kind == "tabular":
            positions = select_rows(self._source_ids, ids)
            x = self.adapter.transform(self._frame.iloc[positions])
            y = self._targets[positions]
            y = self.adapter.target(y).reshape(-1, 1)
        elif self.kind == "image":
            rows = [self._index[key] for key in ids]
            images = np.stack(
                [
                    np.asarray(
                        image_for_row(
                            self.dataset_id, row, self.preprocessing["input_shape_hwc"][:2]
                        ).convert("RGB")
                    )
                    for row in rows
                ]
            )
            x = image_batch(images, implementation)
            y = np.asarray(
                [self.adapter_contract["source_label_to_id"][str(row["target"])] for row in rows],
                dtype=np.int64,
            )
        else:
            x, y = window_batch(self._frame, [self._index[key] for key in ids], self.adapter)
        return x, y

    def contract(self):
        return copy.deepcopy(self.adapter_contract)



### `src.common.model_core`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.common.model_core src/common/model_core.py both
"""One architecture plan supplies all backends, contracts and explanations.

Canonical activations: NHWC images, NTF sequences, dense IO, conv HWIO,
recurrent (input, gates*hidden)/(hidden, gates*hidden), LSTM i/f/g/o.
"""

from __future__ import annotations

import copy

import numpy as np

from src.common.contracts import PROTOCOL, canonical_sha256, read_json, validate_document

BACKENDS = {"scratch": "numpy", "keras": "tensorflow", "pytorch": "torch"}

LEARNED = {"dense", "conv2d", "rnn", "lstm"}



#### build_plan

In [ ]:
%%inline_module src.common.model_core src/common/model_core.py both
def build_plan(family, input_shape, outputs, *, tiny=False):
    protocol = read_json(PROTOCOL)
    if (
        family not in protocol["families"]
        or not input_shape
        or outputs < 1
        or any(not isinstance(v, int) or v < 1 for v in input_shape)
    ):
        raise ValueError("Invalid family/input/output dimensions")
    if len(input_shape) != {"mlp": 1, "cnn": 3, "rnn": 2, "lstm": 2}[family]:
        raise ValueError("Input rank mismatch")
    shape = list(input_shape)
    steps = []

    def add(kind, output, **options):
        nonlocal shape
        step = {
            "id": f"{kind}_{len(steps) + 1}",
            "kind": kind,
            "input_shape": shape,
            "output_shape": list(output),
            **options,
        }
        steps.append(step)
        shape = list(output)

    policy = protocol["families"][family]
    if family == "mlp":
        for units in [2] * 5 if tiny else policy["hidden_units"]:
            add("dense", [units])
            add("activation", shape, activation="relu")
    elif family == "cnn":
        for i, channels in enumerate([2] * 5 if tiny else policy["channels"]):
            add("conv2d", [*shape[:2], channels], kernel_size=3, padding="same")
            add("activation", shape, activation="relu")
            if i in policy["pool_after_convolutions"]:
                if min(shape[:2]) < 2:
                    raise ValueError("Image too small for two pooling stages")
                add("pooling", [shape[0] // 2, shape[1] // 2, shape[2]], pool="max2")
        add("pooling", [shape[-1]], pool="global_average")
        add("dense", [2 if tiny else policy["dense_hidden"]])
        add("activation", shape, activation="relu")
    else:
        for i, units in enumerate([2] * 3 if tiny else policy["hidden_units"]):
            add(family, [shape[0], units] if i < 2 else [units], return_sequences=i < 2)
        for units in [2, 2] if tiny else policy["dense_hidden"]:
            add("dense", [units])
            add("activation", shape, activation="relu")
    add("dense", [outputs])
    return {
        "plan_version": "phase25.1",
        "family": family,
        "input_shape": list(input_shape),
        "output_shape": shape,
        "steps": steps,
        "tiny_synthetic_only": tiny,
    }



#### parameter_shapes

In [ ]:
%%inline_module src.common.model_core src/common/model_core.py both
def parameter_shapes(step):
    kind = step["kind"]
    width = step["input_shape"][-1]
    units = step["output_shape"][-1]
    if kind == "dense":
        return {"w": (width, units), "b": (units,)}
    if kind == "conv2d":
        return {"w": (step["kernel_size"], step["kernel_size"], width, units), "b": (units,)}
    if kind in {"rnn", "lstm"}:
        gates = 4 if kind == "lstm" else 1
        return {"w": (width, gates * units), "u": (units, gates * units), "b": (gates * units,)}
    return {}



#### canonical_weights

In [ ]:
%%inline_module src.common.model_core src/common/model_core.py both
def canonical_weights(plan, seed, dtype="float32"):
    rng = np.random.default_rng(seed)
    result = {}
    for step in plan["steps"]:
        for name, shape in parameter_shapes(step).items():
            value = (
                np.zeros(shape, dtype=dtype)
                if name == "b"
                else rng.normal(0, np.sqrt(1 / np.prod(shape[:-1])), shape).astype(dtype)
            )
            if step["kind"] == "lstm" and name == "b":
                units = step["output_shape"][-1]
                value[units : 2 * units] = 1
            result[f"{step['id']}/{name}"] = value
    return result



#### formula_for

In [ ]:
%%inline_module src.common.model_core src/common/model_core.py both
def formula_for(step):
    kind = step["kind"]
    expressions = {
        "dense": "y = xW + b",
        "conv2d": "y = conv_same(x, W) + b",
        "rnn": "h_t = tanh(x_t W + h_previous U + b)",
        "lstm": "(i,f,g,o) = (sigmoid,sigmoid,tanh,sigmoid)(x_t W + h_previous U + b); c_t = f*c_previous + i*g; h_t = o*tanh(c_t)",
    }
    if kind not in expressions:
        return None
    meanings = {
        "x": "tensor đầu vào (x_t: tại bước t)",
        "y": "tensor đầu ra",
        "W": "trọng số đầu vào",
        "b": "bias học được",
    }
    if kind in {"rnn", "lstm"}:
        meanings.update(
            {
                "h_t": "trạng thái ở bước t",
                "h_previous": "trạng thái bước trước, bắt đầu bằng 0",
                "U": "trọng số trạng thái",
                "t": "chỉ số bước thời gian",
            }
        )
    if kind == "rnn":
        meanings["tanh"] = "hàm ép giá trị vào khoảng -1 đến 1"
    if kind == "conv2d":
        meanings["conv_same"] = "trượt bộ lọc 3×3, đệm 0 để giữ kích thước ảnh"
    if kind == "lstm":
        meanings.update(
            {
                "i,f,o": "cổng nhận, quên, xuất trong 0..1",
                "g": "ứng viên bộ nhớ",
                "c_t": "bộ nhớ mới",
                "c_previous": "bộ nhớ bước trước, bắt đầu bằng 0",
                "sigmoid": "hàm ép vào khoảng 0 đến 1",
                "tanh": "hàm ép vào khoảng -1 đến 1",
                "*": "nhân từng phần tử",
            }
        )
    return {
        "expression": expressions[kind],
        "symbols": [{"symbol": k, "meaning": v} for k, v in meanings.items()],
        "why_needed": "Đối chiếu phép tính forward trong code scratch.",
        "plain_explanation": {
            "dense": "Kết hợp các giá trị đầu vào bằng trọng số rồi cộng bias.",
            "conv2d": "Tìm đặc trưng cục bộ bằng bộ lọc trượt trên ảnh.",
            "rnn": "Kết hợp dữ liệu hiện tại và thông tin nhớ từ bước trước.",
            "lstm": "Chọn thông tin cần giữ, bổ sung và xuất qua các cổng.",
        }[kind],
        "example": f"Shape {step['input_shape']} → {step['output_shape']} cho một mẫu; batch được giữ nguyên.",
    }



#### make_contract

In [ ]:
%%inline_module src.common.model_core src/common/model_core.py both
def make_contract(plan, implementation, evidence_kind, provenance=None):
    if implementation not in BACKENDS:
        raise ValueError("Unknown implementation")
    if evidence_kind != "synthetic_fixture" and provenance is None:
        raise ValueError("Measured contracts require provenance")
    if plan["tiny_synthetic_only"] and evidence_kind != "synthetic_fixture":
        raise ValueError("Tiny plan is not a production model")
    family = plan["family"]
    dataset = {
        "mlp": "ch2_cdc_diabetes",
        "cnn": "ch3_cifar10",
        "rnn": "ch4_gold_pink_sheet",
        "lstm": "ch4_gold_pink_sheet",
    }[family]
    digest = canonical_sha256(plan)
    provenance = provenance or {
        "dataset_id": dataset,
        "config_sha256": digest,
        "dataset_sha256": digest,
        "comparison_group_sha256": digest,
        "preprocessing": {
            "path": "artifacts/manifests/phase25/synthetic_preprocessing.json",
            "sha256": digest,
        },
    }
    layers = []
    functions = []
    keras_names = {
        "dense": "Dense",
        "conv2d": "Conv2D",
        "rnn": "SimpleRNN",
        "lstm": "LSTM",
        "activation": "ReLU",
        "pooling": "MaxPooling2D/GlobalAveragePooling2D",
    }
    torch_names = {
        "dense": "Linear",
        "conv2d": "Conv2d",
        "rnn": "RNN",
        "lstm": "LSTM",
        "activation": "ReLU",
        "pooling": "MaxPool2d/AdaptiveAvgPool2d",
    }
    roles = {
        "dense": "Kết hợp các đặc trưng",
        "conv2d": "Tìm đặc trưng trong vùng ảnh",
        "rnn": "Nhớ thông tin theo thời gian",
        "lstm": "Giữ/quên thông tin bằng bộ nhớ và cổng",
        "activation": "Giữ tín hiệu dương, tạo khả năng học phi tuyến",
        "pooling": "Rút gọn vùng ảnh",
    }
    for step in plan["steps"]:
        count = int(sum(np.prod(shape) for shape in parameter_shapes(step).values()))
        inp = ["batch", *step["input_shape"]]
        out = ["batch", *step["output_shape"]]
        layers.append(
            {
                "id": step["id"],
                "kind": step["kind"],
                "parameter_count": count,
                "input_shape": inp,
                "output_shape": out,
                "function_id": step["id"],
            }
        )
        functions.append(
            {
                "id": step["id"],
                "signature": f"{step['kind']}(x, weights) -> y",
                "role": roles[step["kind"]],
                "input_shape": inp,
                "output_shape": out,
                "parameters": count,
                "corner_case": "Sai rank/shape bị từ chối; trạng thái RNN/LSTM reset giữa các window; không bỏ batch cuối.",
                "formula": formula_for(step),
                "mapping": {
                    "scratch": f"src.common.numpy_engine.NumpyNetwork.forward[{step['kind']}]",
                    "keras": f"keras.layers.{keras_names[step['kind']]}",
                    "pytorch": f"torch.nn.{torch_names[step['kind']]}",
                },
            }
        )
    contract = {
        "schema_version": "3.0.0",
        "evidence_kind": evidence_kind,
        "model_id": f"{provenance['dataset_id']}_{family}_{implementation}",
        "dataset_id": provenance["dataset_id"],
        "family": family,
        "implementation": implementation,
        "backend": BACKENDS[implementation],
        "parameter_count": sum(x["parameter_count"] for x in layers),
        "input_shape": ["batch", *plan["input_shape"]],
        "output_shape": ["batch", *plan["output_shape"]],
        "layers": layers,
        "functions": functions,
        "composition": [x["id"] for x in functions],
        "composition_plain": "đầu vào → "
        + " → ".join(f"{roles[x['kind']]} {x['output_shape']}" for x in plan["steps"])
        + " → đầu ra (logit/giá trị chuẩn hóa)",
        "recurrent_unroll": "Ở mỗi tầng l=1..3, lặp t=1..T theo thứ tự; khởi tạo h (và c) bằng 0 cho mỗi window. Tầng 1–2 trả toàn chuỗi, tầng 3 trả bước cuối → 3 Dense."
        if family in {"rnn", "lstm"}
        else None,
        **{
            key: provenance[key]
            for key in ("config_sha256", "dataset_sha256", "comparison_group_sha256", "preprocessing")
        },
    }
    validate_model(contract)
    return contract



#### validate_model

In [ ]:
%%inline_module src.common.model_core src/common/model_core.py both
def validate_model(contract):
    validate_document("model", contract)
    functions = {row["id"]: row for row in contract["functions"]}
    if contract["composition"] != [row["function_id"] for row in contract["layers"]]:
        raise ValueError("Composition order differs from execution")
    previous = contract["input_shape"]
    for layer in contract["layers"]:
        function = functions[layer["function_id"]]
        if (
            layer["input_shape"] != previous
            or any(function[k] != layer[k] for k in ("input_shape", "output_shape"))
            or function["parameters"] != layer["parameter_count"]
        ):
            raise ValueError("Shape/function/parameter trace mismatch")
        kind = layer["kind"]
        width = layer["input_shape"][-1]
        units = layer["output_shape"][-1]
        expected = 0
        if kind == "dense":
            expected = (width + 1) * units
        elif kind == "conv2d":
            expected = (9 * width + 1) * units
        elif kind in {"rnn", "lstm"}:
            expected = (width + units + 1) * units * (4 if kind == "lstm" else 1)
        if layer["parameter_count"] != expected:
            raise ValueError("Parameter count does not match tensor dimensions")
        previous = layer["output_shape"]
    if previous != contract["output_shape"]:
        raise ValueError("Final output shape mismatch")



#### render_functions

In [ ]:
%%inline_module src.common.model_core src/common/model_core.py both
def render_functions(contract):
    validate_model(contract)
    functions = copy.deepcopy(contract["functions"])
    technical = "F = " + " ∘ ".join(reversed(contract["composition"]))
    table = "| Hàm | Vai trò | Input → Output | Tham số |\n|---|---|---|---:|\n"
    table += "".join(
        f"| {f['id']} | {f['role']} | {f['input_shape']} → {f['output_shape']} | {f['parameters']} |\n"
        for f in functions
    )
    markdown = f"# {contract['model_id']}\n\n{contract['composition_plain']}\n\n`{technical}`\n\n{contract['recurrent_unroll'] or ''}\n\n{table}"
    for function in functions:
        markdown += f"\n## {function['id']}\n\n`{function['signature']}` — {function['role']}.\n\n{function['corner_case']}\n\nMapping: {function['mapping']}\n"
        formula = function["formula"]
        if formula:
            markdown += (
                f"\n{formula['plain_explanation']}\n\n`{formula['expression']}`\n\n"
                + "; ".join(f"{s['symbol']}: {s['meaning']}" for s in formula["symbols"])
                + f".\n\n{formula['example']}\n"
            )
    return {
        "technical": technical,
        "plain": contract["composition_plain"],
        "functions": functions,
        "markdown": markdown,
    }



#### ModelRegistry

In [ ]:
%%inline_module src.common.model_core src/common/model_core.py both
class ModelRegistry:
    def __init__(self):
        self._models = {}

    def register(self, contract):
        validate_model(contract)
        if contract["model_id"] in self._models:
            raise ValueError("Duplicate model ID")
        self._models[contract["model_id"]] = copy.deepcopy(contract)

    def get(self, model_id):
        return copy.deepcopy(self._models[model_id])



### `scripts.prepare_phase25`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module scripts.prepare_phase25 scripts/prepare_phase25.py both
"""Generate initialized architecture contracts, never trained-model claims."""

from __future__ import annotations

import csv

import io

import json

from jsonschema import Draft202012Validator

from src.common.artifact_core import code_fingerprint, sha256_file

from src.common.contracts import PROTOCOL, ROOT, canonical_sha256, read_json

from src.common.data_access import DatasetView

from src.common.eda_artifacts import reference, write_json

from src.common.model_core import build_plan, canonical_weights, make_contract, render_functions

BASE = ROOT / "artifacts/manifests/phase25"

FUNCTIONS = ROOT / "report/references/functions/v3"

CORE_FILES = [
    f"src/common/{name}.py"
    for name in (
        "artifact_core",
        "backend_models",
        "data_access",
        "device_core",
        "metric_core",
        "model_core",
        "native_training",
        "numpy_engine",
        "parity",
        "tensor_adapters",
        "training_core",
    )
]



#### validate_adapter

In [ ]:
%%inline_module scripts.prepare_phase25 scripts/prepare_phase25.py both
def validate_adapter(document):
    schema = read_json(ROOT / "src/common/schemas/phase25/tensor_adapter.schema.json")
    errors = list(Draft202012Validator(schema).iter_errors(document))
    if errors:
        raise ValueError(errors[0].message)
    classes = document["class_names"]
    mapping = document["source_label_to_id"]
    if classes and (
        set(classes) != {str(i) for i in range(len(classes))}
        or set(mapping.values()) != set(range(len(classes)))
    ):
        raise ValueError("Label mapping is not contiguous/bijective")



#### architecture_for

In [ ]:
%%inline_module scripts.prepare_phase25 scripts/prepare_phase25.py both
def architecture_for(view, family):
    if view.kind == "image":
        shape = view.preprocessing["input_shape_hwc"]
        outputs = view.spec["class_count"]
    elif view.kind == "timeseries":
        shape = [view.spec["lookback"], view.preprocessing["output_width"]]
        outputs = len(view.spec["horizons"])
    else:
        shape = [view.preprocessing["output_width"]]
        outputs = 1
    return build_plan(family, shape, outputs)



#### prepare

In [ ]:
%%inline_module scripts.prepare_phase25 scripts/prepare_phase25.py both
def prepare():
    if (BASE / "core_lock.json").exists():
        from scripts.verify_phase25 import verify

        return verify()
    protocol = read_json(PROTOCOL)
    entries = []
    adapters = []
    summary = []
    code_sha = code_fingerprint(ROOT, CORE_FILES)
    for row in protocol["datasets"]:
        if row["chapter"] not in {2, 3, 4}:
            continue
        view = DatasetView(row["dataset_id"])
        adapter = view.contract()
        validate_adapter(adapter)
        adapter_path = BASE / "adapters" / f"{view.dataset_id}.json"
        write_json(adapter_path, adapter)
        adapters.append(reference(adapter_path))
        for family in {2: ["mlp"], 3: ["cnn"], 4: ["rnn", "lstm"]}[row["chapter"]]:
            plan = architecture_for(view, family)
            Draft202012Validator(
                read_json(ROOT / "src/common/schemas/phase25/training_plan.schema.json")
            ).validate(plan)
            key = f"{view.dataset_id}_{family}"
            plan_path = BASE / "plans" / f"{key}.json"
            write_json(plan_path, plan)
            init_hashes = {
                str(seed): canonical_sha256({k: v.tolist() for k, v in canonical_weights(plan, seed).items()})
                for seed in protocol["seeds"]["stability"]
            }
            group = {
                "dataset_id": view.dataset_id,
                "family": family,
                "plan_sha256": canonical_sha256(plan),
                "adapter": reference(adapter_path),
                **view.provenance,
                "protocol_sha256": sha256_file(PROTOCOL),
            }
            group_hash = canonical_sha256(group)
            config = {
                "phase": 25,
                "lifecycle": "initialized_only",
                "inference_ready": False,
                "dataset_id": view.dataset_id,
                "family": family,
                "plan": reference(plan_path),
                "adapter": reference(adapter_path),
                "comparison_group_sha256": group_hash,
                "canonical_initial_weights_by_seed_sha256": init_hashes,
                "provenance": view.provenance,
                "code_sha256": code_sha,
                "protocol_sha256": sha256_file(PROTOCOL),
                "environment_sha256": sha256_file(ROOT / "uv.lock"),
                "training": protocol["training"],
                "seeds": protocol["seeds"]["stability"],
                "real_dataset_training_executed": False,
                "test_evaluation_executed": False,
            }
            config_path = BASE / "configs" / f"{key}.json"
            write_json(config_path, config)
            for implementation in protocol["implementation_order"]:
                provenance = {
                    "dataset_id": view.dataset_id,
                    "config_sha256": sha256_file(config_path),
                    "dataset_sha256": view.provenance["dataset_sha256"],
                    "comparison_group_sha256": group_hash,
                    "preprocessing": {k: adapter["preprocessing"][k] for k in ("path", "sha256")},
                }
                contract = make_contract(plan, implementation, "measured", provenance)
                directory = BASE / "contracts" / contract["model_id"]
                contract_path = directory / "model_contract.json"
                write_json(contract_path, contract)
                rendered = render_functions(contract)
                write_json(
                    directory / "functions.json", {k: v for k, v in rendered.items() if k != "markdown"}
                )
                FUNCTIONS.mkdir(parents=True, exist_ok=True)
                (FUNCTIONS / f"{contract['model_id']}.md").write_text(
                    rendered["markdown"], encoding="utf-8", newline="\n"
                )
                buffer = io.StringIO(newline="")
                writer = csv.writer(buffer, lineterminator="\n")
                writer.writerow(
                    [
                        "function",
                        "signature",
                        "role",
                        "input_shape",
                        "output_shape",
                        "parameters",
                        "corner_case",
                        "scratch",
                        "keras",
                        "pytorch",
                    ]
                )
                for f in rendered["functions"]:
                    writer.writerow(
                        [
                            f["id"],
                            f["signature"],
                            f["role"],
                            f["input_shape"],
                            f["output_shape"],
                            f["parameters"],
                            f["corner_case"],
                            *f["mapping"].values(),
                        ]
                    )
                (directory / "functions.csv").write_text(buffer.getvalue(), encoding="utf-8", newline="\n")
                entries.append(
                    {
                        "model_id": contract["model_id"],
                        "dataset_id": view.dataset_id,
                        "family": family,
                        "implementation": implementation,
                        "lifecycle": "initialized_only",
                        "inference_ready": False,
                        "contract": reference(contract_path),
                        "config": reference(config_path),
                        "adapter": reference(adapter_path),
                    }
                )
            summary.append(
                {
                    "dataset_id": view.dataset_id,
                    "family": family,
                    "input_shape": plan["input_shape"],
                    "output_shape": plan["output_shape"],
                    "learnable_layers": len([s for s in contract["layers"] if s["parameter_count"]]),
                    "parameter_count": contract["parameter_count"],
                    "initial_weight_hashes": init_hashes,
                }
            )
    registry = {
        "phase": 25,
        "lifecycle": "initialized_only",
        "inference_ready": False,
        "adapters": adapters,
        "models": entries,
        "architecture_summary": summary,
        "code_sha256": code_sha,
        "code_files": [{"path": p, "sha256": sha256_file(ROOT / p)} for p in CORE_FILES],
        "real_dataset_training_executed": False,
        "test_evaluation_executed": False,
    }
    write_json(BASE / "initialized_contract_registry.json", registry)
    return {
        "status": "PASS",
        "adapters": len(adapters),
        "model_contracts": len(entries),
        "architecture_plans": len(summary),
        "real_dataset_training_executed": False,
        "test_evaluation_executed": False,
    }



#### main

In [ ]:
%%inline_module scripts.prepare_phase25 scripts/prepare_phase25.py both
def main():
    print(json.dumps(prepare(), ensure_ascii=False, indent=2))



#### Thiết lập / điểm vào

In [ ]:
%%inline_module scripts.prepare_phase25 scripts/prepare_phase25.py both
if __name__ == "__main__":
    main()



### `scripts.verify_phase25`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module scripts.verify_phase25 scripts/verify_phase25.py both
"""Read-only acceptance gates for Phase25; no model TEST execution."""

from __future__ import annotations

import argparse

import csv

import json

import xml.etree.ElementTree as ET

from datetime import UTC, datetime

from scripts.prepare_phase25 import BASE, CORE_FILES, FUNCTIONS, architecture_for, validate_adapter

from src.common.artifact_core import code_fingerprint, sha256_file, write_once

from src.common.contracts import PROTOCOL, ROOT, canonical_sha256, read_json

from src.common.data_access import DatasetView

from src.common.eda_artifacts import reference, write_json

from src.common.model_core import canonical_weights, validate_model



#### verify_files

In [ ]:
%%inline_module scripts.verify_phase25 scripts/verify_phase25.py both
def verify_files(root, records):
    for row in records:
        if sha256_file(root / row["path"]) != row["sha256"]:
            raise ValueError(f"Locked hash mismatch: {row['path']}")
    return len(records)



#### check_xml

In [ ]:
%%inline_module scripts.verify_phase25 scripts/verify_phase25.py both
def check_xml(path):
    suites = list(ET.parse(path).getroot().iter("testsuite"))
    if not suites or any(
        int(row.attrib.get(k, 0)) for row in suites for k in ("failures", "errors", "skipped")
    ):
        raise ValueError("Test evidence has failures/errors/skips")
    total = sum(int(row.attrib["tests"]) for row in suites)
    if total < 1:
        raise ValueError("Empty test evidence")
    return total



#### verify

In [ ]:
%%inline_module scripts.verify_phase25 scripts/verify_phase25.py both
def verify(create_lock=False):
    protected = {}
    for phase, filename in ((22, "spec_lock.json"), (24, "data_lock.json")):
        lock = read_json(ROOT / f"artifacts/manifests/phase{phase}/{filename}")
        protected[str(phase)] = verify_files(ROOT, lock["files"])
    protocol = read_json(PROTOCOL)
    registry = read_json(BASE / "initialized_contract_registry.json")
    if registry["code_sha256"] != code_fingerprint(ROOT, CORE_FILES):
        raise ValueError("Core changed after contract generation")
    verify_files(ROOT, registry["code_files"])
    if (
        registry["inference_ready"]
        or registry["lifecycle"] != "initialized_only"
        or len(registry["models"]) != 36
        or len(registry["adapters"]) != 9
    ):
        raise ValueError("Wrong registry lifecycle/count")
    seen = set()
    groups = {}
    summary = {}
    for ref in registry["adapters"]:
        verify_files(ROOT, [ref])
        document = read_json(ROOT / ref["path"])
        validate_adapter(document)
        if document != DatasetView(document["dataset_id"]).contract():
            raise ValueError("Adapter differs from locked data state")
    for entry in registry["models"]:
        verify_files(ROOT, [entry["contract"], entry["config"], entry["adapter"]])
        contract = read_json(ROOT / entry["contract"]["path"])
        validate_model(contract)
        if contract["model_id"] in seen or entry["inference_ready"]:
            raise ValueError("Duplicate/incorrectly published model")
        seen.add(contract["model_id"])
        view = DatasetView(entry["dataset_id"])
        plan = architecture_for(view, entry["family"])
        config = read_json(ROOT / entry["config"]["path"])
        for seed in protocol["seeds"]["stability"]:
            digest = canonical_sha256({k: v.tolist() for k, v in canonical_weights(plan, seed).items()})
            if config["canonical_initial_weights_by_seed_sha256"][str(seed)] != digest:
                raise ValueError("Canonical initialization changed")
        if (
            contract["config_sha256"] != entry["config"]["sha256"]
            or contract["dataset_sha256"] != view.provenance["dataset_sha256"]
        ):
            raise ValueError("Model provenance mismatch")
        key = (entry["dataset_id"], entry["family"])
        groups.setdefault(key, []).append(
            (contract["implementation"], contract["comparison_group_sha256"], contract["parameter_count"])
        )
        summary[key] = contract["parameter_count"]
        folder = (ROOT / entry["contract"]["path"]).parent
        if read_json(folder / "functions.json")["plain"] != contract["composition_plain"]:
            raise ValueError("Function renderer differs")
        with (folder / "functions.csv").open(encoding="utf-8", newline="") as stream:
            if len(list(csv.DictReader(stream))) != len(contract["functions"]):
                raise ValueError("Function CSV row mismatch")
    if len(groups) != 12 or any(
        {v[0] for v in values} != {"scratch", "keras", "pytorch"}
        or len({v[1] for v in values}) != 1
        or len({v[2] for v in values}) != 1
        for values in groups.values()
    ):
        raise ValueError("Comparison groups differ")
    probes = {}
    for implementation in ("keras", "pytorch"):
        for device in ("cpu", "gpu"):
            probe = read_json(BASE / f"probe_{implementation}_{device}.json")
            if probe["code_sha256"] != registry["code_sha256"] or probe["probe_code_sha256"] != sha256_file(
                ROOT / "scripts/phase25_backend_probe.py"
            ):
                raise ValueError("Probe ran different code")
            if (
                probe["status"] != "PASS"
                or len(probe["parity"]) != 4
                or probe["real_dataset_training_executed"]
                or probe["test_evaluation_executed"]
            ):
                raise ValueError("Probe did not pass safely")
            if probe["protocol_sha256"] != sha256_file(PROTOCOL) or probe[
                "phase24_lock_sha256"
            ] != sha256_file(ROOT / "artifacts/manifests/phase24/data_lock.json"):
                raise ValueError("Probe dependency mismatch")
            for row in probe["parity"]:
                if (
                    row["gradient_parity"]["max_relative_l2_error"]
                    > protocol["comparison"]["numeric_parity"]["gradient_relative_error"]
                ):
                    raise ValueError("Gradient parity above locked tolerance")
            if device == "gpu":
                if len(probe["memory_probes"]) != 12:
                    raise ValueError("Missing real-dimension GPU probes")
                for row in probe["memory_probes"]:
                    if (
                        row["parameter_count"] != summary[(row["dataset_id"], row["family"])]
                        or not 0 < row["peak_vram_bytes"] < 8 * 1024**3
                    ):
                        raise ValueError("Memory/parameter probe mismatch")
                    if not any(tag in row["actual_device"].lower() for tag in ("gpu", "cuda")):
                        raise ValueError("GPU placement mismatch")
            probes[f"{implementation}_{device}"] = {
                "parity_families": len(probe["parity"]),
                "dimension_probes": len(probe["memory_probes"]),
                "max_peak_vram_bytes": max([r["peak_vram_bytes"] for r in probe["memory_probes"]], default=0),
            }
    tests = {
        name: check_xml(BASE / filename)
        for name, filename in (
            ("core_torch", "core-tests-torch.xml"),
            ("core_keras", "core-tests-keras.xml"),
            ("regression", "regression-tests.xml"),
        )
    }
    coverage = read_json(BASE / "core_coverage_combined.json")
    module_coverage = {name: row["summary"]["percent_covered"] for name, row in coverage["files"].items()}
    if len(module_coverage) != len(CORE_FILES) or min(module_coverage.values()) < 80:
        raise ValueError(f"Core module coverage below 80%: {module_coverage}")
    seal = read_json(ROOT / "artifacts/manifests/ch5/phase15_test_run.json")
    if seal["run_count"] != 1:
        raise ValueError("Ch5 TEST run count changed")
    files = []
    for name in ("adapters", "configs", "contracts", "plans"):
        files.extend((BASE / name).rglob("*.*"))
    files += list(FUNCTIONS.glob("*.md")) + [ROOT / path for path in CORE_FILES]
    files += list((ROOT / "tests/core").glob("*.py")) + list(
        (ROOT / "src/common/schemas/phase25").glob("*.json")
    )
    files += [
        ROOT / "scripts" / name
        for name in ("prepare_phase25.py", "phase25_backend_probe.py", "verify_phase25.py")
    ]
    files += [
        ROOT / "experiments/configs/v3/phase25_coverage.ini",
        BASE / "initialized_contract_registry.json",
    ]
    files += [ROOT / "scripts/check_phase25.ps1", ROOT / "docs/model_core_v3.md"]
    files += [BASE / f"probe_{i}_{d}.json" for i in ("keras", "pytorch") for d in ("cpu", "gpu")]
    files += [
        BASE / filename
        for filename in (
            "core-tests-torch.xml",
            "core-tests-keras.xml",
            "regression-tests.xml",
            "core_coverage_combined.json",
        )
    ]
    records = [reference(path) for path in sorted(set(files))]
    lock_path = BASE / "core_lock.json"
    if lock_path.exists():
        if read_json(lock_path)["files"] != records:
            raise ValueError("Phase25 changed after lock; create a versioned revision")
    elif create_lock:
        write_once(
            lock_path,
            {
                "phase": 25,
                "files": records,
                "phase22_lock_sha256": sha256_file(ROOT / "artifacts/manifests/phase22/spec_lock.json"),
                "phase24_lock_sha256": sha256_file(ROOT / "artifacts/manifests/phase24/data_lock.json"),
                "real_dataset_training_executed": False,
                "test_evaluation_executed": False,
            },
        )
    return {
        "phase": 25,
        "status": "PASS",
        "at": datetime.now(UTC).isoformat(),
        "model_contracts": 36,
        "adapters": 9,
        "architecture_plans": 12,
        "canonical_seed_initializations": 36,
        "protected_files": protected,
        "tests": tests,
        "coverage_percent": coverage["totals"]["percent_covered"],
        "module_coverage": module_coverage,
        "probes": probes,
        "ch5_test_run_count": 1,
        "locked_files": len(records),
        "core_lock_created": lock_path.exists(),
        "real_dataset_training_executed": False,
        "test_evaluation_executed": False,
    }



#### main

In [ ]:
%%inline_module scripts.verify_phase25 scripts/verify_phase25.py both
def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--lock", action="store_true")
    args = parser.parse_args()
    try:
        result = verify(args.lock)
    except (OSError, ValueError, KeyError, IndexError) as error:
        result = {"phase": 25, "status": "FAIL", "error": f"{type(error).__name__}: {error}"}
    write_json(BASE / "verification.json", result)
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result["status"] != "PASS"



#### Thiết lập / điểm vào

In [ ]:
%%inline_module scripts.verify_phase25 scripts/verify_phase25.py both
if __name__ == "__main__":
    raise SystemExit(main())



### `src.common.numpy_engine`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.common.numpy_engine src/common/numpy_engine.py both
"""Educational NumPy forward/backward primitives, without autograd/frameworks.

This reference engine is first verified on synthetic data in Phase25. Real
dataset training/evaluation is implemented and accepted in Phases26–29.
"""

from __future__ import annotations

import numpy as np

from src.common.model_core import parameter_shapes



#### sigmoid

In [ ]:
%%inline_module src.common.numpy_engine src/common/numpy_engine.py both
def sigmoid(value):
    # Equivalent stable expression, without clipping the mathematical derivative.
    return np.exp(-np.logaddexp(0, -value))



#### loss_and_gradient

In [ ]:
%%inline_module src.common.numpy_engine src/common/numpy_engine.py both
def loss_and_gradient(logits, target, task, sample_weight=None):
    logits = np.asarray(logits)
    target = np.asarray(target)
    if logits.ndim != 2 or len(logits) == 0 or not np.isfinite(logits).all() or not np.isfinite(target).all():
        raise ValueError("Loss requires finite, nonempty [batch, outputs] logits")
    if task == "multiclass":
        if (
            target.shape != (len(logits),)
            or not np.equal(target, np.floor(target)).all()
            or np.any(target < 0)
            or np.any(target >= logits.shape[1])
        ):
            raise ValueError("Invalid class IDs")
        labels = target.astype(int)
        shift = logits - logits.max(axis=1, keepdims=True)
        probs = np.exp(shift)
        probs /= probs.sum(axis=1, keepdims=True)
        rows = np.log(np.exp(shift).sum(axis=1)) - shift[np.arange(len(logits)), labels]
        gradient = probs.copy()
        gradient[np.arange(len(logits)), labels] -= 1
    elif task in {"mse", "binary"}:
        if target.shape != logits.shape:
            raise ValueError("Target shape mismatch")
        if task == "binary":
            if not np.isin(target, [0, 1]).all():
                raise ValueError("Binary labels must be zero or one")
            rows = (np.logaddexp(0, logits) - target * logits).mean(axis=1)
            gradient = (sigmoid(logits) - target) / logits.shape[1]
        else:
            rows = np.mean((logits - target) ** 2, axis=1)
            gradient = 2 * (logits - target) / logits.shape[1]
    else:
        raise ValueError("Unknown loss")
    weights = np.ones(len(logits)) if sample_weight is None else np.asarray(sample_weight, dtype=float)
    if (
        weights.shape != (len(logits),)
        or not np.isfinite(weights).all()
        or np.any(weights < 0)
        or weights.sum() <= 0
    ):
        raise ValueError("Invalid sample weights")
    normalized = weights / weights.sum()
    return float(rows @ normalized), (gradient * normalized[:, None]).astype(logits.dtype)



#### numerical_gradient

In [ ]:
%%inline_module src.common.numpy_engine src/common/numpy_engine.py both
def numerical_gradient(function, value, epsilon=1e-5):
    if value.dtype != np.float64 or epsilon <= 0:
        raise ValueError("Numerical checks require float64 and positive epsilon")
    gradient = np.zeros_like(value)
    for index in np.ndindex(value.shape):
        previous = value[index]
        try:
            value[index] = previous + epsilon
            positive = function()
            value[index] = previous - epsilon
            negative = function()
        finally:
            value[index] = previous
        gradient[index] = (positive - negative) / (2 * epsilon)
    return gradient



#### NumpyNetwork

In [ ]:
%%inline_module src.common.numpy_engine src/common/numpy_engine.py both
class NumpyNetwork:
    def __init__(self, plan, weights):
        self.plan = plan
        expected = {
            f"{s['id']}/{k}": shape for s in plan["steps"] for k, shape in parameter_shapes(s).items()
        }
        if set(expected) != set(weights) or any(
            weights[k].shape != shape or not np.isfinite(weights[k]).all() for k, shape in expected.items()
        ):
            raise ValueError("Canonical weight names/shapes/values mismatch")
        self.weights = {key: value.copy() for key, value in weights.items()}
        self.cache = []

    def forward(self, x, *, trace=False):
        x = np.asarray(x, dtype=next(iter(self.weights.values())).dtype)
        if (
            x.ndim != len(self.plan["input_shape"]) + 1
            or list(x.shape[1:]) != self.plan["input_shape"]
            or len(x) == 0
            or not np.isfinite(x).all()
        ):
            raise ValueError("Input tensor shape/value mismatch")
        self.cache = []
        traces = []
        for step in self.plan["steps"]:
            before = x
            kind = step["kind"]
            prefix = step["id"]
            params = {key: self.weights[f"{prefix}/{key}"] for key in parameter_shapes(step)}
            extra = None
            if kind == "dense":
                x = before @ params["w"] + params["b"]
            elif kind == "activation":
                x = np.maximum(before, 0)
            elif kind == "conv2d":
                padded = np.pad(before, ((0, 0), (1, 1), (1, 1), (0, 0)))
                x = np.zeros((len(before), *step["output_shape"]), dtype=before.dtype)
                for i in range(3):
                    for j in range(3):
                        x += np.einsum(
                            "nhwc,co->nhwo",
                            padded[:, i : i + before.shape[1], j : j + before.shape[2]],
                            params["w"][i, j],
                        )
                x += params["b"]
                extra = padded
            elif kind == "pooling":
                if step["pool"] == "global_average":
                    x = before.mean(axis=(1, 2))
                else:
                    n, h, w, c = before.shape
                    hh, ww = h // 2, w // 2
                    patches = (
                        before[:, : hh * 2, : ww * 2]
                        .reshape(n, hh, 2, ww, 2, c)
                        .transpose(0, 1, 3, 5, 2, 4)
                        .reshape(n, hh, ww, c, 4)
                    )
                    extra = patches.argmax(axis=-1)
                    x = patches.max(axis=-1)
            else:
                units = step["output_shape"][-1]
                h = np.zeros((len(before), units), dtype=before.dtype)
                c = np.zeros_like(h)
                states = []
                outputs = []
                for t in range(before.shape[1]):
                    z = before[:, t] @ params["w"] + h @ params["u"] + params["b"]
                    if kind == "rnn":
                        next_h = np.tanh(z)
                        states.append((before[:, t], h, next_h))
                        h = next_h
                    else:
                        zi, zf, zg, zo = np.split(z, 4, axis=1)
                        i, f, g, o = sigmoid(zi), sigmoid(zf), np.tanh(zg), sigmoid(zo)
                        next_c = f * c + i * g
                        next_h = o * np.tanh(next_c)
                        states.append((before[:, t], h, c, i, f, g, o, next_c))
                        h, c = next_h, next_c
                    outputs.append(h)
                full = np.stack(outputs, axis=1)
                x = full if step["return_sequences"] else full[:, -1]
                extra = states
            self.cache.append((step, before, params, extra))
            traces.append({"id": prefix, "input_shape": list(before.shape), "output_shape": list(x.shape)})
        return (x, traces) if trace else x

    def backward(self, gradient):
        gradients = {}
        for step, before, params, extra in reversed(self.cache):
            kind = step["kind"]
            prefix = step["id"]
            grads = {key: np.zeros_like(value) for key, value in params.items()}
            if kind == "dense":
                grads["w"] = before.T @ gradient
                grads["b"] = gradient.sum(axis=0)
                gradient = gradient @ params["w"].T
            elif kind == "activation":
                gradient = gradient * (before > 0)
            elif kind == "conv2d":
                grads["b"] = gradient.sum(axis=(0, 1, 2))
                padded = extra
                dx = np.zeros_like(padded)
                for i in range(3):
                    for j in range(3):
                        patch = padded[:, i : i + before.shape[1], j : j + before.shape[2]]
                        grads["w"][i, j] = np.einsum("nhwc,nhwo->co", patch, gradient)
                        dx[:, i : i + before.shape[1], j : j + before.shape[2]] += np.einsum(
                            "nhwo,co->nhwc", gradient, params["w"][i, j]
                        )
                gradient = dx[:, 1:-1, 1:-1]
            elif kind == "pooling":
                if step["pool"] == "global_average":
                    gradient = np.broadcast_to(
                        gradient[:, None, None, :] / (before.shape[1] * before.shape[2]), before.shape
                    ).copy()
                else:
                    n, h, w, c = before.shape
                    hh, ww = h // 2, w // 2
                    patches = np.zeros((n, hh, ww, c, 4), dtype=before.dtype)
                    np.put_along_axis(patches, extra[..., None], gradient[..., None], axis=-1)
                    dx = np.zeros_like(before)
                    dx[:, : hh * 2, : ww * 2] = (
                        patches.reshape(n, hh, ww, c, 2, 2)
                        .transpose(0, 1, 4, 2, 5, 3)
                        .reshape(n, hh * 2, ww * 2, c)
                    )
                    gradient = dx
            else:
                full_gradient = np.zeros(
                    (len(before), before.shape[1], step["output_shape"][-1]), dtype=before.dtype
                )
                if step["return_sequences"]:
                    full_gradient += gradient
                else:
                    full_gradient[:, -1] = gradient
                dh = np.zeros_like(full_gradient[:, 0])
                dc = np.zeros_like(dh)
                dx = np.zeros_like(before)
                for t in reversed(range(before.shape[1])):
                    dh = dh + full_gradient[:, t]
                    if kind == "rnn":
                        xt, previous_h, h = extra[t]
                        dz = dh * (1 - h * h)
                    else:
                        xt, previous_h, previous_c, i, f, g, o, c = extra[t]
                        tanhc = np.tanh(c)
                        dc = dc + dh * o * (1 - tanhc * tanhc)
                        dz = np.concatenate(
                            (
                                dc * g * i * (1 - i),
                                dc * previous_c * f * (1 - f),
                                dc * i * (1 - g * g),
                                dh * tanhc * o * (1 - o),
                            ),
                            axis=1,
                        )
                        dc = dc * f
                    grads["w"] += xt.T @ dz
                    grads["u"] += previous_h.T @ dz
                    grads["b"] += dz.sum(axis=0)
                    dx[:, t] = dz @ params["w"].T
                    dh = dz @ params["u"].T
                gradient = dx
            gradients.update({f"{prefix}/{name}": value for name, value in grads.items()})
        return gradients, gradient

    def loss_gradients(self, x, y, task, sample_weight=None):
        logits = self.forward(x)
        loss, gradient = loss_and_gradient(logits, y, task, sample_weight)
        gradients, dx = self.backward(gradient)
        return loss, gradients, dx



### `src.ch02_ml.baselines`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch02_ml.baselines src/ch02_ml/baselines.py both
"""Classical baselines exported as numeric data, never executable pickle."""

from __future__ import annotations

import numpy as np

from sklearn.dummy import DummyClassifier, DummyRegressor

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

from sklearn.linear_model import LogisticRegression, Ridge

from src.common.numpy_engine import sigmoid



#### fit_baseline

In [ ]:
%%inline_module src.ch02_ml.baselines src/ch02_ml/baselines.py both
def fit_baseline(name, x, y, task, *, seed, settings=None, class_weights=None):
    settings = settings or {"n_estimators": 200, "max_depth": 12, "min_samples_leaf": 5, "n_jobs": 2}
    if task not in {"mse", "binary"}:
        raise ValueError("Unknown task")
    if name == "majority" and task == "binary":
        model = DummyClassifier(strategy="most_frequent")
    elif name == "median" and task == "mse":
        model = DummyRegressor(strategy="median")
    elif name == "logistic_regression" and task == "binary":
        model = LogisticRegression(
            C=1.0,
            max_iter=1000,
            solver="lbfgs",
            class_weight={int(k): v for k, v in class_weights.items()} if class_weights else None,
            random_state=seed,
        )
    elif name == "ridge" and task == "mse":
        model = Ridge(alpha=1.0)
    elif name == "random_forest":
        options = {**settings, "random_state": seed}
        if task == "binary":
            options["class_weight"] = {int(k): v for k, v in class_weights.items()} if class_weights else None
        model = (RandomForestClassifier if task == "binary" else RandomForestRegressor)(**options)
    else:
        raise ValueError("Unsupported baseline/task")
    model.fit(x, np.asarray(y).reshape(-1))
    metadata = {"format": "phase26_data_only.1", "name": name, "task": task, "input_width": x.shape[1]}
    arrays = {}
    if name in {"majority", "median"}:
        arrays["constant"] = np.asarray(
            [model.classes_[np.argmax(model.class_prior_)]]
            if task == "binary"
            else model.constant_.reshape(-1)
        )
    elif name in {"logistic_regression", "ridge"}:
        arrays["coef"] = np.asarray(model.coef_).reshape(1, -1)
        arrays["intercept"] = np.asarray(model.intercept_).reshape(1)
    else:
        metadata["tree_count"] = len(model.estimators_)
        for i, estimator in enumerate(model.estimators_):
            tree = estimator.tree_
            for field in ("children_left", "children_right", "feature", "threshold", "value"):
                arrays[f"tree_{i:03d}/{field}"] = np.asarray(getattr(tree, field)).copy()
    return model, PortableBaseline(metadata, arrays)



#### PortableBaseline

In [ ]:
%%inline_module src.ch02_ml.baselines src/ch02_ml/baselines.py both
class PortableBaseline:
    def __init__(self, metadata, arrays):
        self.metadata = dict(metadata)
        self.arrays = {k: np.asarray(v) for k, v in arrays.items()}
        if metadata["format"] != "phase26_data_only.1":
            raise ValueError("Unknown baseline checkpoint format")

    def predict(self, x):
        x = np.asarray(x, dtype=np.float32)
        if x.ndim != 2 or x.shape[1] != self.metadata["input_width"] or not np.isfinite(x).all():
            raise ValueError("Invalid baseline features")
        name = self.metadata["name"]
        classification = self.metadata["task"] == "binary"
        if name in {"majority", "median"}:
            value = self.arrays["constant"][0]
            return np.tile([1 - value, value], (len(x), 1)) if classification else np.full((len(x), 1), value)
        if name in {"logistic_regression", "ridge"}:
            value = x @ self.arrays["coef"].T + self.arrays["intercept"]
            if classification:
                p = sigmoid(value.reshape(-1))
                return np.column_stack([1 - p, p])
            return value
        result = np.zeros((len(x), 2 if classification else 1), dtype=float)
        for i in range(self.metadata["tree_count"]):
            prefix = f"tree_{i:03d}/"
            left = self.arrays[prefix + "children_left"]
            right = self.arrays[prefix + "children_right"]
            feature = self.arrays[prefix + "feature"]
            threshold = self.arrays[prefix + "threshold"]
            values = self.arrays[prefix + "value"]
            nodes = np.zeros(len(x), dtype=int)
            active = left[nodes] >= 0
            while active.any():
                rows = np.flatnonzero(active)
                current = nodes[rows]
                nodes[rows] = np.where(
                    x[rows, feature[current]] <= threshold[current], left[current], right[current]
                )
                active = left[nodes] >= 0
            leaf = values[nodes, 0]
            if classification:
                leaf = leaf / leaf.sum(axis=1, keepdims=True)
            result += leaf
        return result / self.metadata["tree_count"]



### `src.ch02_ml.diagnostics`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch02_ml.diagnostics src/ch02_ml/diagnostics.py both
"""Descriptive calibration/group diagnostics; no threshold or TEST tuning."""

import numpy as np



#### groups_for_frame

In [ ]:
%%inline_module src.ch02_ml.diagnostics src/ch02_ml/diagnostics.py both
def groups_for_frame(dataset, frame):
    if dataset == "ch2_cdc_diabetes":
        return np.asarray(frame.sex.astype(str).to_numpy(), dtype=str)
    if dataset == "ch2_bank_marketing":
        return np.where(frame.age.to_numpy() < 60, "age_lt60", "age_ge60")
    if dataset == "ch2_ames_housing":
        return np.where(frame.year_built.to_numpy() < 1980, "built_before1980", "built_from1980")
    raise ValueError("Unknown group diagnostic dataset")



#### calibration

In [ ]:
%%inline_module src.ch02_ml.diagnostics src/ch02_ml/diagnostics.py both
def calibration(target, probability, bins=10):
    y = np.asarray(target).reshape(-1)
    p = np.asarray(probability).reshape(-1)
    if (
        bins < 1
        or y.shape != p.shape
        or not len(y)
        or not np.isin(y, [0, 1]).all()
        or not np.isfinite(p).all()
        or np.any((p < 0) | (p > 1))
    ):
        raise ValueError("Invalid calibration input")
    indices = np.minimum((p * bins).astype(int), bins - 1)
    rows = []
    ece = 0.0
    for i in range(bins):
        mask = indices == i
        count = int(mask.sum())
        confidence = float(p[mask].mean()) if count else None
        rate = float(y[mask].mean()) if count else None
        if count:
            ece += count / len(y) * abs(confidence - rate)
        rows.append(
            {
                "bin": i,
                "lower": i / bins,
                "upper": (i + 1) / bins,
                "count": count,
                "mean_probability": confidence,
                "observed_rate": rate,
            }
        )
    return {"brier": float(np.mean((p - y) ** 2)), "ece": float(ece), "bins": rows, "fit_calibrator": False}



#### group_diagnostics

In [ ]:
%%inline_module src.ch02_ml.diagnostics src/ch02_ml/diagnostics.py both
def group_diagnostics(target, prediction, groups, task):
    y = np.asarray(target).reshape(-1)
    p = np.asarray(prediction).reshape(-1)
    groups = np.asarray(groups).reshape(-1)
    if y.shape != p.shape or groups.shape != y.shape or not np.isfinite(y).all() or not np.isfinite(p).all():
        raise ValueError("Invalid diagnostic arrays")
    rows = {}
    for group in np.unique(groups):
        mask = groups == group
        a, b = y[mask], p[mask]
        count = int(mask.sum())
        if task == "binary":
            positive = a == 1
            negative = a == 0
            pred = b > 0.5
            rows[str(group)] = {
                "count": count,
                "prevalence": float(a.mean()),
                "predicted_positive_rate": float(pred.mean()),
                "recall": float(pred[positive].mean()) if positive.any() else None,
                "false_positive_rate": float(pred[negative].mean()) if negative.any() else None,
                "brier": float(np.mean((b - a) ** 2)),
                "undefined_reason": "No positive or negative support"
                if not positive.any() or not negative.any()
                else None,
            }
        elif task == "mse":
            rows[str(group)] = {
                "count": count,
                "mae": float(np.abs(a - b).mean()),
                "rmse": float(np.sqrt(np.mean((a - b) ** 2))),
            }
        else:
            raise ValueError("Unknown diagnostic task")
    return rows



### `src.ch02_ml.protocol`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch02_ml.protocol src/ch02_ml/protocol.py both
"""Chapter2 prospective configuration and protected development data access."""

from __future__ import annotations

import hashlib

import os

import subprocess

import uuid

from pathlib import Path

import numpy as np

from scripts.prepare_phase25 import CORE_FILES

from src.ch02_ml.diagnostics import groups_for_frame

from src.common.artifact_core import code_fingerprint, sha256_file, write_once

from src.common.contracts import PROTOCOL, ROOT, canonical_sha256, read_json

from src.common.data_access import DatasetView

from src.common.eda_artifacts import read_rows

from src.common.eda_sources import load_tabular

from src.common.tensor_adapters import select_rows

BASE = ROOT / "artifacts/manifests/phase26/revisions/r2"

CACHE = ROOT / "data/processed/v3/phase26/r2"

MODELS = ROOT / "artifacts/models/v3/ch02_ml"

METRICS = ROOT / "artifacts/metrics/phase26"

PREDICTIONS = ROOT / "artifacts/predictions/phase26"

DATASETS = ("ch2_cdc_diabetes", "ch2_ames_housing", "ch2_bank_marketing")

SEEDS = (17, 42, 73)

IMPLEMENTATIONS = ("scratch", "keras", "pytorch")

PHASE26_FILES = [
    f"src/ch02_ml/{name}.py"
    for name in ("protocol", "training", "baselines", "diagnostics", "test_gate", "experiment", "workflow")
] + [
    "scripts/phase26_worker.py",
    "scripts/prepare_phase26.py",
    "scripts/phase26_controller.py",
    "scripts/verify_phase26.py",
    "experiments/configs/v3/phase26_coverage.ini",
    "scripts/run_experiment.sh",
    "scripts/runtime_common.sh",
    "scripts/tensorflow_libraries.py",
]

SOURCE_FILES = sorted(
    set(
        CORE_FILES
        + PHASE26_FILES
        + [
            f"src/common/{name}.py"
            for name in (
                "contracts",
                "acquisition",
                "source_inspection",
                "provenance",
                "eda_sources",
                "eda_artifacts",
                "data_v3",
            )
        ]
    )
)



#### ref

In [ ]:
%%inline_module src.ch02_ml.protocol src/ch02_ml/protocol.py both
def ref(path):
    return {"path": Path(path).relative_to(ROOT).as_posix(), "sha256": sha256_file(path)}



#### array_sha256

In [ ]:
%%inline_module src.ch02_ml.protocol src/ch02_ml/protocol.py both
def array_sha256(value):
    value = np.ascontiguousarray(value)
    return hashlib.sha256(
        canonical_sha256({"shape": list(value.shape), "dtype": str(value.dtype)}).encode() + value.tobytes()
    ).hexdigest()



#### dataset_spec

In [ ]:
%%inline_module src.ch02_ml.protocol src/ch02_ml/protocol.py both
def dataset_spec(dataset):
    if dataset not in DATASETS:
        raise ValueError("Not a Chapter2 dataset")
    return next(row for row in read_json(PROTOCOL)["datasets"] if row["dataset_id"] == dataset)



#### task_for

In [ ]:
%%inline_module src.ch02_ml.protocol src/ch02_ml/protocol.py both
def task_for(dataset):
    return "mse" if dataset_spec(dataset)["task"] == "regression" else "binary"



#### baseline_names

In [ ]:
%%inline_module src.ch02_ml.protocol src/ch02_ml/protocol.py both
def baseline_names(dataset):
    return dataset_spec(dataset)["baselines"]



#### run_ids

In [ ]:
%%inline_module src.ch02_ml.protocol src/ch02_ml/protocol.py both
def run_ids():
    return [
        f"{dataset}_{'mlp_' + method if method in IMPLEMENTATIONS else method}_s{seed}"
        for dataset in DATASETS
        for seed in SEEDS
        for method in (*IMPLEMENTATIONS, *baseline_names(dataset))
    ]



#### code_info

In [ ]:
%%inline_module src.ch02_ml.protocol src/ch02_ml/protocol.py both
def code_info():
    head = subprocess.run(
        ["git", "rev-parse", "HEAD"], cwd=ROOT, check=True, capture_output=True, text=True
    ).stdout.strip()
    return {"git_commit": head, "dirty_tree_sha256": code_fingerprint(ROOT, SOURCE_FILES)}



#### check_sources

In [ ]:
%%inline_module src.ch02_ml.protocol src/ch02_ml/protocol.py both
def check_sources():
    release = read_json(BASE / "release.json")
    if release["code"]["dirty_tree_sha256"] != code_info()["dirty_tree_sha256"]:
        raise ValueError("Phase26 source changed; do not resume/release old configuration silently")
    return release



#### development_data

In [ ]:
%%inline_module src.ch02_ml.protocol src/ch02_ml/protocol.py both
def development_data(dataset, partition):
    dataset_spec(dataset)
    if partition not in {"train", "validation"}:
        raise ValueError("Development data excludes TEST")
    view = DatasetView(dataset)
    ids = view.ids(partition)
    x, y = view.batch(ids, "scratch")
    return {"ids": ids, "x": x, "y": y, "view": view}



#### cache_data

In [ ]:
%%inline_module src.ch02_ml.protocol src/ch02_ml/protocol.py both
def cache_data(dataset):
    path = CACHE / f"{dataset}_development.npz"
    receipt_path = BASE / "data" / f"{dataset}.json"
    if path.exists() and receipt_path.exists():
        receipt = read_json(receipt_path)
        if sha256_file(path) != receipt["cache"]["sha256"]:
            raise ValueError("Development tensor cache hash mismatch")
        with np.load(path, allow_pickle=False) as archive:
            arrays = {k: archive[k] for k in archive.files}
        return arrays
    train = development_data(dataset, "train")
    view = train["view"]
    val_ids = view.ids("validation")
    val_x, val_y = view.batch(val_ids, "scratch")
    positions = select_rows(view._source_ids, val_ids)
    arrays = {
        "train_x": train["x"],
        "train_y": train["y"],
        "train_ids": np.asarray(train["ids"]),
        "validation_x": val_x,
        "validation_y": val_y,
        "validation_ids": np.asarray(val_ids),
        "validation_target_original": view._targets[positions].astype(float),
        "validation_groups": groups_for_frame(dataset, view._frame.iloc[positions]),
    }
    if any(value.dtype.kind == "O" for value in arrays.values()):
        raise ValueError("Development cache must not contain pickle/object arrays")
    from src.common.eda_artifacts import clean

    demo_positions = select_rows(view._source_ids, train["ids"][:4])
    demo = {
        "dataset_id": dataset,
        "split": "train",
        "ids": train["ids"][:4],
        "input_features": clean(view._frame.iloc[demo_positions].to_dict(orient="records")),
        "known_outputs": clean(view._targets[demo_positions].tolist()),
        "canonical_x_sha256": array_sha256(train["x"][:4]),
    }
    write_once(BASE / "data" / f"{dataset}_demo.json", demo)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".{uuid.uuid4().hex}.tmp")
    with temporary.open("xb") as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(temporary, path)
    receipt = {
        "dataset_id": dataset,
        "cache": ref(path),
        "array_hashes": {k: array_sha256(v) for k, v in arrays.items()},
        "provenance": view.provenance,
        "training_ids": len(train["ids"]),
        "validation_ids": len(val_ids),
        "test_accessed": False,
    }
    write_once(receipt_path, receipt)
    return arrays



#### test_data

In [ ]:
%%inline_module src.ch02_ml.protocol src/ch02_ml/protocol.py both
def test_data(dataset, gate):
    """Only this evaluator path opens target values; caller must already own a claim."""
    if gate.state()["state"] != "evaluating" or gate.state()["dataset_id"] != dataset:
        raise ValueError("No active dataset TEST authorization")
    view = DatasetView(dataset)
    split = read_json(view.split_dir / "split_contract.json")
    if gate.state()["split_sha256"] != view.provenance["split_sha256"]:
        raise ValueError("TEST split hash changed")
    partition = next(row for row in split["partitions"] if row["name"] == "test")
    if sha256_file(ROOT / partition["ids"]["path"]) != partition["ids"]["sha256"]:
        raise ValueError("TEST IDs changed")
    ids = read_rows(ROOT / partition["ids"]["path"])
    frame, target, source_ids, _, _ = load_tabular(dataset)
    positions = select_rows(source_ids, ids)
    x = view.adapter.transform(frame.iloc[positions])
    y = view.adapter.target(target.to_numpy()[positions]).reshape(-1, 1)
    return {
        "ids": ids,
        "x": x,
        "y": y,
        "target_original": target.to_numpy()[positions].astype(float),
        "view": view,
        "raw": frame.iloc[positions].reset_index(drop=True),
    }



### `src.ch02_ml.test_gate`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch02_ml.test_gate src/ch02_ml/test_gate.py both
"""One-shot TEST journal. Crashes consume a claim; terminal seals cannot reopen."""

from __future__ import annotations

import json

import os

import re

import uuid

from datetime import UTC, datetime

from pathlib import Path

from src.common.artifact_core import write_once

from src.common.contracts import canonical_sha256, check_seal_transition, read_json, validate_document



#### atomic_json

In [ ]:
%%inline_module src.ch02_ml.test_gate src/ch02_ml/test_gate.py both
def atomic_json(path, document):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".{uuid.uuid4().hex}.tmp")
    with temporary.open("x", encoding="utf-8", newline="\n") as stream:
        json.dump(document, stream, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False)
        stream.write("\n")
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temporary, path)



#### EvaluationGate

In [ ]:
%%inline_module src.ch02_ml.test_gate src/ch02_ml/test_gate.py both
class EvaluationGate:
    def __init__(self, root):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)

    def state(self):
        return read_json(self.root / "seal.json")

    def lock(self, manifest):
        if (self.root / "manifest.json").exists() or (self.root / "seal.json").exists():
            raise ValueError("TEST manifest already locked")
        if not manifest["runs"] or len(set(manifest["runs"])) != len(manifest["runs"]):
            raise ValueError("Missing/duplicate approved runs")
        stamp = datetime.now(UTC).isoformat()
        seal = {
            "schema_version": "3.0.0",
            "evidence_kind": manifest.get("evidence_kind", "measured"),
            "seal_id": manifest["dataset_id"] + "_phase26_test_v1",
            "dataset_id": manifest["dataset_id"],
            "split_sha256": manifest["split_sha256"],
            "locked_config_sha256": canonical_sha256(manifest),
            "evaluator_sha256": manifest["code_sha256"],
            "state": "locked",
            "evaluation_count": 0,
            "approved_run_ids": manifest["runs"],
            "created_at": stamp,
            "events": [
                {
                    "at": stamp,
                    "event": "lock",
                    "actor": "phase26_offline_controller",
                    "reason": "All configurations, model selections and code frozen before TEST",
                }
            ],
        }
        validate_document("seal", seal)
        write_once(self.root / "manifest.json", manifest)
        write_once(self.root / "seal.json", seal)

    def _transition(self, target, event, reason):
        seal = self.state()
        check_seal_transition(
            seal, target, config_sha256=canonical_sha256(read_json(self.root / "manifest.json"))
        )
        seal["state"] = target
        seal["evaluation_count"] = 0 if target == "locked" else 1
        seal["events"].append(
            {
                "at": datetime.now(UTC).isoformat(),
                "event": event,
                "actor": "phase26_offline_controller",
                "reason": reason,
            }
        )
        validate_document("seal", seal)
        atomic_json(self.root / "seal.json", seal)

    def begin(self, code_sha256):
        seal = self.state()
        if seal["state"] != "locked" or code_sha256 != seal["evaluator_sha256"]:
            raise ValueError("TEST state/code mismatch; cannot begin")
        try:
            with (self.root / "evaluation.started").open("x", encoding="utf-8") as stream:
                stream.write(datetime.now(UTC).isoformat())
        except FileExistsError:
            raise ValueError("TEST batch was already claimed") from None
        self._transition("evaluating", "start", "One authorized evaluation batch; no retry or tuning")

    def _run_path(self, run, suffix):
        if not re.fullmatch(r"[a-z][a-z0-9_-]{0,95}", run) or run not in self.state()["approved_run_ids"]:
            raise ValueError("Run not approved")
        return self.root / (run + suffix)

    def claim(self, run, code_sha256):
        seal = self.state()
        if seal["state"] != "evaluating" or code_sha256 != seal["evaluator_sha256"]:
            raise ValueError("No active authorized TEST evaluation")
        try:
            with self._run_path(run, ".claimed").open("x", encoding="utf-8") as stream:
                stream.write(datetime.now(UTC).isoformat())
        except FileExistsError:
            raise ValueError("TEST prediction already claimed; do not repeat") from None

    def record(self, run, result):
        if self.state()["state"] != "evaluating":
            raise ValueError("TEST is not evaluating")
        path = self._run_path(run, ".result.json")
        if path.exists():
            raise ValueError("TEST result already recorded")
        write_once(path, result)

    def complete(self):
        if self.state()["state"] != "evaluating":
            raise ValueError("TEST is not evaluating")
        for run in self.state()["approved_run_ids"]:
            path = self._run_path(run, ".result.json")
            if not path.exists() or read_json(path).get("status") != "PASS":
                raise ValueError("Incomplete/failed TEST batch")
        self._transition("evaluated", "complete", "All approved predictions and metrics saved once")

    def fail(self, reason):
        self._transition("failed_sealed", "fail", reason)



### `src.common.native_training`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.common.native_training src/common/native_training.py both
"""Native GPU updates with one canonical Adam denominator and clipping policy.

TensorFlow uses explicit slots so epsilon is applied to sqrt(v_hat), unlike
stock Keras Adam's epsilon-hat. No weights/gradients move to NumPy during update.
"""

from __future__ import annotations

import numpy as np



#### NativeAdam

In [ ]:
%%inline_module src.common.native_training src/common/native_training.py both
class NativeAdam:
    def __init__(self, backend, learning_rate=0.001, beta1=0.9, beta2=0.999, epsilon=1e-7, max_norm=1):
        if learning_rate <= 0 or epsilon <= 0 or max_norm <= 0 or not 0 <= beta1 < 1 or not 0 <= beta2 < 1:
            raise ValueError("Invalid optimizer settings")
        self.backend = backend
        self.lr = learning_rate
        self.b1 = beta1
        self.b2 = beta2
        self.eps = epsilon
        self.max_norm = max_norm
        self.t = 0
        self.is_torch = hasattr(backend, "torch")
        if self.is_torch:
            self.variables = [p for p in backend.ops.parameters() if p.requires_grad]
            self.optimizer = backend.torch.optim.Adam(
                self.variables, lr=learning_rate, betas=(beta1, beta2), eps=epsilon, foreach=False
            )
        else:
            tf = backend.tf
            self.variables = backend.model.trainable_weights
            with tf.device(backend.device):
                self.m = [tf.Variable(tf.zeros_like(p), trainable=False) for p in self.variables]
                self.v = [tf.Variable(tf.zeros_like(p), trainable=False) for p in self.variables]

    def step(self, x, y, task, sample_weight=None):
        if task not in {"mse", "binary", "multiclass"}:
            raise ValueError("Unknown loss")
        if self.is_torch:
            torch = self.backend.torch
            self.optimizer.zero_grad(set_to_none=True)
            logits = self.backend.forward(x)
            target = torch.as_tensor(y, device=self.backend.device)
            if task == "mse":
                rows = ((logits - target) ** 2).mean(dim=1)
            elif task == "binary":
                rows = torch.nn.functional.binary_cross_entropy_with_logits(
                    logits, target, reduction="none"
                ).mean(dim=1)
            else:
                rows = torch.nn.functional.cross_entropy(logits, target.long(), reduction="none")
            weights = (
                torch.ones_like(rows)
                if sample_weight is None
                else torch.as_tensor(sample_weight, device=self.backend.device, dtype=rows.dtype)
            )
            if (
                weights.shape != rows.shape
                or not bool(torch.isfinite(weights).all())
                or bool((weights < 0).any())
                or float(weights.sum()) <= 0
            ):
                raise ValueError("Invalid sample weights")
            loss = (rows * weights).sum() / weights.sum()
            if not bool(torch.isfinite(loss)):
                raise ValueError("Nonfinite loss")
            loss.backward()
            norm = torch.nn.utils.clip_grad_norm_(self.variables, self.max_norm, error_if_nonfinite=True)
            self.optimizer.step()
            value = float(loss.detach().cpu())
            norm = float(norm.detach().cpu())
        else:
            tf = self.backend.tf
            with tf.device(self.backend.device), tf.GradientTape() as tape:
                logits = self.backend.model(x, training=True)
                target = tf.convert_to_tensor(y)
                if task == "mse":
                    rows = tf.reduce_mean(tf.square(logits - tf.cast(target, logits.dtype)), axis=1)
                elif task == "binary":
                    rows = tf.reduce_mean(
                        tf.nn.sigmoid_cross_entropy_with_logits(
                            labels=tf.cast(target, logits.dtype), logits=logits
                        ),
                        axis=1,
                    )
                else:
                    rows = tf.nn.sparse_softmax_cross_entropy_with_logits(
                        labels=tf.cast(target, tf.int64), logits=logits
                    )
                weights = (
                    tf.ones_like(rows)
                    if sample_weight is None
                    else tf.convert_to_tensor(sample_weight, dtype=rows.dtype)
                )
                if (
                    weights.shape != rows.shape
                    or not bool(tf.reduce_all(tf.math.is_finite(weights)))
                    or bool(tf.reduce_any(weights < 0))
                    or float(tf.reduce_sum(weights)) <= 0
                ):
                    raise ValueError("Invalid sample weights")
                loss = tf.reduce_sum(rows * weights) / tf.reduce_sum(weights)
            with tf.device(self.backend.device):
                tf.debugging.assert_all_finite(loss, "Nonfinite loss")
                gradients = tape.gradient(loss, self.variables)
                for gradient in gradients:
                    tf.debugging.assert_all_finite(gradient, "Nonfinite gradients")
                gradients, norm = tf.clip_by_global_norm(gradients, self.max_norm)
            self.t += 1
            with tf.device(self.backend.device):
                for p, g, m, v in zip(self.variables, gradients, self.m, self.v, strict=True):
                    m.assign(self.b1 * m + (1 - self.b1) * g)
                    v.assign(self.b2 * v + (1 - self.b2) * tf.square(g))
                    mhat = m / (1 - self.b1**self.t)
                    vhat = v / (1 - self.b2**self.t)
                    p.assign_sub(self.lr * mhat / (tf.sqrt(vhat) + self.eps))
            value = float(loss.numpy())
            norm = float(norm.numpy())
        return {"loss": value, "gradient_global_norm_before_clip": norm}

    def weights(self):
        backend = self.backend
        if self.is_torch:
            result = {}
            for name, param, kind, key in backend.parameter_pairs():
                value = param.detach().cpu().numpy()
                if key in {"w", "u"}:
                    value = value.transpose(2, 3, 1, 0) if kind == "conv2d" else value.T
                result[name] = np.ascontiguousarray(value)
            return result
        return {name: variable.numpy().copy() for name, variable in backend.parameter_pairs()}



### `src.common.training_core`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.common.training_core src/common/training_core.py both
"""Shared reproducibility and trainer hooks. No dataset training is started here."""

from __future__ import annotations

import random

import time

import numpy as np



#### seed_everything

In [ ]:
%%inline_module src.common.training_core src/common/training_core.py both
def seed_everything(seed, backend="numpy"):
    if not isinstance(seed, int) or seed < 0:
        raise ValueError("Invalid seed")
    random.seed(seed)
    np.random.seed(seed)
    if backend == "torch":
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True)
    elif backend == "tensorflow":
        import tensorflow as tf

        tf.keras.utils.set_random_seed(seed)
        tf.config.experimental.enable_op_determinism()
    elif backend != "numpy":
        raise ValueError("Unknown backend")



#### batch_order

In [ ]:
%%inline_module src.common.training_core src/common/training_core.py both
def batch_order(size, batch_size, seed, epoch):
    if size < 1 or batch_size < 1 or epoch < 0:
        raise ValueError("Invalid batch schedule")
    order = np.random.default_rng(np.random.SeedSequence([seed, epoch])).permutation(size).tolist()
    return [order[i : i + batch_size] for i in range(0, size, batch_size)]



#### clip_gradients

In [ ]:
%%inline_module src.common.training_core src/common/training_core.py both
def clip_gradients(gradients, max_norm=1):
    if max_norm <= 0 or not gradients or any(not np.isfinite(x).all() for x in gradients.values()):
        raise ValueError("Invalid/nonfinite gradients")
    norm = float(np.sqrt(sum(np.sum(np.square(v.astype(np.float64))) for v in gradients.values())))
    scale = min(1.0, max_norm / max(norm, np.finfo(float).tiny))
    return {key: value * scale for key, value in gradients.items()}, norm



#### Adam

In [ ]:
%%inline_module src.common.training_core src/common/training_core.py both
class Adam:
    """Canonical Algorithm1: sqrt(v_hat)+eps, not Keras epsilon-hat."""

    def __init__(self, weights, learning_rate=0.001, beta1=0.9, beta2=0.999, epsilon=1e-7):
        if learning_rate <= 0 or epsilon <= 0 or not 0 <= beta1 < 1 or not 0 <= beta2 < 1:
            raise ValueError("Invalid Adam settings")
        self.lr, self.beta1, self.beta2, self.epsilon = learning_rate, beta1, beta2, epsilon
        self.m = {k: np.zeros_like(v) for k, v in weights.items()}
        self.v = {k: np.zeros_like(v) for k, v in weights.items()}
        self.t = 0

    def step(self, weights, gradients):
        if (
            set(weights) != set(self.m)
            or set(gradients) != set(weights)
            or any(
                gradients[k].shape != weights[k].shape or not np.isfinite(gradients[k]).all() for k in weights
            )
        ):
            raise ValueError("Adam gradient keys/shapes/values mismatch")
        self.t += 1
        for key in weights:
            self.m[key] = self.beta1 * self.m[key] + (1 - self.beta1) * gradients[key]
            self.v[key] = self.beta2 * self.v[key] + (1 - self.beta2) * gradients[key] ** 2
            mhat = self.m[key] / (1 - self.beta1**self.t)
            vhat = self.v[key] / (1 - self.beta2**self.t)
            weights[key] -= self.lr * mhat / (np.sqrt(vhat) + self.epsilon)



#### EarlyStopping

In [ ]:
%%inline_module src.common.training_core src/common/training_core.py both
class EarlyStopping:
    def __init__(self, patience=6, min_delta=1e-5):
        if patience < 1 or min_delta < 0:
            raise ValueError("Invalid early stopping")
        self.patience, self.min_delta = patience, min_delta
        self.best = float("inf")
        self.patience_reference = float("inf")
        self.best_epoch = None
        self.bad_epochs = 0

    def observe(self, validation_loss, epoch):
        if not np.isfinite(validation_loss):
            raise ValueError("Nonfinite validation loss; record failed run")
        improved = validation_loss < self.best
        if improved:
            self.best = validation_loss
            self.best_epoch = epoch
        if validation_loss < self.patience_reference - self.min_delta:
            self.patience_reference = validation_loss
            self.bad_epochs = 0
        else:
            self.bad_epochs += 1
        return improved, self.bad_epochs >= self.patience



#### inference_timing

In [ ]:
%%inline_module src.common.training_core src/common/training_core.py both
def inference_timing(function, synchronize=lambda: None, warmup=5, repeats=30):
    if warmup < 0 or repeats < 1:
        raise ValueError("Invalid timing repetitions")
    for _ in range(warmup):
        function()
        synchronize()
    values = []
    for _ in range(repeats):
        synchronize()
        start = time.perf_counter()
        function()
        synchronize()
        values.append((time.perf_counter() - start) * 1000)
    return {
        "inference_p50_ms": float(np.percentile(values, 50)),
        "inference_p95_ms": float(np.percentile(values, 95)),
        "warmup": warmup,
        "repeats": repeats,
    }



### `src.ch02_ml.training`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch02_ml.training src/ch02_ml/training.py both
"""Chapter2 matched trainers; reuse frozen Phase25 models, not legacy notebooks."""

from __future__ import annotations

import time

import numpy as np

from src.common.contracts import canonical_sha256

from src.common.native_training import NativeAdam

from src.common.numpy_engine import NumpyNetwork, loss_and_gradient, sigmoid

from src.common.training_core import Adam, EarlyStopping, batch_order, clip_gradients



#### validate_weights

In [ ]:
%%inline_module src.ch02_ml.training src/ch02_ml/training.py both
def validate_weights(weights, reference):
    if set(weights) != set(reference) or any(
        np.asarray(weights[key]).shape != reference[key].shape
        or np.asarray(weights[key]).dtype != reference[key].dtype
        or not np.isfinite(weights[key]).all()
        for key in reference
    ):
        raise ValueError("Weight keys/shapes/dtypes/finite values mismatch")



#### validate_state

In [ ]:
%%inline_module src.ch02_ml.training src/ch02_ml/training.py both
def validate_state(state, reference):
    if (
        set(state) != {"weights", "m", "v", "iterations"}
        or not isinstance(state["iterations"], int)
        or state["iterations"] < 0
    ):
        raise ValueError("Invalid optimizer state")
    for field in ("weights", "m", "v"):
        validate_weights(state[field], reference)



#### ScratchTrainer

In [ ]:
%%inline_module src.ch02_ml.training src/ch02_ml/training.py both
class ScratchTrainer:
    def __init__(self, plan, weights):
        self.model = NumpyNetwork(plan, weights)
        self.optimizer = Adam(self.model.weights)

    @property
    def iterations(self):
        return self.optimizer.t

    def predict(self, x):
        return self.model.forward(x)

    def weights(self):
        return {k: v.copy() for k, v in self.model.weights.items()}

    def assign(self, weights):
        validate_weights(weights, self.model.weights)
        for key, value in weights.items():
            self.model.weights[key][...] = value

    def step(self, x, y, task, sample_weight):
        loss, grad, _ = self.model.loss_gradients(x, y, task, sample_weight)
        grad, norm = clip_gradients(grad)
        self.optimizer.step(self.model.weights, grad)
        return {"loss": loss, "gradient_global_norm_before_clip": norm}

    def export_state(self):
        return {
            "weights": self.weights(),
            "m": {k: v.copy() for k, v in self.optimizer.m.items()},
            "v": {k: v.copy() for k, v in self.optimizer.v.items()},
            "iterations": self.iterations,
        }

    def restore_state(self, state):
        validate_state(state, self.model.weights)
        self.assign(state["weights"])
        for attr in ("m", "v"):
            if set(state[attr]) != set(self.model.weights):
                raise ValueError("Adam slot keys mismatch")
            setattr(self.optimizer, attr, {k: np.asarray(v).copy() for k, v in state[attr].items()})
        self.optimizer.t = int(state["iterations"])



#### TorchTrainer

In [ ]:
%%inline_module src.ch02_ml.training src/ch02_ml/training.py both
class TorchTrainer:
    def __init__(self, plan, weights, device="gpu"):
        from src.common.backend_models import TorchBackend

        self.model = TorchBackend(plan, weights, device)
        self.optimizer = NativeAdam(self.model)

    @property
    def iterations(self):
        values = list(self.optimizer.optimizer.state.values())
        return int(values[0]["step"].item()) if values else 0

    def predict(self, x):
        return self.model.predict(x)

    def weights(self):
        return self.optimizer.weights()

    def assign(self, weights):
        self.model.assign(weights)

    def step(self, x, y, task, sample_weight):
        return self.optimizer.step(x, y, task, sample_weight)

    def export_state(self):
        m = {}
        v = {}
        for name, param, kind, key in self.model.parameter_pairs():
            state = self.optimizer.optimizer.state.get(param, {})
            for field, destination in (("exp_avg", m), ("exp_avg_sq", v)):
                array = (
                    state[field].detach().cpu().numpy().T
                    if state and key == "w"
                    else state[field].detach().cpu().numpy()
                    if state
                    else np.zeros_like(self.weights()[name])
                )
                destination[name] = np.ascontiguousarray(array)
        return {"weights": self.weights(), "m": m, "v": v, "iterations": self.iterations}

    def restore_state(self, state):
        validate_state(state, self.weights())
        self.assign(state["weights"])
        torch = self.model.torch
        for name, param, kind, key in self.model.parameter_pairs():
            target = self.optimizer.optimizer.state[param]
            target["step"] = torch.tensor(float(state["iterations"]), dtype=torch.float32)
            for field, source in (("exp_avg", "m"), ("exp_avg_sq", "v")):
                value = state[source][name].T if key == "w" else state[source][name]
                target[field] = torch.as_tensor(
                    np.ascontiguousarray(value), dtype=param.dtype, device=self.model.device
                ).clone()



#### KerasTrainer

In [ ]:
%%inline_module src.ch02_ml.training src/ch02_ml/training.py both
class KerasTrainer:
    """Keras model + TensorFlow graph step, canonical Adam (not stock epsilon-hat)."""

    def __init__(self, plan, weights, device="gpu"):
        from src.common.backend_models import KerasBackend

        self.model = KerasBackend(plan, weights, device)
        self.optimizer = NativeAdam(self.model)
        tf = self.model.tf
        with tf.device(self.model.device):
            self.step_counter = tf.Variable(0, dtype=tf.int64, trainable=False)
        signature = [
            tf.TensorSpec([None, plan["input_shape"][0]], tf.float32),
            tf.TensorSpec([None, 1], tf.float32),
            tf.TensorSpec([None], tf.float32),
        ]
        self._steps = {
            task: tf.function(self._graph_step(task), input_signature=signature, autograph=False)
            for task in ("mse", "binary")
        }
        self._predict = tf.function(
            lambda x: self.model.model(x, training=False), input_signature=[signature[0]], autograph=False
        )

    def _graph_step(self, task):
        tf = self.model.tf
        optimizer = self.optimizer

        def step(x, y, sample_weight):
            tf.debugging.assert_all_finite(sample_weight, "Nonfinite sample weights")
            tf.debugging.assert_non_negative(sample_weight)
            tf.debugging.assert_positive(tf.reduce_sum(sample_weight))
            tf.debugging.assert_equal(tf.shape(x)[0], tf.shape(sample_weight)[0])
            tf.debugging.assert_equal(tf.shape(x)[0], tf.shape(y)[0])
            with tf.device(self.model.device), tf.GradientTape() as tape:
                logits = self.model.model(x, training=True)
                rows = tf.reduce_mean(
                    tf.square(logits - y)
                    if task == "mse"
                    else tf.nn.sigmoid_cross_entropy_with_logits(labels=y, logits=logits),
                    axis=1,
                )
                loss = tf.reduce_sum(rows * sample_weight) / tf.reduce_sum(sample_weight)
            with tf.device(self.model.device):
                tf.debugging.assert_all_finite(loss, "Nonfinite loss")
                gradients = tape.gradient(loss, optimizer.variables)
                for gradient in gradients:
                    tf.debugging.assert_all_finite(gradient, "Nonfinite gradient")
                gradients, norm = tf.clip_by_global_norm(gradients, optimizer.max_norm)
                self.step_counter.assign_add(1)
                t = tf.cast(self.step_counter, tf.float64)
                for p, g, m, v in zip(optimizer.variables, gradients, optimizer.m, optimizer.v, strict=True):
                    m.assign(optimizer.b1 * m + (1 - optimizer.b1) * g)
                    v.assign(optimizer.b2 * v + (1 - optimizer.b2) * tf.square(g))
                    mhat = m / tf.cast(1 - tf.pow(tf.constant(optimizer.b1, tf.float64), t), tf.float32)
                    vhat = v / tf.cast(1 - tf.pow(tf.constant(optimizer.b2, tf.float64), t), tf.float32)
                    p.assign_sub(optimizer.lr * mhat / (tf.sqrt(vhat) + optimizer.eps))
            return loss, norm

        return step

    @property
    def iterations(self):
        return int(self.step_counter.numpy())

    def predict(self, x):
        with self.model.tf.device(self.model.device):
            return self._predict(x).numpy()

    def weights(self):
        return self.optimizer.weights()

    def assign(self, weights):
        self.model.assign(weights)

    def step(self, x, y, task, sample_weight):
        if task not in self._steps:
            raise ValueError("Unknown MLP loss")
        loss, norm = self._steps[task](x, y, sample_weight)
        return {"loss": float(loss.numpy()), "gradient_global_norm_before_clip": float(norm.numpy())}

    def export_state(self):
        names = [name for name, _ in self.model.parameter_pairs()]
        return {
            "weights": self.weights(),
            "m": {k: v.numpy().copy() for k, v in zip(names, self.optimizer.m, strict=True)},
            "v": {k: v.numpy().copy() for k, v in zip(names, self.optimizer.v, strict=True)},
            "iterations": self.iterations,
        }

    def restore_state(self, state):
        validate_state(state, self.weights())
        self.assign(state["weights"])
        self.step_counter.assign(state["iterations"])
        for slots, field in ((self.optimizer.m, "m"), (self.optimizer.v, "v")):
            for (name, _), slot in zip(self.model.parameter_pairs(), slots, strict=True):
                slot.assign(state[field][name])



#### predict_batches

In [ ]:
%%inline_module src.ch02_ml.training src/ch02_ml/training.py both
def predict_batches(trainer, x, batch_size=4096):
    if batch_size < 1 or len(x) == 0:
        raise ValueError("Invalid prediction batch")
    result = np.concatenate([trainer.predict(x[i : i + batch_size]) for i in range(0, len(x), batch_size)])
    if not np.isfinite(result).all():
        raise ValueError("Nonfinite prediction")
    return result



#### task_outputs

In [ ]:
%%inline_module src.ch02_ml.training src/ch02_ml/training.py both
def task_outputs(logits, task):
    if task == "mse":
        return logits
    if task != "binary":
        raise ValueError("Unknown MLP task")
    p = sigmoid(np.asarray(logits).reshape(-1))
    return np.column_stack([1 - p, p])



#### fit

In [ ]:
%%inline_module src.ch02_ml.training src/ch02_ml/training.py both
def fit(trainer, x, y, val_x, val_y, task, *, seed, budget, class_weights=None, progress=None, resume=None):
    stop = EarlyStopping(budget["patience"], budget["min_delta"])
    history = []
    best = None
    elapsed = 0.0
    if resume is not None:
        trainer.restore_state(resume["trainer"])
        history = resume["history"]
        best = resume["best_weights"]
        elapsed = resume["elapsed_seconds"]
        for name, value in resume["early_stopping"].items():
            setattr(stop, name, value)
    train_weights = (
        np.ones(len(y), dtype="float32")
        if class_weights is None
        else np.asarray([class_weights[str(int(value))] for value in y.reshape(-1)], dtype="float32")
    )
    val_weights = (
        np.ones(len(val_y), dtype="float32")
        if class_weights is None
        else np.asarray([class_weights[str(int(value))] for value in val_y.reshape(-1)], dtype="float32")
    )
    started = time.perf_counter()
    for epoch in range(len(history), budget["max_epochs"]):
        if stop.bad_epochs >= stop.patience:
            break
        order = batch_order(len(x), budget["batch_size"], seed, epoch)
        loss_sum = weight_sum = 0.0
        norms = []
        for positions in order:
            if elapsed + time.perf_counter() - started > budget["wall_time_limit_minutes"] * 60:
                raise TimeoutError("Locked wall-time budget exceeded")
            output = trainer.step(x[positions], y[positions], task, train_weights[positions])
            weight = float(train_weights[positions].sum())
            loss_sum += output["loss"] * weight
            weight_sum += weight
            norms.append(output["gradient_global_norm_before_clip"])
        val_logits = predict_batches(trainer, val_x)
        validation_loss = loss_and_gradient(val_logits, val_y, task, val_weights)[0]
        improved, terminate = stop.observe(validation_loss, epoch)
        if improved:
            best = trainer.weights()
        history.append(
            {
                "epoch": epoch,
                "train_loss": loss_sum / weight_sum,
                "validation_loss": validation_loss,
                "gradient_norm_mean": float(np.mean(norms)),
                "gradient_norm_max": float(np.max(norms)),
                "batch_order_sha256": canonical_sha256(order),
            }
        )
        if progress is not None:
            progress(
                {
                    "history": history,
                    "best_weights": best,
                    "early_stopping": {
                        "best": stop.best,
                        "best_epoch": stop.best_epoch,
                        "bad_epochs": stop.bad_epochs,
                        "patience_reference": stop.patience_reference,
                    },
                    "trainer": trainer.export_state(),
                    "elapsed_seconds": elapsed + time.perf_counter() - started,
                }
            )
        print(
            f"epoch={epoch + 1} train_loss={history[-1]['train_loss']:.6f} validation_loss={validation_loss:.6f} best_epoch={stop.best_epoch + 1}",
            flush=True,
        )
        if terminate:
            break
    if best is None:
        raise ValueError("No finite best checkpoint")
    elapsed += time.perf_counter() - started
    trainer.assign(best)
    return {
        "history": history,
        "best_epoch": stop.best_epoch,
        "best_validation_loss": stop.best,
        "weights": best,
        "train_seconds": elapsed,
        "optimizer_steps": trainer.iterations,
    }



### `src.common.device_core`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.common.device_core src/common/device_core.py both
"""Explicit device selection and allocator measurements, never byte estimates."""

from __future__ import annotations

import resource

import time



#### DeviceMonitor

In [ ]:
%%inline_module src.common.device_core src/common/device_core.py both
class DeviceMonitor:
    def __init__(self, backend, device):
        if (
            backend not in {"numpy", "torch", "tensorflow"}
            or device not in {"cpu", "gpu"}
            or backend == "numpy"
            and device != "cpu"
        ):
            raise ValueError("Unsupported backend/device")
        self.backend = backend
        self.device = device
        self.started = None
        if backend == "torch":
            import torch

            self.framework = torch
            if device == "gpu" and not torch.cuda.is_available():
                raise RuntimeError("GPU requested but unavailable")
        elif backend == "tensorflow":
            import tensorflow as tf

            self.framework = tf
            if device == "gpu" and not tf.config.list_physical_devices("GPU"):
                raise RuntimeError("GPU requested but unavailable")

    def synchronize(self):
        if self.device == "gpu":
            if self.backend == "torch":
                self.framework.cuda.synchronize()
            else:
                self.framework.constant(0.0).numpy()  # workload probes also materialize the returned tensor

    def start(self):
        self.synchronize()
        if self.device == "gpu":
            if self.backend == "torch":
                self.framework.cuda.reset_peak_memory_stats()
            else:
                self.framework.config.experimental.reset_memory_stats("GPU:0")
        self.started = time.perf_counter()

    def finish(self):
        if self.started is None:
            raise ValueError("Memory monitor was not started")
        self.synchronize()
        peak = reserved = 0
        if self.device == "gpu":
            if self.backend == "torch":
                peak = self.framework.cuda.max_memory_allocated()
                reserved = self.framework.cuda.max_memory_reserved()
            else:
                peak = self.framework.config.experimental.get_memory_info("GPU:0")["peak"]
        return {
            "backend": self.backend,
            "requested_device": self.device,
            "wall_seconds": time.perf_counter() - self.started,
            "peak_vram_bytes": int(peak),
            "peak_vram_reserved_bytes": int(reserved),
            "process_peak_ram_bytes": int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024),
            "ram_scope": "Linux process lifetime high-water RSS; not per-model incremental RAM",
            "vram_scope": "Framework allocator peak since reset; excludes driver/other processes; TensorFlow reserved not reported",
        }



### `src.common.metric_core`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.common.metric_core src/common/metric_core.py both
"""Finite metrics in original units; undefined metrics are null with a reason."""

from __future__ import annotations

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    recall_score,
    roc_auc_score,
)



#### metric

In [ ]:
%%inline_module src.common.metric_core src/common/metric_core.py both
def metric(value=None, reason=None):
    return {"value": float(value) if value is not None else None, "undefined_reason": reason}



#### classification_metrics

In [ ]:
%%inline_module src.common.metric_core src/common/metric_core.py both
def classification_metrics(target, probabilities):
    target = np.asarray(target)
    probs = np.asarray(probabilities, dtype=float)
    if (
        target.ndim != 1
        or not len(target)
        or probs.ndim != 2
        or len(probs) != len(target)
        or probs.shape[1] < 2
        or not np.isfinite(probs).all()
        or np.any(probs < 0)
        or np.any(probs > 1)
        or not np.allclose(probs.sum(axis=1), 1)
        or not np.isin(target, np.arange(probs.shape[1])).all()
    ):
        raise ValueError("Invalid classification labels/probabilities")
    classes = np.arange(probs.shape[1])
    prediction = probs.argmax(axis=1)
    result = {
        "accuracy": metric(accuracy_score(target, prediction)),
        "macro_f1": metric(f1_score(target, prediction, labels=classes, average="macro", zero_division=0)),
        "weighted_f1": metric(
            f1_score(target, prediction, labels=classes, average="weighted", zero_division=0)
        ),
        "confusion_matrix": confusion_matrix(target, prediction, labels=classes).tolist(),
        "per_class_recall": {
            str(c): metric(recall_score(target, prediction, labels=[c], average=None, zero_division=0)[0])
            if np.any(target == c)
            else metric(reason="No true samples for this class")
            for c in classes
        },
    }
    if len(np.unique(target)) != len(classes):
        result.update(
            {
                key: metric(reason="Not all classes have positive and negative support")
                for key in ("auroc", "average_precision")
            }
        )
    elif len(classes) == 2:
        result["auroc"] = metric(roc_auc_score(target, probs[:, 1]))
        result["average_precision"] = metric(average_precision_score(target, probs[:, 1]))
    else:
        result["auroc"] = metric(roc_auc_score(target, probs, multi_class="ovr", average="macro"))
        result["average_precision"] = metric(
            average_precision_score(np.eye(len(classes))[target.astype(int)], probs, average="macro")
        )
    return result



#### regression_metrics

In [ ]:
%%inline_module src.common.metric_core src/common/metric_core.py both
def regression_metrics(target, prediction, horizons=None, series=None):
    target = np.asarray(target, dtype=float)
    prediction = np.asarray(prediction, dtype=float)
    if (
        target.shape != prediction.shape
        or not target.size
        or target.ndim not in {1, 2}
        or not np.isfinite(target).all()
        or not np.isfinite(prediction).all()
    ):
        raise ValueError("Invalid regression arrays")
    residual = prediction - target
    denom = np.abs(target) + np.abs(prediction)
    ratios = np.divide(2 * np.abs(residual), denom, out=np.zeros_like(residual), where=denom != 0)
    centered = np.sum((target - target.mean(axis=0)) ** 2)
    result = {
        "mae": metric(np.abs(residual).mean()),
        "rmse": metric(np.sqrt(np.mean(residual**2))),
        "smape": metric(100 * ratios.mean()),
        "r2": metric(1 - np.sum(residual**2) / centered)
        if len(target) > 1 and centered > 0
        else metric(reason="Insufficient samples or constant target"),
    }
    if horizons is not None:
        if target.ndim != 2 or len(horizons) != target.shape[1] or len(set(horizons)) != len(horizons):
            raise ValueError("Horizon shape mismatch")
        result["by_horizon"] = {
            str(h): regression_metrics(target[:, i], prediction[:, i]) for i, h in enumerate(horizons)
        }
    if series is not None:
        series = np.asarray(series)
        if series.shape != (len(target),):
            raise ValueError("Series shape mismatch")
        result["by_series"] = {
            str(s): regression_metrics(target[series == s], prediction[series == s], horizons)
            for s in np.unique(series)
        }
        result["macro_series_rmse"] = metric(
            np.mean([row["rmse"]["value"] for row in result["by_series"].values()])
        )
    return result



### `src.common.parity`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.common.parity src/common/parity.py both
"""Declared per-tensor symmetric relative L2 gradient error (locked limit 1e-3)."""

import numpy as np



#### gradient_parity

In [ ]:
%%inline_module src.common.parity src/common/parity.py both
def gradient_parity(reference, actual, tolerance=1e-3):
    if not reference or set(reference) != set(actual) or tolerance <= 0:
        raise ValueError("Gradient keys/tolerance mismatch")
    results = {}
    maximum = 0.0
    for name, value in reference.items():
        other = actual[name]
        if value.shape != other.shape or not np.isfinite(value).all() or not np.isfinite(other).all():
            raise ValueError("Gradient shape/value mismatch")
        left = value.astype(np.float64).ravel()
        right = other.astype(np.float64).ravel()
        error = float(np.linalg.norm(left - right) / max(np.linalg.norm(left) + np.linalg.norm(right), 1e-12))
        results[name] = error
        maximum = max(maximum, float(np.max(np.abs(left - right))))
        if error > tolerance:
            raise ValueError(f"Gradient relative L2 error {name}: {error} > {tolerance}")
    return {
        "definition": "norm(g_ref-g_actual)/(norm(g_ref)+norm(g_actual)), denominator floor 1e-12; per tensor",
        "tolerance": tolerance,
        "max_relative_l2_error": max(results.values()),
        "per_tensor_relative_l2_error": results,
        "max_abs_error": maximum,
    }



### `src.ch02_ml.experiment`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch02_ml.experiment src/ch02_ml/experiment.py both
"""Real Chapter2 runs, immutable selections and schema-valid measured results."""

from __future__ import annotations

import gzip

import json

import os

import platform

import shutil

import time

import uuid

from datetime import UTC, datetime

import numpy as np

from src.ch02_ml.baselines import PortableBaseline, fit_baseline

from src.ch02_ml.diagnostics import calibration, group_diagnostics, groups_for_frame

from src.ch02_ml.protocol import (
    BASE,
    IMPLEMENTATIONS,
    METRICS,
    MODELS,
    PREDICTIONS,
    array_sha256,
    cache_data,
    check_sources,
    dataset_spec,
    ref,
    test_data,
)

from src.ch02_ml.test_gate import EvaluationGate, atomic_json

from src.ch02_ml.training import (
    KerasTrainer,
    ScratchTrainer,
    TorchTrainer,
    fit,
    predict_batches,
    task_outputs,
)

from src.common.artifact_core import ArtifactStore, sha256_file, write_once

from src.common.contracts import ROOT, canonical_sha256, read_json, validate_document

from src.common.data_access import DatasetView

from src.common.device_core import DeviceMonitor

from src.common.metric_core import classification_metrics, regression_metrics

from src.common.model_core import canonical_weights

from src.common.numpy_engine import NumpyNetwork

from src.common.parity import gradient_parity

from src.common.training_core import inference_timing, seed_everything



#### configure

In [ ]:
%%inline_module src.ch02_ml.experiment src/ch02_ml/experiment.py both
def configure(method, seed):
    backend = {"scratch": "numpy", "keras": "tensorflow", "pytorch": "torch"}.get(method, "numpy")
    if backend == "torch":
        import torch

        torch.set_num_threads(2)
    elif backend == "tensorflow":
        import tensorflow as tf

        tf.config.threading.set_intra_op_parallelism_threads(2)
        tf.config.threading.set_inter_op_parallelism_threads(2)
        for gpu in tf.config.list_physical_devices("GPU"):
            tf.config.experimental.set_memory_growth(gpu, True)
        tf.config.experimental.enable_tensor_float_32_execution(False)
    seed_everything(seed, backend)
    return backend



#### make_trainer

In [ ]:
%%inline_module src.ch02_ml.experiment src/ch02_ml/experiment.py both
def make_trainer(method, plan, weights, device=None):
    if method == "scratch":
        return ScratchTrainer(plan, weights)
    if method == "pytorch":
        return TorchTrainer(plan, weights, device or "gpu")
    if method == "keras":
        return KerasTrainer(plan, weights, device or "gpu")
    raise ValueError("Unknown MLP method")



#### provenance

In [ ]:
%%inline_module src.ch02_ml.experiment src/ch02_ml/experiment.py both
def provenance(entry, config):
    return {
        **config["provenance"],
        "adapter_sha256": config["adapter"]["sha256"],
        "config_sha256": entry["config"]["sha256"],
        "code_sha256": config["code"]["dirty_tree_sha256"],
        "environment_sha256": config["environment_sha256"],
        "model_contract_sha256": entry["model_contract"]["sha256"],
    }



#### metrics_for

In [ ]:
%%inline_module src.ch02_ml.experiment src/ch02_ml/experiment.py both
def metrics_for(target_original, raw_output, task, adapter):
    if task == "binary":
        probs = task_outputs(raw_output, task) if raw_output.shape[1] == 1 else raw_output
        values = classification_metrics(np.asarray(target_original, dtype=int), probs)
        diagnostic = calibration(target_original, probs[:, 1])
        values["brier"] = {"value": diagnostic["brier"], "undefined_reason": None}
        values["ece"] = {"value": diagnostic["ece"], "undefined_reason": None}
        return values, probs, diagnostic
    prediction = adapter.inverse_target(raw_output).reshape(-1)
    values = regression_metrics(target_original, prediction)
    positive = np.abs(target_original) > 1e-8
    values["mape"] = {
        "value": float(np.mean(np.abs(prediction - target_original) / np.abs(target_original)) * 100)
        if positive.all()
        else None,
        "undefined_reason": None if positive.all() else "Zero/near-zero true targets",
    }
    return values, prediction[:, None], None



#### metric_rows

In [ ]:
%%inline_module src.ch02_ml.experiment src/ch02_ml/experiment.py both
def metric_rows(values, task):
    rows = []
    scalar = {key: value for key, value in values.items() if isinstance(value, dict) and "value" in value}
    for label, value in values.get("per_class_recall", {}).items():
        scalar[f"recall_class_{label}"] = value
    for name, value in scalar.items():
        direction = (
            "max"
            if name in {"accuracy", "macro_f1", "weighted_f1", "auroc", "average_precision", "r2"}
            or name.startswith("recall_")
            else "min"
        )
        rows.append(
            {
                "name": name,
                **value,
                "unit": "USD"
                if task == "mse" and name in {"mae", "rmse"}
                else "percent"
                if name in {"mape", "smape"}
                else "ratio",
                "direction": direction,
                "horizon": None,
            }
        )
    return rows



#### save_predictions

In [ ]:
%%inline_module src.ch02_ml.experiment src/ch02_ml/experiment.py both
def save_predictions(run, split, ids, targets, predictions):
    path = PREDICTIONS / split / f"{run}.jsonl.gz"
    if path.exists():
        raise ValueError("Prediction artifact already exists; do not overwrite")
    path.parent.mkdir(parents=True, exist_ok=True)
    with (
        path.open("xb") as outgoing,
        gzip.GzipFile(filename="", fileobj=outgoing, mode="wb", mtime=0) as packed,
    ):
        for index, key in enumerate(ids):
            row = {
                "id": key,
                "target": float(targets[index]),
                "prediction": [float(v) for v in predictions[index]],
            }
            packed.write((json.dumps(row, sort_keys=True, allow_nan=False) + "\n").encode())
    return ref(path)



#### save_progress

In [ ]:
%%inline_module src.ch02_ml.experiment src/ch02_ml/experiment.py both
def save_progress(run, state, config_hash):
    folder = MODELS / run
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / "progress.npz"
    arrays = {
        f"{field}/{key}": value
        for field in ("weights", "m", "v")
        for key, value in state["trainer"][field].items()
    }
    arrays.update({f"best/{key}": value for key, value in state["best_weights"].items()})
    temporary = path.with_name(path.name + f".{uuid.uuid4().hex}.tmp")
    with temporary.open("xb") as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(temporary, path)
    metadata = {k: v for k, v in state.items() if k not in {"trainer", "best_weights"}}
    metadata.update(
        {"iterations": state["trainer"]["iterations"], "config_sha256": config_hash, "arrays": ref(path)}
    )
    atomic_json(folder / "progress.json", metadata)



#### load_progress

In [ ]:
%%inline_module src.ch02_ml.experiment src/ch02_ml/experiment.py both
def load_progress(run, config_hash):
    path = MODELS / run / "progress.json"
    if not path.exists():
        return None
    metadata = read_json(path)
    record = metadata["arrays"]
    if metadata["config_sha256"] != config_hash or sha256_file(ROOT / record["path"]) != record["sha256"]:
        raise ValueError("Resume state/config integrity mismatch")
    with np.load(ROOT / record["path"], allow_pickle=False) as archive:
        arrays = {key: archive[key] for key in archive.files}
    state = {
        "history": metadata["history"],
        "early_stopping": metadata["early_stopping"],
        "elapsed_seconds": metadata["elapsed_seconds"],
        "trainer": {
            field: {k.removeprefix(field + "/"): v for k, v in arrays.items() if k.startswith(field + "/")}
            for field in ("weights", "m", "v")
        },
        "best_weights": {k.removeprefix("best/"): v for k, v in arrays.items() if k.startswith("best/")},
    }
    state["trainer"]["iterations"] = metadata["iterations"]
    return state



#### run_record

In [ ]:
%%inline_module src.ch02_ml.experiment src/ch02_ml/experiment.py both
def run_record(entry, config, split, values, efficiency, started, finished, artifacts, seal_id=None):
    if entry["method"] not in IMPLEMENTATIONS:
        return {
            "schema_version": "phase26.classical_run.1",
            "run_id": entry["run_id"],
            "dataset_id": entry["dataset_id"],
            "method": entry["method"],
            "family": "classical",
            "implementation": "scikit_learn",
            "backend": "sklearn_fit_numpy_portable_predict",
            "seed": entry["seed"],
            "split": split,
            "track": "matched_subset",
            "status": "succeeded",
            "metrics": metric_rows(values, config["task"]),
            "efficiency": efficiency,
            "config": entry["config"],
            "model_contract": entry["model_contract"],
            "provenance": provenance(entry, config),
            "code": config["code"],
            "device": "cpu",
            "test_seal_id": seal_id,
            "artifacts": artifacts,
            "started_at": started,
            "finished_at": finished,
        }
    view = DatasetView(entry["dataset_id"])
    spec = read_json(view.split_dir / "split_contract.json")
    document = {
        "schema_version": "3.0.0",
        "evidence_kind": "measured",
        "run_id": entry["run_id"],
        "dataset_id": entry["dataset_id"],
        "split_id": spec["split_id"],
        "split_sha256": config["provenance"]["split_sha256"],
        "config_sha256": entry["config"]["sha256"],
        "code": config["code"],
        "environment_sha256": config["environment_sha256"],
        "model_contract": entry["model_contract"],
        "implementation": entry["method"],
        "backend": {"scratch": "numpy", "keras": "tensorflow", "pytorch": "torch"}[entry["method"]],
        "family": "mlp",
        "track": "matched_subset",
        "seed": entry["seed"],
        "subset_sha256": config["provenance"]["subset_sha256"],
        "status": "succeeded",
        "split": split,
        "test_seal_id": seal_id,
        "device": "cpu" if entry["method"] == "scratch" else "cuda:0",
        "dtype": "float32",
        "batch_size": config["training"]["batch_size"],
        "started_at": started,
        "finished_at": finished,
        "metrics": metric_rows(values, config["task"]),
        "efficiency": efficiency,
        "artifacts": artifacts,
        "warnings": [
            "Retrospective group split; no clinical or market-deployment claim",
            "RAM is process-lifetime peak RSS; VRAM is framework allocator only",
            "Inference includes transfer; TensorFlow graph versus PyTorch/NumPy eager",
        ],
        "dataset_sha256": config["provenance"]["dataset_sha256"],
        "comparison_group_sha256": entry["comparison_group_sha256"],
        "protocol_sha256": config["protocol_sha256"],
        "preprocessing": view.contract()["preprocessing"],
    }
    validate_document("run", document, publish=True)
    return document



#### train_run

In [ ]:
%%inline_module src.ch02_ml.experiment src/ch02_ml/experiment.py both
def train_run(run):
    release = check_sources()
    entry = next(row for row in release["runs"] if row["run_id"] == run)
    for record in (entry["config"], entry["model_contract"]):
        if sha256_file(ROOT / record["path"]) != record["sha256"]:
            raise ValueError("Declared configuration/contract changed before training")
    folder = BASE / "runs" / run
    selection_path = folder / "selection.json"
    if selection_path.exists():
        selected = read_json(selection_path)
        for record in (
            selected["checkpoint"],
            selected["checkpoint_metadata"],
            selected["validation_metrics"],
            selected["validation_predictions"],
        ):
            if sha256_file(ROOT / record["path"]) != record["sha256"]:
                raise ValueError("Completed run artifact changed")
        return {"run_id": run, "status": "reused"}
    if shutil.disk_usage(ROOT).free < 5 * 1024**3:
        raise OSError("Host C free storage below 5 GiB reserve")
    claim = folder / "training_claim.json"
    if claim.exists():
        old = read_json(claim)
        if old["pid"] != os.getpid() and (ROOT.__class__(f"/proc/{old['pid']}")).exists():
            raise ValueError("A training process still owns this run")
    atomic_json(
        claim,
        {
            "pid": os.getpid(),
            "run_id": run,
            "config_sha256": entry["config"]["sha256"],
            "started_at": datetime.now(UTC).isoformat(),
        },
    )
    config = read_json(ROOT / entry["config"]["path"])
    method = entry["method"]
    task = config["task"]
    view = DatasetView(entry["dataset_id"])
    backend = configure(method, entry["seed"])
    device = "gpu" if method in {"keras", "pytorch"} else "cpu"
    started_at = datetime.now(UTC).isoformat()
    prep_start = time.perf_counter()
    data = cache_data(entry["dataset_id"])
    preprocessing_seconds = time.perf_counter() - prep_start
    monitor = DeviceMonitor(backend, device)
    monitor.start()
    training_start = time.perf_counter()
    if method in IMPLEMENTATIONS:
        weights = canonical_weights(config["plan"], entry["seed"])
        if (
            canonical_sha256({k: v.tolist() for k, v in weights.items()})
            != config["canonical_initial_weights_sha256"]
        ):
            raise ValueError("Initialization hash mismatch")
        trainer = make_trainer(method, config["plan"], weights)
        # Paired first-batch checks use only fixed TRAIN samples, never TEST.
        reference = NumpyNetwork(config["plan"], weights)
        x = data["train_x"][:4]
        y = data["train_y"][:4]
        np.testing.assert_allclose(trainer.predict(x), reference.forward(x), rtol=1e-4, atol=1e-5)
        if method != "scratch":
            _, reference_grad, _ = reference.loss_gradients(x, y, task)
            _, native_grad = trainer.model.loss_gradients(x, y, task)
            gradient_parity(reference_grad, native_grad)
        resumed = load_progress(run, entry["config"]["sha256"])
        result = fit(
            trainer,
            data["train_x"],
            data["train_y"],
            data["validation_x"],
            data["validation_y"],
            task,
            seed=entry["seed"],
            budget=config["training"],
            class_weights=config["class_weights"],
            progress=lambda state: save_progress(run, state, entry["config"]["sha256"]),
            resume=resumed,
        )
        weights = result["weights"]
        raw = predict_batches(trainer, data["validation_x"])
        predict = lambda value: predict_batches(trainer, value)
        history = result["history"]
        best_epoch = result["best_epoch"]
        train_seconds = result["train_seconds"]
        reference = NumpyNetwork(config["plan"], weights)
        np.testing.assert_allclose(
            trainer.predict(data["train_x"][:4]), reference.forward(data["train_x"][:4]), rtol=1e-4, atol=1e-5
        )
        portable_meta = None
    else:
        fitted, portable = fit_baseline(
            method,
            data["train_x"],
            data["train_y"],
            task,
            seed=entry["seed"],
            settings=config["forest"],
            class_weights=config["class_weights"],
        )
        train_seconds = time.perf_counter() - training_start
        raw = portable.predict(data["validation_x"])
        expected = (
            fitted.predict_proba(data["validation_x"])
            if task == "binary"
            else fitted.predict(data["validation_x"]).reshape(-1, 1)
        )
        np.testing.assert_allclose(raw, expected, rtol=1e-5, atol=1e-5)
        weights = portable.arrays
        portable_meta = portable.metadata
        predict = portable.predict
        history = []
        best_epoch = None
    memory = monitor.finish()
    timing = inference_timing(lambda: predict(data["train_x"][:1]), monitor.synchronize)
    if device == "gpu" and not 0 < memory["peak_vram_bytes"] < 8 * 1024**3:
        raise ValueError("Missing/excessive GPU allocator measurement")
    values, predictions, calibration_info = metrics_for(
        data["validation_target_original"], raw, task, view.adapter
    )
    diagnostics = {
        "calibration": calibration_info,
        "groups": group_diagnostics(
            data["validation_target_original"],
            predictions[:, 1] if task == "binary" else predictions[:, 0],
            data["validation_groups"],
            task,
        ),
        "groups_are_descriptive_only": True,
    }
    meta = ArtifactStore(MODELS / run).save_checkpoint("best", weights, provenance(entry, config))
    checkpoint = ref(MODELS / run / "best.npz")
    checkpoint_metadata = ref(MODELS / run / "best.json")
    loaded = ArtifactStore(MODELS / run).load_checkpoint(
        "best", provenance(entry, config), expected_manifest_sha256=checkpoint_metadata["sha256"]
    )
    cpu_output = (
        NumpyNetwork(config["plan"], loaded).forward(data["train_x"][:4])
        if method in IMPLEMENTATIONS
        else PortableBaseline(portable_meta, loaded).predict(data["train_x"][:4])
    )
    native_output = predict(data["train_x"][:4])
    np.testing.assert_allclose(cpu_output, native_output, rtol=1e-4, atol=1e-5)
    write_once(
        folder / "cpu_demo.json",
        {
            "status": "PASS",
            "input_split": "train",
            "ids": data["train_ids"][:4].tolist(),
            "canonical_x_sha256": array_sha256(data["train_x"][:4]),
            "raw_output": cpu_output.tolist(),
            "max_absolute_error": float(np.max(np.abs(cpu_output - native_output))),
            "checkpoint": checkpoint,
            "load_policy": "numpy_no_pickle",
            "frameworks_imported_for_cpu_predict": False,
        },
    )
    if portable_meta is not None:
        write_once(folder / "baseline_metadata.json", portable_meta)
    write_once(folder / "history.json", history)
    write_once(
        folder / "validation_details.json",
        {
            "metrics": values,
            "diagnostics": diagnostics,
            "memory": memory,
            "preprocessing_seconds": preprocessing_seconds,
            "platform": platform.platform(),
            "python": platform.python_version(),
            "numpy": np.__version__,
            "inference_scope": "fixed TRAIN sample; model forward and host transfer, no preprocessing or inverse target",
        },
    )
    prediction_ref = save_predictions(
        run, "validation", data["validation_ids"].tolist(), data["validation_target_original"], predictions
    )
    efficiency = {
        "train_seconds": train_seconds,
        "inference_p50_ms": timing["inference_p50_ms"],
        "inference_p95_ms": timing["inference_p95_ms"],
        "warmup_batches": 5,
        "timed_repeats": 30,
        "peak_ram_bytes": memory["process_peak_ram_bytes"],
        "peak_vram_bytes": memory["peak_vram_bytes"],
        "checkpoint_bytes": meta["size_bytes"],
    }
    finished = datetime.now(UTC).isoformat()
    record = run_record(
        entry,
        config,
        "validation",
        values,
        efficiency,
        started_at,
        finished,
        [checkpoint, checkpoint_metadata, prediction_ref],
    )
    metric_path = METRICS / "validation" / f"{run}.json"
    write_once(metric_path, record)
    selection = {
        "run_id": run,
        "status": "succeeded",
        "lifecycle": "trained_validation_selected",
        "inference_ready": True,
        "selection_split": "validation",
        "best_epoch": best_epoch,
        "checkpoint": checkpoint,
        "checkpoint_metadata": checkpoint_metadata,
        "validation_predictions": prediction_ref,
        "validation_metrics": ref(metric_path),
        "history": ref(folder / "history.json"),
        "details": ref(folder / "validation_details.json"),
        "model_contract": entry["model_contract"],
        "config": entry["config"],
        "comparison_group_sha256": entry["comparison_group_sha256"],
        "classical_metadata": ref(folder / "baseline_metadata.json") if portable_meta else None,
        "cpu_loadable": True,
        "cpu_demo": ref(folder / "cpu_demo.json"),
        "test_evaluation_executed": False,
    }
    write_once(selection_path, selection)
    return {
        "run_id": run,
        "status": "succeeded",
        "epochs": len(history),
        "best_epoch": best_epoch,
        "primary_metric": values[dataset_spec(entry["dataset_id"])["primary_metric"]]["value"],
        "train_seconds": train_seconds,
    }



#### evaluate_run

In [ ]:
%%inline_module src.ch02_ml.experiment src/ch02_ml/experiment.py both
def evaluate_run(run):
    release = check_sources()
    entry = next(row for row in release["runs"] if row["run_id"] == run)
    config = read_json(ROOT / entry["config"]["path"])
    dataset = entry["dataset_id"]
    gate = EvaluationGate(BASE / "test_seals" / dataset)
    gate.claim(run, release["code"]["dirty_tree_sha256"])
    manifest = read_json(gate.root / "manifest.json")
    for record in manifest["artifacts"]:
        if sha256_file(ROOT / record["path"]) != record["sha256"]:
            raise ValueError("TEST approved artifact changed")
    selection = read_json(BASE / "runs" / run / "selection.json")
    weights = ArtifactStore(MODELS / run).load_checkpoint(
        "best", provenance(entry, config), expected_manifest_sha256=selection["checkpoint_metadata"]["sha256"]
    )
    configure(entry["method"], entry["seed"])
    started = datetime.now(UTC).isoformat()
    data = test_data(dataset, gate)
    if entry["method"] in IMPLEMENTATIONS:
        trainer = make_trainer(entry["method"], config["plan"], weights)
        raw = predict_batches(trainer, data["x"])
    else:
        raw = PortableBaseline(read_json(ROOT / selection["classical_metadata"]["path"]), weights).predict(
            data["x"]
        )
    values, predictions, calibration_info = metrics_for(
        data["target_original"], raw, config["task"], data["view"].adapter
    )
    prediction_ref = save_predictions(run, "test", data["ids"], data["target_original"], predictions)
    groups = groups_for_frame(dataset, data["raw"])
    details = {
        "metrics": values,
        "calibration": calibration_info,
        "groups": group_diagnostics(
            data["target_original"],
            predictions[:, 1] if config["task"] == "binary" else predictions[:, 0],
            groups,
            config["task"],
        ),
        "diagnostic_only_no_tuning": True,
    }
    write_once(BASE / "runs" / run / "test_details.json", details)
    old = read_json(ROOT / selection["validation_metrics"]["path"])
    record = run_record(
        entry,
        config,
        "test",
        values,
        old["efficiency"],
        started,
        datetime.now(UTC).isoformat(),
        [selection["checkpoint"], prediction_ref],
        gate.state()["seal_id"],
    )
    path = METRICS / "test" / f"{run}.json"
    write_once(path, record)
    gate.record(
        run,
        {
            "status": "PASS",
            "predictions": prediction_ref,
            "metrics": ref(path),
            "details": ref(BASE / "runs" / run / "test_details.json"),
        },
    )
    return {"run_id": run, "status": "PASS", "test_prediction_count": len(data["ids"])}



### `src.ch03_cnn.protocol`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch03_cnn.protocol src/ch03_cnn/protocol.py both
"""Prospective CNN experiment contract; configuration must precede updates."""

import subprocess

from pathlib import Path

from scripts.prepare_phase25 import CORE_FILES

from src.ch02_ml.protocol import PHASE26_FILES

from src.common.artifact_core import code_fingerprint, sha256_file

from src.common.contracts import PROTOCOL, ROOT, read_json

BASE = ROOT / "artifacts/manifests/phase27"

MODELS = ROOT / "artifacts/models/v3/ch03_cnn"

METRICS = ROOT / "artifacts/metrics/phase27"

PREDICTIONS = ROOT / "artifacts/predictions/phase27"

DATASETS = ("ch3_idrid", "ch3_eurosat", "ch3_cifar10")

SEEDS = (17, 42, 73)

IMPLEMENTATIONS = ("scratch", "keras", "pytorch")

SOURCE_FILES = sorted(
    set(
        CORE_FILES
        + PHASE26_FILES
        + [
            f"src/ch03_cnn/{name}.py"
            for name in ("image_cache", "training", "workflow", "protocol", "experiment")
        ]
        + [
            "scripts/prepare_phase27.py",
            "scripts/phase27_worker.py",
            "scripts/phase27_controller.py",
            "experiments/configs/v3/phase27_coverage.ini",
        ]
        + [
            f"src/common/{name}.py"
            for name in (
                "contracts",
                "acquisition",
                "source_inspection",
                "provenance",
                "eda_sources",
                "eda_artifacts",
                "data_v3",
            )
        ]
    )
)



#### ref

In [ ]:
%%inline_module src.ch03_cnn.protocol src/ch03_cnn/protocol.py both
def ref(path):
    return {"path": Path(path).relative_to(ROOT).as_posix(), "sha256": sha256_file(path)}



#### code_info

In [ ]:
%%inline_module src.ch03_cnn.protocol src/ch03_cnn/protocol.py both
def code_info():
    head = subprocess.run(
        ["git", "rev-parse", "HEAD"], cwd=ROOT, check=True, capture_output=True, text=True
    ).stdout.strip()
    return {"git_commit": head, "dirty_tree_sha256": code_fingerprint(ROOT, SOURCE_FILES)}



#### run_ids

In [ ]:
%%inline_module src.ch03_cnn.protocol src/ch03_cnn/protocol.py both
def run_ids():
    return [
        f"{dataset}_{'cnn_' + method if method in IMPLEMENTATIONS else method}_s{seed}"
        for dataset in DATASETS
        for seed in SEEDS
        for method in (*IMPLEMENTATIONS, "majority")
    ]



#### check_sources

In [ ]:
%%inline_module src.ch03_cnn.protocol src/ch03_cnn/protocol.py both
def check_sources():
    release = read_json(BASE / "release.json")
    if (
        release["code"] != code_info()
        or release["protocol"] != ref(PROTOCOL)
        or release["environment"] != ref(ROOT / "uv.lock")
    ):
        raise ValueError("CNN declared source/protocol/environment changed; versioned revision required")
    for record in release["source_files"]:
        if ref(ROOT / record["path"]) != record:
            raise ValueError("CNN source hash mismatch")
    for entry in release["runs"]:
        for key in ("config", "model_contract"):
            if ref(ROOT / entry[key]["path"]) != entry[key]:
                raise ValueError("CNN config/contract hash mismatch")
    return release



### `src.ch04_rnn_lstm.diagnostics`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch04_rnn_lstm.diagnostics src/ch04_rnn_lstm/diagnostics.py both
"""Fixed TRAIN input-gradient illustration, not causal importance or TEST tuning."""

import numpy as np

from src.common.numpy_engine import NumpyNetwork



#### gradient_diagnostics

In [ ]:
%%inline_module src.ch04_rnn_lstm.diagnostics src/ch04_rnn_lstm/diagnostics.py both
def gradient_diagnostics(plan, weights, x, y):
    model = NumpyNetwork(plan, weights)
    loss, gradients, dx = model.loss_gradients(x, y, "mse")
    per_time = np.sqrt(np.mean(dx.astype("float64") ** 2, axis=(0, 2)))
    first, last = float(per_time[0]), float(per_time[-1])
    return {
        "input_split": "train",
        "loss": loss,
        "input_gradient_rms_by_time_oldest_to_newest": per_time.tolist(),
        "oldest_to_newest_ratio": first / last if last > 0 else None,
        "ratio_undefined_reason": None if last > 0 else "Newest input gradient is zero",
        "weight_gradient_l2": {
            key: float(np.linalg.norm(value.astype("float64"))) for key, value in gradients.items()
        },
        "recurrent_spectral_norms": {
            key: float(np.linalg.norm(value.astype("float64"), 2))
            for key, value in weights.items()
            if key.endswith("/u")
        },
        "scope": "Gradient of normalized MSE on first fixed TRAIN windows; descriptive local sensitivity, not causal attribution or proof of long-memory ability",
    }



### `src.ch04_rnn_lstm.protocol`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch04_rnn_lstm.protocol src/ch04_rnn_lstm/protocol.py both
"""Phase28 validation-only declaration; Phase29 is the only joint TEST owner."""

from pathlib import Path

from src.ch03_cnn.protocol import SOURCE_FILES as PREVIOUS_FILES

from src.common.artifact_core import code_fingerprint, sha256_file

from src.common.contracts import PROTOCOL, ROOT, read_json

BASE = ROOT / "artifacts/manifests/phase28"

MODELS = ROOT / "artifacts/models/v3/ch04_rnn"

METRICS = ROOT / "artifacts/metrics/phase28"

PREDICTIONS = ROOT / "artifacts/predictions/phase28"

DATASETS = ("ch4_gold_pink_sheet", "ch4_bis_house_prices", "ch4_household_power")

SEEDS = (17, 42, 73)

IMPLEMENTATIONS = ("scratch", "keras", "pytorch")

FOLDS = (None, 1, 2, 3)

SOURCE_FILES = sorted(
    set(
        PREVIOUS_FILES
        + [
            f"src/ch04_rnn_lstm/{name}.py"
            for name in ("window_data", "training", "diagnostics", "protocol", "experiment")
        ]
        + [
            "scripts/prepare_phase28.py",
            "scripts/phase28_controller.py",
            "scripts/phase28_worker.py",
            "scripts/phase28_preflight.py",
            "experiments/configs/v3/phase28_coverage.ini",
        ]
    )
)



#### ref

In [ ]:
%%inline_module src.ch04_rnn_lstm.protocol src/ch04_rnn_lstm/protocol.py both
def ref(path):
    return {"path": Path(path).relative_to(ROOT).as_posix(), "sha256": sha256_file(path)}



#### label

In [ ]:
%%inline_module src.ch04_rnn_lstm.protocol src/ch04_rnn_lstm/protocol.py both
def label(fold):
    return "main" if fold is None else f"fold{fold}"



#### run_id

In [ ]:
%%inline_module src.ch04_rnn_lstm.protocol src/ch04_rnn_lstm/protocol.py both
def run_id(dataset, method, seed, fold):
    return f"{dataset}_{'rnn_' if method in IMPLEMENTATIONS else ''}{method}_s{seed}_{label(fold)}"



#### code_info

In [ ]:
%%inline_module src.ch04_rnn_lstm.protocol src/ch04_rnn_lstm/protocol.py both
def code_info():
    import subprocess

    head = subprocess.run(
        ["git", "rev-parse", "HEAD"], cwd=ROOT, check=True, capture_output=True, text=True
    ).stdout.strip()
    return {"git_commit": head, "dirty_tree_sha256": code_fingerprint(ROOT, SOURCE_FILES)}



#### checked

In [ ]:
%%inline_module src.ch04_rnn_lstm.protocol src/ch04_rnn_lstm/protocol.py both
def checked(record):
    path = (ROOT / record["path"]).resolve()
    if not path.is_relative_to(ROOT.resolve()) or ref(path) != record:
        raise ValueError("Approved artifact is missing, changed or escaping workspace")
    return path



#### check_sources

In [ ]:
%%inline_module src.ch04_rnn_lstm.protocol src/ch04_rnn_lstm/protocol.py both
def check_sources():
    release = read_json(BASE / "release.json")
    if (
        release["code"] != code_info()
        or release["protocol"] != ref(PROTOCOL)
        or release["environment"] != ref(ROOT / "uv.lock")
    ):
        raise ValueError("Declared recurrent source/protocol/environment changed; revision required")
    for record in release["source_files"]:
        checked(record)
    for entry in release["runs"]:
        for key in ("config", "model_contract"):
            checked(entry[key])
    return release



### `src.ch04_rnn_lstm.training`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch04_rnn_lstm.training src/ch04_rnn_lstm/training.py both
"""Canonical recurrent Adam states and batch-only RNN training; no TEST API."""

import time

import numpy as np

from src.ch02_ml.training import KerasTrainer, TorchTrainer, validate_state

from src.common.contracts import canonical_sha256

from src.common.native_training import NativeAdam

from src.common.numpy_engine import loss_and_gradient

from src.common.training_core import EarlyStopping, batch_order



#### RnnTorchTrainer

In [ ]:
%%inline_module src.ch04_rnn_lstm.training src/ch04_rnn_lstm/training.py both
class RnnTorchTrainer(TorchTrainer):
    def export_state(self):
        weights, slots = self.weights(), {"m": {}, "v": {}}
        for name, param, _, key in self.model.parameter_pairs():
            state = self.optimizer.optimizer.state.get(param, {})
            for destination, field in (("m", "exp_avg"), ("v", "exp_avg_sq")):
                value = state[field].detach().cpu().numpy() if state else np.zeros_like(weights[name])
                if state and key in {"w", "u"}:
                    value = value.T
                slots[destination][name] = np.ascontiguousarray(value)
        return {"weights": weights, **slots, "iterations": self.iterations}

    def restore_state(self, state):
        validate_state(state, self.weights())
        self.assign(state["weights"])
        torch = self.model.torch
        for name, param, _, key in self.model.parameter_pairs():
            target = self.optimizer.optimizer.state[param]
            target["step"] = torch.tensor(float(state["iterations"]), dtype=torch.float32)
            for field, source in (("exp_avg", "m"), ("exp_avg_sq", "v")):
                value = state[source][name].T if key in {"w", "u"} else state[source][name]
                target[field] = torch.as_tensor(
                    np.ascontiguousarray(value), dtype=param.dtype, device=self.model.device
                ).clone()



#### RnnKerasTrainer

In [ ]:
%%inline_module src.ch04_rnn_lstm.training src/ch04_rnn_lstm/training.py both
class RnnKerasTrainer(KerasTrainer):
    def __init__(self, plan, weights, device="gpu", jit_compile=True):
        from src.common.backend_models import KerasBackend

        self.model = KerasBackend(plan, weights, device)
        self.optimizer = NativeAdam(self.model)
        tf = self.model.tf
        with tf.device(self.model.device):
            self.step_counter = tf.Variable(0, dtype=tf.int64, trainable=False)
        signature = [
            tf.TensorSpec([None, *plan["input_shape"]], tf.float32),
            tf.TensorSpec([None, *plan["output_shape"]], tf.float32),
            tf.TensorSpec([None], tf.float32),
        ]
        self._steps = {
            "mse": tf.function(
                self._graph_step("mse"), input_signature=signature, autograph=False, jit_compile=jit_compile
            )
        }
        self._predict = tf.function(
            lambda x: self.model.model(x, training=False),
            input_signature=[signature[0]],
            autograph=False,
            jit_compile=jit_compile,
        )

    def step(self, x, y, task, sample_weight):
        # XLA may elide tf.debugging assertions, so enforce these at the host boundary too.
        if (
            len(x) != len(y)
            or np.asarray(sample_weight).shape != (len(x),)
            or not np.isfinite(x).all()
            or not np.isfinite(y).all()
            or not np.isfinite(sample_weight).all()
            or np.any(sample_weight < 0)
            or np.sum(sample_weight) <= 0
        ):
            raise ValueError("Invalid/nonfinite recurrent batch or weights")
        result = super().step(x, y, task, sample_weight)
        if not np.isfinite(list(result.values())).all():
            raise ValueError("Nonfinite recurrent loss/gradient")
        return result



#### predict_windows

In [ ]:
%%inline_module src.ch04_rnn_lstm.training src/ch04_rnn_lstm/training.py both
def predict_windows(trainer, data, ids, batch_size=32):
    if not ids or batch_size < 1:
        raise ValueError("Invalid prediction batch")
    result = np.concatenate(
        [trainer.predict(data.batch(ids[i : i + batch_size])[0]) for i in range(0, len(ids), batch_size)]
    )
    if not np.isfinite(result).all():
        raise ValueError("Nonfinite RNN prediction")
    return result



#### fit_windows

In [ ]:
%%inline_module src.ch04_rnn_lstm.training src/ch04_rnn_lstm/training.py both
def fit_windows(trainer, data, *, seed, budget, progress=None, resume=None):
    stop = EarlyStopping(budget["patience"], budget["min_delta"])
    history, best, elapsed = [], None, 0.0
    if resume is not None:
        trainer.restore_state(resume["trainer"])
        history, best, elapsed = resume["history"].copy(), resume["best_weights"], resume["elapsed_seconds"]
        for key, value in resume["early_stopping"].items():
            setattr(stop, key, value)
    started = time.perf_counter()
    val_y = np.concatenate(
        [
            data.batch(data.validation_ids[i : i + budget["batch_size"]])[1]
            for i in range(0, len(data.validation_ids), budget["batch_size"])
        ]
    )
    for epoch in range(len(history), budget["max_epochs"]):
        if stop.bad_epochs >= stop.patience:
            break
        order = batch_order(len(data.train_ids), budget["batch_size"], seed, epoch)
        loss_sum, count, norms = 0.0, 0, []
        for positions in order:
            if elapsed + time.perf_counter() - started >= budget["wall_time_limit_minutes"] * 60:
                raise TimeoutError("Locked recurrent wall-time budget exceeded")
            x, y = data.batch([data.train_ids[int(i)] for i in positions])
            output = trainer.step(x, y, "mse", np.ones(len(y), "float32"))
            loss_sum += output["loss"] * len(y)
            count += len(y)
            norms.append(output["gradient_global_norm_before_clip"])
        predicted = predict_windows(trainer, data, data.validation_ids, budget["batch_size"])
        loss = loss_and_gradient(predicted, val_y, "mse")[0]
        improved, terminate = stop.observe(loss, epoch)
        if improved:
            best = trainer.weights()
        history.append(
            {
                "epoch": epoch,
                "train_loss": loss_sum / count,
                "validation_loss": loss,
                "gradient_norm_mean": float(np.mean(norms)),
                "gradient_norm_max": float(np.max(norms)),
                "gradient_clip_fraction": float(np.mean(np.asarray(norms) > 1)),
                "batch_order_sha256": canonical_sha256(order),
            }
        )
        if progress:
            progress(
                {
                    "history": history.copy(),
                    "best_weights": best,
                    "early_stopping": {
                        key: getattr(stop, key)
                        for key in ("best", "best_epoch", "bad_epochs", "patience_reference")
                    },
                    "trainer": trainer.export_state(),
                    "elapsed_seconds": elapsed + time.perf_counter() - started,
                }
            )
        print(
            f"epoch={epoch + 1} train_loss={history[-1]['train_loss']:.6f} validation_loss={loss:.6f} best_epoch={stop.best_epoch + 1}",
            flush=True,
        )
        if terminate:
            break
    if best is None:
        raise ValueError("No finite validation-selected RNN weights")
    trainer.assign(best)
    return {
        "history": history,
        "best_epoch": stop.best_epoch,
        "best_validation_loss": stop.best,
        "weights": best,
        "train_seconds": elapsed + time.perf_counter() - started,
        "optimizer_steps": trainer.iterations,
    }



### `src.ch04_rnn_lstm.window_data`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch04_rnn_lstm.window_data src/ch04_rnn_lstm/window_data.py both
"""Small row-feature cache; only causal development windows materialize as batches."""

import numpy as np

from src.common.artifact_core import sha256_file

from src.common.contracts import canonical_sha256, read_json

from src.common.data_access import DatasetView



#### WindowData

In [ ]:
%%inline_module src.ch04_rnn_lstm.window_data src/ch04_rnn_lstm/window_data.py both
class WindowData:
    def __init__(self, dataset_id, fold=None):
        self.view = DatasetView(dataset_id, fold)
        if self.view.kind != "timeseries":
            raise ValueError("Require a frozen time-series dataset")
        self.dataset_id, self.fold, self.adapter = dataset_id, fold, self.view.adapter
        self.train_ids = self.view.ids("train")
        self.validation_ids = self.view.ids("validation")
        self.allowed = set(self.view.ids("train", matched=False)) | set(self.validation_ids)
        self.view._load()
        self.frame = self.view._frame
        self.rows = {key: row.copy() for key, row in self.view._index.items()}
        self.lookback = self.view.spec["lookback"]
        self.horizons = self.view.spec["horizons"]
        self.reconstructed_fold_ids = sorted(self.allowed - set(self.rows))
        if self.reconstructed_fold_ids and fold is None:
            raise ValueError("Main split contains missing frozen window IDs")
        origin_positions = {
            f"{series}/{timestamp.isoformat()}": index
            for index, (series, timestamp) in enumerate(zip(self.frame.series, self.frame.time, strict=True))
        }
        validation_set = set(self.validation_ids)
        for key in self.reconstructed_fold_ids:
            origin = origin_positions[key]
            targets = [origin + horizon for horizon in self.horizons]
            start = origin - self.lookback + 1
            if start < 0 or max(targets) >= len(self.frame):
                raise ValueError("Reconstructed fold window outside source")
            self.rows[key] = {
                "id": key,
                "series": str(self.frame.series.iloc[origin]),
                "partition": "validation" if key in validation_set else "train",
                "context_start": self.frame.time.iloc[start].isoformat(),
                "context_end": self.frame.time.iloc[origin].isoformat(),
                "target_times": [self.frame.time.iloc[i].isoformat() for i in targets],
                "start_index": start,
                "origin_index": origin,
                "target_indices": targets,
            }
        self.features = np.full((len(self.frame), self.adapter.state["output_width"]), np.nan, "float32")
        context_rows = set()
        series_values = self.frame.series.to_numpy()
        if fold is not None:
            limits = read_json(self.view.split_dir.parent / "rolling_folds.json")[fold - 1]
            train_end = np.datetime64(limits["train_end"])
            val_end = np.datetime64(limits["validation_end"])
        else:
            parts = read_json(self.view.split_dir / "split_contract.json")["partitions"]
            train_end = np.datetime64(
                next(p["target_end"] for p in parts if p["name"] == "train").replace("+00:00", "")
            )
            val_end = np.datetime64(
                next(p["target_end"] for p in parts if p["name"] == "validation").replace("+00:00", "")
            )
        times = self.frame.time.to_numpy(dtype="datetime64[ns]")
        for key in self.allowed:
            row = self.rows[key]
            indices = list(range(row["start_index"], row["origin_index"] + 1))
            target_times = times[row["target_indices"]]
            is_val = key in validation_set
            if (
                len(indices) != self.lookback
                or min(row["target_indices"]) <= row["origin_index"]
                or max(row["target_indices"]) >= len(self.frame)
                or not np.all(series_values[indices + row["target_indices"]] == row["series"])
                or np.any(target_times > val_end)
                or (np.any(target_times <= train_end) if is_val else np.any(target_times > train_end))
            ):
                raise ValueError("Noncausal, cross-series or invalid frozen window")
            row["partition"] = "validation" if is_val else "train"
            context_rows.update(indices)
        positions = sorted(context_rows)
        self.features[positions] = self.adapter.transform(
            self.frame.iloc[positions][self.adapter.state["input_columns"]]
        )
        self.feature_bytes = self.features.nbytes
        if self.feature_bytes > 64 * 1024**2:
            raise ValueError("Row feature cache exceeds 64 MiB budget")
        self.provenance = self.view.provenance.copy()
        if fold is not None:
            self.provenance["split_sha256"] = canonical_sha256(
                {
                    name: sha256_file(self.view.split_dir / f"{name}.ids.jsonl.gz")
                    for name in ("train", "validation", "matched_train")
                }
            )

    def selected(self, ids):
        if not ids or len(ids) != len(set(ids)) or any(key not in self.allowed for key in ids):
            raise ValueError("Empty/duplicate/unknown/TEST window IDs")
        return [self.rows[key] for key in ids]

    def targets(self, ids):
        rows = self.selected(ids)
        values = self.frame.target.to_numpy(dtype="float64")
        result = np.asarray([values[row["target_indices"]] for row in rows])
        if not np.isfinite(result).all():
            raise ValueError("Missing labels must never be imputed")
        return result

    def batch(self, ids):
        rows = self.selected(ids)
        positions = np.asarray([np.arange(row["start_index"], row["origin_index"] + 1) for row in rows])
        x = np.ascontiguousarray(self.features[positions])
        if not np.isfinite(x).all():
            raise ValueError("Nonfinite/sealed input features")
        return x, self.adapter.target(self.targets(ids))

    def series(self, ids):
        return [row["series"] for row in self.selected(ids)]

    def baseline(self, ids, method):
        if method not in {"last_value", "seasonal_naive"}:
            raise ValueError("Unknown causal baseline")
        values = self.frame.target.to_numpy(dtype="float64")
        result = []
        for row in self.selected(ids):
            start, origin = row["start_index"], row["origin_index"]
            history = values[start : origin + 1]
            observed = history[np.isfinite(history)]
            if not len(observed):
                raise ValueError("No observed history for causal baseline")
            last = observed[-1]
            seasonal = []
            for horizon in self.horizons:
                index = (
                    origin
                    + horizon
                    - int(np.ceil(horizon / self.view.spec["seasonality"])) * self.view.spec["seasonality"]
                )
                value = values[index] if start <= index <= origin else np.nan
                seasonal.append(float(value) if np.isfinite(value) else float(last))
            result.append(seasonal if method == "seasonal_naive" else [float(last)] * len(self.horizons))
        return np.asarray(result, dtype="float64")

    def contract(self):
        return {
            "dataset_id": self.dataset_id,
            "fold": self.fold,
            "train_count": len(self.train_ids),
            "validation_count": len(self.validation_ids),
            "train_ids_sha256": canonical_sha256(self.train_ids),
            "validation_ids_sha256": canonical_sha256(self.validation_ids),
            "feature_bytes": self.feature_bytes,
            "provenance": self.provenance,
            "input_shape": [self.lookback, self.features.shape[1]],
            "horizons": self.horizons,
            "adapter": self.view.contract(),
            "test_windows_materialized": False,
            "fold_index_revision": "phase28.1; source-index gaps reconstructed from immutable IDs/calendar, never alter splits/scalers",
            "reconstructed_fold_ids": self.reconstructed_fold_ids,
            "raw_read_scope": "Frozen full row table is read; transform/gather only development context rows; no TEST window predictions or labels used for selection",
            "baseline_policy": "Observed target history at/before origin; missing seasonal source falls back to last observed within lookback; no target imputation",
        }



### `src.ch04_rnn_lstm.experiment`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch04_rnn_lstm.experiment src/ch04_rnn_lstm/experiment.py both
"""Validation-only measured RNN runs, safe optimizer resume and CPU checkpoints."""

import gzip

import json

import os

import platform

import shutil

import time

import zipfile

from datetime import UTC, datetime

from pathlib import Path

import numpy as np

from src.ch02_ml.experiment import configure, provenance

from src.ch02_ml.protocol import array_sha256

from src.ch02_ml.test_gate import atomic_json

from src.ch02_ml.training import ScratchTrainer

from src.ch04_rnn_lstm.diagnostics import gradient_diagnostics

from src.ch04_rnn_lstm.protocol import (
    BASE,
    IMPLEMENTATIONS,
    METRICS,
    MODELS,
    PREDICTIONS,
    check_sources,
    checked,
    ref,
)

from src.ch04_rnn_lstm.training import RnnKerasTrainer, RnnTorchTrainer, fit_windows, predict_windows

from src.ch04_rnn_lstm.window_data import WindowData

from src.common.artifact_core import ArtifactStore, write_once

from src.common.contracts import ROOT, canonical_sha256, read_json, validate_document

from src.common.device_core import DeviceMonitor

from src.common.metric_core import regression_metrics

from src.common.model_core import canonical_weights

from src.common.numpy_engine import NumpyNetwork

from src.common.parity import gradient_parity

from src.common.training_core import inference_timing



#### make_trainer

In [ ]:
%%inline_module src.ch04_rnn_lstm.experiment src/ch04_rnn_lstm/experiment.py both
def make_trainer(method, plan, weights):
    if method == "scratch":
        return ScratchTrainer(plan, weights)
    if method == "pytorch":
        return RnnTorchTrainer(plan, weights, "gpu")
    if method == "keras":
        return RnnKerasTrainer(plan, weights, "gpu")
    raise ValueError("Unknown RNN implementation")



#### save_progress

In [ ]:
%%inline_module src.ch04_rnn_lstm.experiment src/ch04_rnn_lstm/experiment.py both
def save_progress(run, state, config_hash):
    folder = MODELS / run / "resume"
    folder.mkdir(parents=True, exist_ok=True)
    epoch = len(state["history"])
    path = folder / f"epoch_{epoch:03d}.npz"
    arrays = {
        f"{field}/{key}": value
        for field in ("weights", "m", "v")
        for key, value in state["trainer"][field].items()
    }
    arrays.update({f"best/{key}": value for key, value in state["best_weights"].items()})
    if any(value.dtype.kind not in "fiu" or not np.isfinite(value).all() for value in arrays.values()):
        raise ValueError("Unsafe optimizer state")
    with path.open("xb") as stream:
        np.savez_compressed(stream, **arrays)
    metadata = {key: value for key, value in state.items() if key not in {"trainer", "best_weights"}}
    metadata.update(
        {"iterations": state["trainer"]["iterations"], "config_sha256": config_hash, "arrays": ref(path)}
    )
    meta_path = folder / f"epoch_{epoch:03d}.json"
    write_once(meta_path, metadata)
    atomic_json(MODELS / run / "progress.json", ref(meta_path))



#### load_progress

In [ ]:
%%inline_module src.ch04_rnn_lstm.experiment src/ch04_rnn_lstm/experiment.py both
def load_progress(run, config_hash):
    pointer = MODELS / run / "progress.json"
    if not pointer.exists():
        return None
    metadata = read_json(checked(read_json(pointer)))
    if metadata["config_sha256"] != config_hash:
        raise ValueError("Resume config changed")
    path = checked(metadata["arrays"])
    if path.stat().st_size > 64 * 1024**2:
        raise ValueError("Resume exceeds 64 MiB budget")
    with zipfile.ZipFile(path) as archive:
        if sum(item.file_size for item in archive.infolist()) > 64 * 1024**2:
            raise ValueError("Expanded optimizer archive exceeds 64 MiB budget")
    with np.load(path, allow_pickle=False) as archive:
        arrays = {key: archive[key] for key in archive.files}
    if sum(value.nbytes for value in arrays.values()) > 64 * 1024**2 or any(
        value.dtype.kind not in "fiu" or not np.isfinite(value).all() for value in arrays.values()
    ):
        raise ValueError("Unsafe expanded optimizer state")
    return {
        "history": metadata["history"],
        "early_stopping": metadata["early_stopping"],
        "elapsed_seconds": metadata["elapsed_seconds"],
        "trainer": {
            **{
                field: {
                    key.removeprefix(field + "/"): value
                    for key, value in arrays.items()
                    if key.startswith(field + "/")
                }
                for field in ("weights", "m", "v")
            },
            "iterations": metadata["iterations"],
        },
        "best_weights": {
            key.removeprefix("best/"): value for key, value in arrays.items() if key.startswith("best/")
        },
    }



#### metric_rows

In [ ]:
%%inline_module src.ch04_rnn_lstm.experiment src/ch04_rnn_lstm/experiment.py both
def metric_rows(values, unit):
    rows = []
    for horizon, group in [
        (None, values),
        *[(int(h), group) for h, group in values.get("by_horizon", {}).items()],
    ]:
        for name in ("rmse", "mae", "smape", "macro_series_rmse"):
            if name in group:
                rows.append(
                    {
                        "name": name,
                        **group[name],
                        "unit": "percent" if name == "smape" else unit,
                        "direction": "min",
                        "horizon": horizon,
                    }
                )
    return rows



#### record

In [ ]:
%%inline_module src.ch04_rnn_lstm.experiment src/ch04_rnn_lstm/experiment.py both
def record(entry, config, data, values, efficiency, started, artifacts):
    document = {
        "schema_version": "3.0.0",
        "evidence_kind": "measured",
        "run_id": entry["run_id"],
        "dataset_id": entry["dataset_id"],
        "split_id": f"{entry['dataset_id']}_phase28_{'main' if entry['fold'] is None else 'fold' + str(entry['fold'])}",
        "split_sha256": config["provenance"]["split_sha256"],
        "config_sha256": entry["config"]["sha256"],
        "code": config["code"],
        "environment_sha256": config["environment_sha256"],
        "model_contract": entry["model_contract"],
        "implementation": entry["method"],
        "backend": {"scratch": "numpy", "keras": "tensorflow", "pytorch": "torch"}.get(
            entry["method"], "numpy"
        ),
        "family": "rnn",
        "track": "matched_subset",
        "seed": entry["seed"],
        "subset_sha256": config["provenance"]["subset_sha256"],
        "status": "succeeded",
        "split": "validation",
        "test_seal_id": None,
        "device": "cuda:0" if entry["method"] in {"keras", "pytorch"} else "cpu",
        "dtype": "float32",
        "batch_size": config["training"]["batch_size"],
        "started_at": started,
        "finished_at": datetime.now(UTC).isoformat(),
        "metrics": metric_rows(values, config["unit"]),
        "efficiency": efficiency,
        "artifacts": artifacts,
        "warnings": [
            "Validation only; no TEST evaluation in Phase28",
            "Latest revised snapshots, retrospective observation-origin forecast; no real-time publication/vintage claim or investment advice",
            "Each rolling fold uses its frozen scaler; windows reset recurrent state",
            "Train wall includes validation/progress I/O; RAM is process lifetime, VRAM allocator only; CPU/GPU and graph/eager differ",
        ],
        "dataset_sha256": config["provenance"]["dataset_sha256"],
        "comparison_group_sha256": entry["comparison_group_sha256"],
        "protocol_sha256": config["protocol_sha256"],
        "preprocessing": data.view.contract()["preprocessing"],
    }
    if entry["method"] in IMPLEMENTATIONS:
        validate_document("run", document, publish=True)
    else:
        document.update(
            {
                "schema_version": "phase28.causal_baseline_run.1",
                "family": "classical",
                "implementation": "numpy",
                "method": entry["method"],
            }
        )
    return document



#### save_predictions

In [ ]:
%%inline_module src.ch04_rnn_lstm.experiment src/ch04_rnn_lstm/experiment.py both
def save_predictions(run, data, targets, predictions):
    path = PREDICTIONS / "validation" / f"{run}.jsonl.gz"
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("xb") as stream, gzip.GzipFile(filename="", fileobj=stream, mode="wb", mtime=0) as packed:
        for key, target, pred in zip(data.validation_ids, targets, predictions, strict=True):
            row = {
                "id": key,
                "series": data.rows[key]["series"],
                "target_times": data.rows[key]["target_times"],
                "target": target.tolist(),
                "prediction": pred.tolist(),
            }
            packed.write((json.dumps(row, sort_keys=True, allow_nan=False) + "\n").encode())
    return ref(path)



#### train_run

In [ ]:
%%inline_module src.ch04_rnn_lstm.experiment src/ch04_rnn_lstm/experiment.py both
def train_run(run):
    release = check_sources()
    entry = next(row for row in release["runs"] if row["run_id"] == run)
    folder = BASE / "runs" / run
    if (folder / "selection.json").exists():
        selected = read_json(folder / "selection.json")
        for value in selected.values():
            if isinstance(value, dict) and set(value) == {"path", "sha256"}:
                checked(value)
        return {"run_id": run, "status": "reused"}
    if shutil.disk_usage(ROOT).free < 5 * 1024**3:
        raise OSError("Host disk below 5 GiB reserve")
    claim = folder / "training_claim.json"
    if (
        claim.exists()
        and read_json(claim)["pid"] != os.getpid()
        and Path(f"/proc/{read_json(claim)['pid']}").exists()
    ):
        raise ValueError("Live worker owns recurrent run")
    atomic_json(claim, {"pid": os.getpid(), "run_id": run})
    config = read_json(checked(entry["config"]))
    method, started = entry["method"], datetime.now(UTC).isoformat()
    backend = configure(method, entry["seed"])
    tick = time.perf_counter()
    data = WindowData(entry["dataset_id"], entry["fold"])
    if data.contract() != read_json(checked(config["data_contract"])):
        raise ValueError("Declared recurrent data contract changed")
    x_demo, y_demo = data.batch(data.train_ids[:4])
    prep_seconds = time.perf_counter() - tick
    monitor = DeviceMonitor(backend, "gpu" if method in {"keras", "pytorch"} else "cpu")
    monitor.start()
    if method in IMPLEMENTATIONS:
        weights = canonical_weights(config["plan"], entry["seed"])
        if (
            canonical_sha256({k: v.tolist() for k, v in weights.items()})
            != config["canonical_initial_weights_sha256"]
        ):
            raise ValueError("Initialization changed")
        trainer = make_trainer(method, config["plan"], weights)
        reference = NumpyNetwork(config["plan"], weights)
        np.testing.assert_allclose(trainer.predict(x_demo), reference.forward(x_demo), atol=1e-5, rtol=1e-4)
        expected = reference.loss_gradients(x_demo, y_demo, "mse")[1]
        actual = expected if method == "scratch" else trainer.model.loss_gradients(x_demo, y_demo, "mse")[1]
        write_once(
            folder / "initial_parity.json",
            {
                "forward": "PASS",
                "gradient": gradient_parity(expected, actual),
                "input_split": "train",
                "ids": data.train_ids[:4],
            },
        )
        result = fit_windows(
            trainer,
            data,
            seed=entry["seed"],
            budget=config["training"],
            resume=load_progress(run, entry["config"]["sha256"]),
            progress=lambda state: save_progress(run, state, entry["config"]["sha256"]),
        )
        weights, history, best_epoch, train_seconds = (
            result["weights"],
            result["history"],
            result["best_epoch"],
            result["train_seconds"],
        )
        output = data.adapter.inverse_target(predict_windows(trainer, data, data.validation_ids))
        native_demo = trainer.predict(x_demo)
        predict_one = lambda: trainer.predict(x_demo[:1])
        write_once(
            folder / "gradient_diagnostics.json",
            {**gradient_diagnostics(config["plan"], weights, x_demo, y_demo), "ids": data.train_ids[:4]},
        )
    else:
        tick = time.perf_counter()
        weights = {"baseline_kind": np.asarray([int(method == "seasonal_naive")], dtype="int64")}
        train_seconds = time.perf_counter() - tick
        output = data.baseline(data.validation_ids, method)
        native_demo = data.baseline(data.train_ids[:4], method)
        predict_one = lambda: data.baseline(data.train_ids[:1], method)
        history, best_epoch = [], None
    memory = monitor.finish()
    if method in {"keras", "pytorch"} and not 0 < memory["peak_vram_bytes"] < 8 * 1024**3:
        raise ValueError("Invalid GPU memory measurement")
    timing = inference_timing(predict_one, monitor.synchronize)
    targets = data.targets(data.validation_ids)
    values = regression_metrics(targets, output, data.horizons, data.series(data.validation_ids))
    cp = ArtifactStore(MODELS / run).save_checkpoint("best", weights, provenance(entry, config))
    checkpoint, metadata = ref(MODELS / run / "best.npz"), ref(MODELS / run / "best.json")
    loaded = ArtifactStore(MODELS / run).load_checkpoint(
        "best", provenance(entry, config), expected_manifest_sha256=metadata["sha256"]
    )
    cpu = (
        NumpyNetwork(config["plan"], loaded).forward(x_demo)
        if method in IMPLEMENTATIONS
        else data.baseline(data.train_ids[:4], method)
    )
    np.testing.assert_allclose(cpu, native_demo, atol=1e-5, rtol=1e-4)
    write_once(
        folder / "cpu_demo.json",
        {
            "status": "PASS",
            "input_split": "train",
            "ids": data.train_ids[:4],
            "canonical_x_sha256": array_sha256(x_demo),
            "raw_output": cpu.tolist(),
            "max_absolute_error": float(np.max(np.abs(cpu - native_demo))),
            "checkpoint": checkpoint,
            "serving_backend": "numpy_cpu_portable",
            "training_implementation": method,
        },
    )
    write_once(folder / "history.json", history)
    details = {
        "metrics": values,
        "memory": memory,
        "best_epoch": best_epoch,
        "preprocessing_seconds": prep_seconds,
        "train_count": len(data.train_ids),
        "validation_count": len(data.validation_ids),
        "unit": config["unit"],
        "fold": entry["fold"],
        "platform": platform.platform(),
        "numpy": np.__version__,
        "test_evaluation_executed": False,
    }
    write_once(folder / "validation_details.json", details)
    predictions = save_predictions(run, data, targets, output)
    efficiency = {
        "train_seconds": train_seconds,
        "inference_p50_ms": timing["inference_p50_ms"],
        "inference_p95_ms": timing["inference_p95_ms"],
        "warmup_batches": timing["warmup"],
        "timed_repeats": timing["repeats"],
        "peak_ram_bytes": memory["process_peak_ram_bytes"],
        "peak_vram_bytes": memory["peak_vram_bytes"],
        "checkpoint_bytes": cp["size_bytes"],
    }
    metric_path = METRICS / "validation" / f"{run}.json"
    write_once(
        metric_path,
        record(entry, config, data, values, efficiency, started, [checkpoint, metadata, predictions]),
    )
    write_once(
        folder / "selection.json",
        {
            "run_id": run,
            "status": "succeeded",
            "lifecycle": "trained_validation_selected",
            "selection_split": "validation",
            "inference_ready": True,
            "best_epoch": best_epoch,
            "checkpoint": checkpoint,
            "checkpoint_metadata": metadata,
            "validation_metrics": ref(metric_path),
            "validation_predictions": predictions,
            "history": ref(folder / "history.json"),
            "details": ref(folder / "validation_details.json"),
            "cpu_demo": ref(folder / "cpu_demo.json"),
            "config": entry["config"],
            "model_contract": entry["model_contract"],
            "comparison_group_sha256": entry["comparison_group_sha256"],
            "test_evaluation_executed": False,
        },
    )
    return {
        "run_id": run,
        "status": "PASS",
        "epochs": len(history),
        "best_epoch": best_epoch,
        "rmse": values["rmse"]["value"],
        "train_seconds": train_seconds,
    }



### `src.ch04_rnn_lstm.export_r2`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch04_rnn_lstm.export_r2 src/ch04_rnn_lstm/export_r2.py both
"""Approved strict CPU export revision; frozen training remains unchanged.

Only a failed final CPU/GPU assertion in the exact frozen train_run frame may
enter the finalizer. All optimizer states, selected weights and native validation
outputs remain unchanged. The one already-dead process has explicitly missing
original training telemetry, represented by a separate versioned run schema.
"""

import copy

import platform

import time

from datetime import UTC, datetime

import numpy as np

from jsonschema import Draft202012Validator, FormatChecker

from referencing import Registry, Resource

from src.ch02_ml.experiment import configure, provenance

from src.ch02_ml.protocol import array_sha256

from src.ch04_rnn_lstm import experiment as frozen

from src.ch04_rnn_lstm.protocol import BASE, IMPLEMENTATIONS, METRICS, MODELS, checked, ref

from src.ch04_rnn_lstm.training import predict_windows

from src.ch04_rnn_lstm.window_data import WindowData

from src.common.artifact_core import ArtifactStore, write_once

from src.common.contracts import _finite, _run, load_schemas, read_json, validate_document

from src.common.device_core import DeviceMonitor

from src.common.metric_core import regression_metrics

from src.common.numpy_engine import NumpyNetwork

from src.common.training_core import inference_timing

ORPHAN = "ch4_household_power_rnn_pytorch_s17_main"

REVISION = BASE / "revisions/r2/revision.json"

RECOVERY_VERSION = "phase28.recovery_run.2"

MISSING_TELEMETRY = ("train_seconds", "peak_ram_bytes", "peak_vram_bytes")

LOST_TELEMETRY_WARNING = (
    "Original training wall/RAM/VRAM telemetry lost before export failure; null, excluded from cost aggregates. "
    "Recovery process measurements are not original training measurements."
)



#### check_revision

In [ ]:
%%inline_module src.ch04_rnn_lstm.export_r2 src/ch04_rnn_lstm/export_r2.py both
def check_revision():
    release = frozen.check_sources()
    revision = read_json(REVISION)
    for group in ("source_files", "preserved_selections", "preserved_orphan"):
        for record in revision[group]:
            checked(record)
    checked(revision["original_release"])
    checked(revision["authorization"])
    checked(revision["recovery_schema"])
    return release



#### compare_outputs

In [ ]:
%%inline_module src.ch04_rnn_lstm.export_r2 src/ch04_rnn_lstm/export_r2.py both
def compare_outputs(portable, native_cpu, native_gpu):
    arrays = [np.asarray(value) for value in (portable, native_cpu, native_gpu)]
    if any(a.shape != arrays[0].shape or not np.isfinite(a).all() for a in arrays):
        raise ValueError("Nonfinite or mismatched export outputs")

    def comparison(value):
        return {
            "status": "PASS" if np.isclose(arrays[0], value, rtol=1e-4, atol=1e-5).all() else "FAIL",
            "max_absolute_error": float(np.max(np.abs(arrays[0] - value))),
            "output": value.tolist(),
        }

    result = {"rtol": 1e-4, "atol": 1e-5,
              "cpu_reference": comparison(arrays[1]),
              "original_gpu_cpu_gate": comparison(arrays[2])}
    if result["cpu_reference"]["status"] != "PASS":
        raise AssertionError("Strict native CPU/portable CPU export parity failed")
    return result



#### native_cpu

In [ ]:
%%inline_module src.ch04_rnn_lstm.export_r2 src/ch04_rnn_lstm/export_r2.py both
def native_cpu(plan, weights, method, x):
    if method == "pytorch":
        from src.common.backend_models import TorchBackend

        return TorchBackend(plan, weights, "cpu").predict(x)
    if method == "keras":
        from src.common.backend_models import KerasBackend

        model = KerasBackend(plan, weights, "cpu")
        result = model.predict(x)
        model.keras.backend.clear_session()
        return result
    if method == "scratch":
        return NumpyNetwork(plan, weights).forward(x)
    raise ValueError("No native CPU reference for this implementation")



#### completed_progress

In [ ]:
%%inline_module src.ch04_rnn_lstm.export_r2 src/ch04_rnn_lstm/export_r2.py both
def completed_progress(progress, budget, weights):
    if not progress or not progress["history"]:
        raise ValueError("Completed immutable training state required")
    history, stop = progress["history"], progress["early_stopping"]
    if len(history) > budget["max_epochs"] or not (
        len(history) == budget["max_epochs"] or stop["bad_epochs"] >= budget["patience"]
    ):
        raise ValueError("Training not finished; finalization cannot add updates")
    if [row["epoch"] for row in history] != list(range(len(history))):
        raise ValueError("Training history is not contiguous")
    if set(weights) != set(progress["best_weights"]) or any(
        not np.array_equal(weights[k], progress["best_weights"][k]) for k in weights
    ):
        raise ValueError("Export checkpoint differs from the completed best state")
    if not 0 <= stop["best_epoch"] < len(history):
        raise ValueError("Invalid selected epoch")
    return stop["best_epoch"]



#### recovery_schema

In [ ]:
%%inline_module src.ch04_rnn_lstm.export_r2 src/ch04_rnn_lstm/export_r2.py both
def recovery_schema():
    schema = copy.deepcopy(load_schemas()["run"])
    schema["$id"] = "urn:ai-paper:phase28:recovery-run:2"
    schema["title"] = "Phase28 single dead-process recovery; unavailable training telemetry"
    schema["properties"]["schema_version"] = {"const": RECOVERY_VERSION}
    schema["properties"]["run_id"] = {"const": ORPHAN}
    schema["properties"]["split"] = {"const": "validation"}
    schema["properties"]["device"] = {"const": "cuda:0"}
    schema["properties"]["implementation"] = {"const": "pytorch"}
    schema["properties"]["warnings"]["contains"] = {"const": LOST_TELEMETRY_WARNING}
    for key in MISSING_TELEMETRY:
        schema["properties"]["efficiency"]["properties"][key] = {"type": "null"}
    return schema



#### validate_run

In [ ]:
%%inline_module src.ch04_rnn_lstm.export_r2 src/ch04_rnn_lstm/export_r2.py both
def validate_run(document):
    if document["schema_version"] != RECOVERY_VERSION:
        return validate_document("run", document, publish=True)
    schemas = load_schemas()
    registry = Registry().with_resources(
        (schema["$id"], Resource.from_contents(schema)) for schema in schemas.values()
    )
    _finite(document)
    errors = list(Draft202012Validator(
        recovery_schema(), registry=registry, format_checker=FormatChecker()
    ).iter_errors(document))
    if errors:
        raise ValueError("Recovery run schema: " + errors[0].message)
    _run(document)



#### export_failure_context

In [ ]:
%%inline_module src.ch04_rnn_lstm.export_r2 src/ch04_rnn_lstm/export_r2.py both
def export_failure_context(failure):
    """Accept only the final assertion of the pinned original function, not BPTT/parity bugs."""
    trace = failure.__traceback__
    while trace:
        if trace.tb_frame.f_code is frozen.train_run.__code__:
            context = trace.tb_frame.f_locals
            required = {"checkpoint", "metadata", "loaded", "cpu", "native_demo", "memory", "timing",
                        "output", "targets", "values", "history", "train_seconds", "best_epoch"}
            if required.issubset(context) and context.get("method") in IMPLEMENTATIONS:
                return context.copy()
        trace = trace.tb_next
    return None



#### finalize

In [ ]:
%%inline_module src.ch04_rnn_lstm.export_r2 src/ch04_rnn_lstm/export_r2.py both
def finalize(context, *, lost_telemetry=False):
    c = context
    entry, config, data = c["entry"], c["config"], c["data"]
    run, method = entry["run_id"], entry["method"]
    folder = BASE / "runs" / run
    if (folder / "selection.json").exists():
        raise ValueError("Never overwrite an existing completed selection")
    checked(c["checkpoint"])
    checked(c["metadata"])
    progress = frozen.load_progress(run, entry["config"]["sha256"])
    best_epoch = completed_progress(progress, config["training"], c["weights"])
    if c["history"] != progress["history"] or c["best_epoch"] != best_epoch:
        raise ValueError("Finalizer cannot reselect a checkpoint or rewrite training history")
    x, _ = data.batch(data.train_ids[:4])
    loaded = ArtifactStore(MODELS / run).load_checkpoint(
        "best", provenance(entry, config), expected_manifest_sha256=c["metadata"]["sha256"]
    )
    if any(not np.array_equal(loaded[k], c["weights"][k]) for k in loaded):
        raise ValueError("Checkpoint loading differs from selected weights")
    cpu = NumpyNetwork(config["plan"], loaded).forward(x)
    comparisons = compare_outputs(cpu, native_cpu(config["plan"], loaded, method, x), c["native_demo"])
    demo = {
        "status": "PASS", "input_split": "train", "ids": data.train_ids[:4],
        "canonical_x_sha256": array_sha256(x), "raw_output": cpu.tolist(),
        "max_absolute_error": comparisons["cpu_reference"]["max_absolute_error"],
        "checkpoint": c["checkpoint"], "serving_backend": "numpy_cpu_portable",
        "training_implementation": method, "reference_policy": "r2_native_cpu",
        "comparisons": comparisons, "revision": ref(REVISION),
    }
    write_once(folder / "cpu_demo.json", demo)
    write_once(folder / "history.json", c["history"])
    note = {
        "revision": ref(REVISION), "weights_updated_during_finalization": False,
        "original_gpu_cpu_gate": comparisons["original_gpu_cpu_gate"],
        "original_training_telemetry_available": not lost_telemetry,
        "validation_predictions": "recomputed from same frozen best checkpoint after process loss"
        if lost_telemetry else "retained exact native GPU outputs from original failed frame",
        "train_snapshot_elapsed_seconds": progress["elapsed_seconds"],
        "elapsed_scope": "Last epoch snapshot, not full train wall; never used as replacement training timer",
        "recovery_process_memory": c["memory"] if lost_telemetry else None,
        "recovery_started_at": c["started"] if lost_telemetry else None,
        "test_executed": False,
    }
    write_once(folder / "export_recovery_r2.json", note)
    details = {
        "metrics": c["values"], "memory": None if lost_telemetry else c["memory"],
        "best_epoch": best_epoch, "preprocessing_seconds": None if lost_telemetry else c["prep_seconds"],
        "train_count": len(data.train_ids), "validation_count": len(data.validation_ids),
        "unit": config["unit"], "fold": entry["fold"], "platform": platform.platform(),
        "numpy": np.__version__, "test_evaluation_executed": False,
        "export_recovery": ref(folder / "export_recovery_r2.json"),
    }
    write_once(folder / "validation_details.json", details)
    predictions = frozen.save_predictions(run, data, c["targets"], c["output"])
    memory, timing = c["memory"], c["timing"]
    efficiency = {
        "train_seconds": c["train_seconds"], "inference_p50_ms": timing["inference_p50_ms"],
        "inference_p95_ms": timing["inference_p95_ms"], "warmup_batches": timing["warmup"],
        "timed_repeats": timing["repeats"], "peak_ram_bytes": memory["process_peak_ram_bytes"],
        "peak_vram_bytes": memory["peak_vram_bytes"], "checkpoint_bytes": checked(c["checkpoint"]).stat().st_size,
    }
    # For the orphan, these initial numbers measure recovery, never original training.
    # Only the final null-valued recovery schema document is published.
    document = frozen.record(entry, config, data, c["values"], efficiency, c["started"],
                             [c["checkpoint"], c["metadata"], predictions, ref(folder / "export_recovery_r2.json")])
    if lost_telemetry:
        if run != ORPHAN:
            raise ValueError("Nullable telemetry exception applies only to the declared dead process")
        document["schema_version"] = RECOVERY_VERSION
        for key in MISSING_TELEMETRY:
            document["efficiency"][key] = None
        document["warnings"] += [LOST_TELEMETRY_WARNING,
                                 "started_at identifies recovery, not the lost original process start"]
    document["warnings"].append("r2 strict native CPU export; original GPU/CPU result retained separately")
    validate_run(document)
    metric_path = METRICS / "validation" / f"{run}.json"
    write_once(metric_path, document)
    write_once(folder / "selection.json", {
        "run_id": run, "status": "succeeded", "lifecycle": "trained_validation_selected",
        "selection_split": "validation", "inference_ready": True, "best_epoch": best_epoch,
        "checkpoint": c["checkpoint"], "checkpoint_metadata": c["metadata"],
        "validation_metrics": ref(metric_path), "validation_predictions": predictions,
        "history": ref(folder / "history.json"), "details": ref(folder / "validation_details.json"),
        "cpu_demo": ref(folder / "cpu_demo.json"), "config": entry["config"],
        "model_contract": entry["model_contract"], "comparison_group_sha256": entry["comparison_group_sha256"],
        "export_revision": ref(REVISION), "export_recovery": ref(folder / "export_recovery_r2.json"),
        "test_evaluation_executed": False,
    })
    return {"run_id": run, "status": "PASS", "recovery": "r2", "epochs": len(c["history"]),
            "original_gpu_cpu_gate": comparisons["original_gpu_cpu_gate"]["status"],
            "original_training_telemetry_available": not lost_telemetry}



#### recover_orphan

In [ ]:
%%inline_module src.ch04_rnn_lstm.export_r2 src/ch04_rnn_lstm/export_r2.py both
def recover_orphan(entry):
    run = entry["run_id"]
    if run != ORPHAN:
        raise ValueError("Unexpected orphan checkpoint: inspect before recovery")
    config = read_json(checked(entry["config"]))
    progress = frozen.load_progress(run, entry["config"]["sha256"])
    metadata, checkpoint = ref(MODELS / run / "best.json"), ref(MODELS / run / "best.npz")
    weights = ArtifactStore(MODELS / run).load_checkpoint(
        "best", provenance(entry, config), expected_manifest_sha256=metadata["sha256"]
    )
    best_epoch = completed_progress(progress, config["training"], weights)
    started = datetime.now(UTC).isoformat()
    configure(entry["method"], entry["seed"])
    data = WindowData(entry["dataset_id"], entry["fold"])
    if data.contract() != read_json(checked(config["data_contract"])):
        raise ValueError("Recovery data contract changed")
    monitor = DeviceMonitor("torch", "gpu")
    monitor.start()
    tick = time.perf_counter()
    trainer = frozen.make_trainer(entry["method"], config["plan"], weights)
    x, _ = data.batch(data.train_ids[:4])
    native_demo = trainer.predict(x)
    output = data.adapter.inverse_target(predict_windows(trainer, data, data.validation_ids))
    timing = inference_timing(lambda: trainer.predict(x[:1]), monitor.synchronize)
    memory = monitor.finish()
    targets = data.targets(data.validation_ids)
    values = regression_metrics(targets, output, data.horizons, data.series(data.validation_ids))
    return finalize({"entry": entry, "config": config, "data": data, "weights": weights,
                     "history": progress["history"], "best_epoch": best_epoch,
                     "train_seconds": time.perf_counter() - tick, "memory": memory,
                     "timing": timing, "prep_seconds": None, "output": output,
                     "targets": targets, "values": values, "native_demo": native_demo,
                     "checkpoint": checkpoint, "metadata": metadata, "started": started},
                    lost_telemetry=True)



#### run_revision

In [ ]:
%%inline_module src.ch04_rnn_lstm.export_r2 src/ch04_rnn_lstm/export_r2.py both
def run_revision(run):
    entry = next(e for e in check_revision()["runs"] if e["run_id"] == run)
    if (BASE / "runs" / run / "selection.json").exists():
        return frozen.train_run(run)
    if (MODELS / run / "best.npz").exists():
        return recover_orphan(entry)
    try:
        return frozen.train_run(run)
    except AssertionError as failure:
        context = export_failure_context(failure)
        if context is None:
            raise
        import traceback

        write_once(BASE / "attempts" / f"r2_{run}_gpu_cpu_failure.json", {
            "run_id": run, "traceback": traceback.format_exc(), "test_executed": False,
            "stage": "original_frozen_gpu_cpu_export_gate", "revision": ref(REVISION),
        })
        return finalize(context)



### `src.ch04_rnn_lstm.phase29.diagnostics`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch04_rnn_lstm.phase29.diagnostics src/ch04_rnn_lstm/phase29/diagnostics.py both
"""Fixed TRAIN gates/carry diagnostics; not causal attribution or TEST tuning."""

import numpy as np

from src.ch04_rnn_lstm.diagnostics import gradient_diagnostics

from src.common.numpy_engine import NumpyNetwork



#### gate_diagnostics

In [ ]:
%%inline_module src.ch04_rnn_lstm.phase29.diagnostics src/ch04_rnn_lstm/phase29/diagnostics.py both
def gate_diagnostics(plan, weights, x, y):
    if plan.get("family") != "lstm":
        raise ValueError("LSTM plan required for gate diagnostics")
    network = NumpyNetwork(plan, weights)
    network.forward(x)
    layers = []
    for step, _, _, states in network.cache:
        if step["kind"] != "lstm":
            continue
        gates = {}
        for name, pos in (("input", 3), ("forget", 4), ("candidate", 5), ("output", 6)):
            values = np.stack([s[pos] for s in states], axis=1).astype("float64")
            gates[name] = {
                "min": float(values.min()),
                "max": float(values.max()),
                "mean": float(values.mean()),
                "mean_by_time": values.mean(axis=(0, 2)).tolist(),
                "saturation_fraction": float(np.mean(np.abs(values) > 0.95))
                if name == "candidate"
                else float(np.mean((values < 0.05) | (values > 0.95))),
            }
        carry_errors = [np.max(np.abs(s[7] - (s[4] * s[2] + s[3] * s[5]))) for s in states]
        hidden_errors = [
            np.max(np.abs(states[t + 1][1] - s[6] * np.tanh(s[7]))) for t, s in enumerate(states[:-1])
        ]
        layers.append(
            {
                "layer_id": step["id"],
                "gates": gates,
                "carry_equation_max_absolute_error": float(max(carry_errors)),
                "hidden_equation_max_absolute_error": float(max(hidden_errors, default=0)),
                "initial_hidden_max_absolute": float(np.max(np.abs(states[0][1]))),
                "initial_carry_max_absolute": float(np.max(np.abs(states[0][2]))),
            }
        )
    return {
        **gradient_diagnostics(plan, weights, x, y),
        "layers": layers,
        "state_reset_per_window": True,
        "scope": "First fixed TRAIN windows only; sigmoid gates in [0,1], tanh candidate in [-1,1]. Saturation descriptive at .05/.95; no causal or long-memory superiority claim.",
    }



### `src.ch04_rnn_lstm.phase29.protocol`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch04_rnn_lstm.phase29.protocol src/ch04_rnn_lstm/phase29/protocol.py both
"""Phase29 validation-only declaration; Phase29 is the only joint TEST owner."""

from pathlib import Path

from src.ch04_rnn_lstm.protocol import SOURCE_FILES as PREVIOUS_FILES

from src.common.artifact_core import code_fingerprint, sha256_file

from src.common.contracts import PROTOCOL, ROOT, read_json

BASE = ROOT / "artifacts/manifests/phase29"

MODELS = ROOT / "artifacts/models/v3/ch04_lstm"

METRICS = ROOT / "artifacts/metrics/phase29"

PREDICTIONS = ROOT / "artifacts/predictions/phase29"

DATASETS = ("ch4_gold_pink_sheet", "ch4_bis_house_prices", "ch4_household_power")

SEEDS = (17, 42, 73)

IMPLEMENTATIONS = ("scratch", "keras", "pytorch")

FOLDS = (None, 1, 2, 3)

SOURCE_FILES = sorted(
    set(
        PREVIOUS_FILES
        + [
            "src/ch04_rnn_lstm/export_r2.py",
            *[
                f"src/ch04_rnn_lstm/phase29/{name}.py"
                for name in ("__init__", "protocol", "training", "diagnostics", "experiment", "joint_gate")
            ],
            *[
                f"scripts/{name}.py"
                for name in ("prepare_phase29", "phase29_controller", "phase29_worker", "phase29_preflight")
            ],
            "experiments/configs/v3/phase29_coverage.ini",
            "tests/lstm29/test_lstm_core.py",
            "tests/lstm29/test_experiment.py",
            "tests/lstm29/test_protocol.py",
            "tests/lstm29/test_joint_gate.py",
            "docs/phase29_operations.md",
        ]
    )
)



#### ref

In [ ]:
%%inline_module src.ch04_rnn_lstm.phase29.protocol src/ch04_rnn_lstm/phase29/protocol.py both
def ref(path):
    return {"path": Path(path).relative_to(ROOT).as_posix(), "sha256": sha256_file(path)}



#### label

In [ ]:
%%inline_module src.ch04_rnn_lstm.phase29.protocol src/ch04_rnn_lstm/phase29/protocol.py both
def label(fold):
    return "main" if fold is None else f"fold{fold}"



#### run_id

In [ ]:
%%inline_module src.ch04_rnn_lstm.phase29.protocol src/ch04_rnn_lstm/phase29/protocol.py both
def run_id(dataset, method, seed, fold):
    return f"{dataset}_{'lstm_' if method in IMPLEMENTATIONS else ''}{method}_s{seed}_{label(fold)}"



#### code_info

In [ ]:
%%inline_module src.ch04_rnn_lstm.phase29.protocol src/ch04_rnn_lstm/phase29/protocol.py both
def code_info():
    import subprocess

    head = subprocess.run(
        ["git", "rev-parse", "HEAD"], cwd=ROOT, check=True, capture_output=True, text=True
    ).stdout.strip()
    return {"git_commit": head, "dirty_tree_sha256": code_fingerprint(ROOT, SOURCE_FILES)}



#### checked

In [ ]:
%%inline_module src.ch04_rnn_lstm.phase29.protocol src/ch04_rnn_lstm/phase29/protocol.py both
def checked(record):
    path = (ROOT / record["path"]).resolve()
    if not path.is_relative_to(ROOT.resolve()) or ref(path) != record:
        raise ValueError("Approved artifact is missing, changed or escaping workspace")
    return path



#### check_sources

In [ ]:
%%inline_module src.ch04_rnn_lstm.phase29.protocol src/ch04_rnn_lstm/phase29/protocol.py both
def check_sources():
    release = read_json(BASE / "release.json")
    if (
        release["code"] != code_info()
        or release["protocol"] != ref(PROTOCOL)
        or release["environment"] != ref(ROOT / "uv.lock")
    ):
        raise ValueError("Declared recurrent source/protocol/environment changed; revision required")
    for record in release["source_files"]:
        checked(record)
    for entry in release["runs"]:
        for key in ("config", "model_contract"):
            checked(entry[key])
    return release



### `src.ch04_rnn_lstm.phase29.training`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch04_rnn_lstm.phase29.training src/ch04_rnn_lstm/phase29/training.py both
"""LSTM adapters reuse frozen canonical recurrent Adam and window training.

Keras uses an ordinary tf.function (not XLA): native LSTM may dispatch cuDNN.
This execution policy is probed and declared before training, not a backend swap.
"""

from src.ch04_rnn_lstm.training import RnnKerasTrainer, RnnTorchTrainer



#### LstmTorchTrainer

In [ ]:
%%inline_module src.ch04_rnn_lstm.phase29.training src/ch04_rnn_lstm/phase29/training.py both
class LstmTorchTrainer(RnnTorchTrainer):
    def __init__(self, plan, weights, device="gpu"):
        if plan.get("family") != "lstm":
            raise ValueError("LSTM plan required")
        super().__init__(plan, weights, device)



#### LstmKerasTrainer

In [ ]:
%%inline_module src.ch04_rnn_lstm.phase29.training src/ch04_rnn_lstm/phase29/training.py both
class LstmKerasTrainer(RnnKerasTrainer):
    def __init__(self, plan, weights, device="gpu"):
        if plan.get("family") != "lstm":
            raise ValueError("LSTM plan required")
        super().__init__(plan, weights, device, jit_compile=False)



### `src.ch04_rnn_lstm.phase29.experiment`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.ch04_rnn_lstm.phase29.experiment src/ch04_rnn_lstm/phase29/experiment.py both
"""Validation-only measured LSTM runs, safe optimizer resume and CPU checkpoints."""

import gzip

import json

import os

import platform

import shutil

import time

import zipfile

from datetime import UTC, datetime

from pathlib import Path

import numpy as np

from src.ch02_ml.experiment import configure, provenance

from src.ch02_ml.protocol import array_sha256

from src.ch02_ml.test_gate import atomic_json

from src.ch02_ml.training import ScratchTrainer

from src.ch04_rnn_lstm.export_r2 import compare_outputs, native_cpu

from src.ch04_rnn_lstm.phase29.diagnostics import gate_diagnostics

from src.ch04_rnn_lstm.phase29.protocol import (
    BASE,
    IMPLEMENTATIONS,
    METRICS,
    MODELS,
    PREDICTIONS,
    check_sources,
    checked,
    ref,
)

from src.ch04_rnn_lstm.phase29.training import LstmKerasTrainer, LstmTorchTrainer

from src.ch04_rnn_lstm.training import fit_windows, predict_windows

from src.ch04_rnn_lstm.window_data import WindowData

from src.common.artifact_core import ArtifactStore, write_once

from src.common.contracts import ROOT, canonical_sha256, read_json, validate_document

from src.common.device_core import DeviceMonitor

from src.common.metric_core import regression_metrics

from src.common.model_core import canonical_weights

from src.common.numpy_engine import NumpyNetwork

from src.common.parity import gradient_parity

from src.common.training_core import inference_timing



#### make_trainer

In [ ]:
%%inline_module src.ch04_rnn_lstm.phase29.experiment src/ch04_rnn_lstm/phase29/experiment.py both
def make_trainer(method, plan, weights):
    if method == "scratch":
        return ScratchTrainer(plan, weights)
    if method == "pytorch":
        return LstmTorchTrainer(plan, weights, "gpu")
    if method == "keras":
        return LstmKerasTrainer(plan, weights, "gpu")
    raise ValueError("Unknown LSTM implementation")



#### save_progress

In [ ]:
%%inline_module src.ch04_rnn_lstm.phase29.experiment src/ch04_rnn_lstm/phase29/experiment.py both
def save_progress(run, state, config_hash):
    folder = MODELS / run / "resume"
    folder.mkdir(parents=True, exist_ok=True)
    epoch = len(state["history"])
    path = folder / f"epoch_{epoch:03d}.npz"
    arrays = {
        f"{field}/{key}": value
        for field in ("weights", "m", "v")
        for key, value in state["trainer"][field].items()
    }
    arrays.update({f"best/{key}": value for key, value in state["best_weights"].items()})
    if any(value.dtype.kind not in "fiu" or not np.isfinite(value).all() for value in arrays.values()):
        raise ValueError("Unsafe optimizer state")
    with path.open("xb") as stream:
        np.savez_compressed(stream, **arrays)
    metadata = {key: value for key, value in state.items() if key not in {"trainer", "best_weights"}}
    metadata.update(
        {"iterations": state["trainer"]["iterations"], "config_sha256": config_hash, "arrays": ref(path)}
    )
    meta_path = folder / f"epoch_{epoch:03d}.json"
    write_once(meta_path, metadata)
    atomic_json(MODELS / run / "progress.json", ref(meta_path))



#### load_progress

In [ ]:
%%inline_module src.ch04_rnn_lstm.phase29.experiment src/ch04_rnn_lstm/phase29/experiment.py both
def load_progress(run, config_hash):
    pointer = MODELS / run / "progress.json"
    if not pointer.exists():
        return None
    metadata = read_json(checked(read_json(pointer)))
    if metadata["config_sha256"] != config_hash:
        raise ValueError("Resume config changed")
    path = checked(metadata["arrays"])
    if path.stat().st_size > 64 * 1024**2:
        raise ValueError("Resume exceeds 64 MiB budget")
    with zipfile.ZipFile(path) as archive:
        if sum(item.file_size for item in archive.infolist()) > 64 * 1024**2:
            raise ValueError("Expanded optimizer archive exceeds 64 MiB budget")
    with np.load(path, allow_pickle=False) as archive:
        arrays = {key: archive[key] for key in archive.files}
    if sum(value.nbytes for value in arrays.values()) > 64 * 1024**2 or any(
        value.dtype.kind not in "fiu" or not np.isfinite(value).all() for value in arrays.values()
    ):
        raise ValueError("Unsafe expanded optimizer state")
    return {
        "history": metadata["history"],
        "early_stopping": metadata["early_stopping"],
        "elapsed_seconds": metadata["elapsed_seconds"],
        "trainer": {
            **{
                field: {
                    key.removeprefix(field + "/"): value
                    for key, value in arrays.items()
                    if key.startswith(field + "/")
                }
                for field in ("weights", "m", "v")
            },
            "iterations": metadata["iterations"],
        },
        "best_weights": {
            key.removeprefix("best/"): value for key, value in arrays.items() if key.startswith("best/")
        },
    }



#### metric_rows

In [ ]:
%%inline_module src.ch04_rnn_lstm.phase29.experiment src/ch04_rnn_lstm/phase29/experiment.py both
def metric_rows(values, unit):
    rows = []
    for horizon, group in [
        (None, values),
        *[(int(h), group) for h, group in values.get("by_horizon", {}).items()],
    ]:
        for name in ("rmse", "mae", "smape", "macro_series_rmse"):
            if name in group:
                rows.append(
                    {
                        "name": name,
                        **group[name],
                        "unit": "percent" if name == "smape" else unit,
                        "direction": "min",
                        "horizon": horizon,
                    }
                )
    return rows



#### record

In [ ]:
%%inline_module src.ch04_rnn_lstm.phase29.experiment src/ch04_rnn_lstm/phase29/experiment.py both
def record(entry, config, data, values, efficiency, started, artifacts):
    document = {
        "schema_version": "3.0.0",
        "evidence_kind": "measured",
        "run_id": entry["run_id"],
        "dataset_id": entry["dataset_id"],
        "split_id": f"{entry['dataset_id']}_phase29_{'main' if entry['fold'] is None else 'fold' + str(entry['fold'])}",
        "split_sha256": config["provenance"]["split_sha256"],
        "config_sha256": entry["config"]["sha256"],
        "code": config["code"],
        "environment_sha256": config["environment_sha256"],
        "model_contract": entry["model_contract"],
        "implementation": entry["method"],
        "backend": {"scratch": "numpy", "keras": "tensorflow", "pytorch": "torch"}.get(
            entry["method"], "numpy"
        ),
        "family": "lstm",
        "track": "matched_subset",
        "seed": entry["seed"],
        "subset_sha256": config["provenance"]["subset_sha256"],
        "status": "succeeded",
        "split": "validation",
        "test_seal_id": None,
        "device": "cuda:0" if entry["method"] in {"keras", "pytorch"} else "cpu",
        "dtype": "float32",
        "batch_size": config["training"]["batch_size"],
        "started_at": started,
        "finished_at": datetime.now(UTC).isoformat(),
        "metrics": metric_rows(values, config["unit"]),
        "efficiency": efficiency,
        "artifacts": artifacts,
        "warnings": [
            "Validation-selected; joint RNN/LSTM TEST belongs to a later, separately locked Phase29 batch",
            "Latest revised snapshots, retrospective observation-origin forecast; no real-time publication/vintage claim or investment advice",
            "Each rolling fold uses its frozen scaler; windows reset recurrent state",
            "Train wall includes validation/progress I/O; RAM is process lifetime, VRAM allocator only; CPU/GPU and graph/eager differ",
        ],
        "dataset_sha256": config["provenance"]["dataset_sha256"],
        "comparison_group_sha256": entry["comparison_group_sha256"],
        "protocol_sha256": config["protocol_sha256"],
        "preprocessing": data.view.contract()["preprocessing"],
    }
    if entry["method"] in IMPLEMENTATIONS:
        validate_document("run", document, publish=True)
    else:
        document.update(
            {
                "schema_version": "phase29.causal_baseline_run.1",
                "family": "classical",
                "implementation": "numpy",
                "method": entry["method"],
            }
        )
    return document



#### save_predictions

In [ ]:
%%inline_module src.ch04_rnn_lstm.phase29.experiment src/ch04_rnn_lstm/phase29/experiment.py both
def save_predictions(run, data, targets, predictions):
    path = PREDICTIONS / "validation" / f"{run}.jsonl.gz"
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("xb") as stream, gzip.GzipFile(filename="", fileobj=stream, mode="wb", mtime=0) as packed:
        for key, target, pred in zip(data.validation_ids, targets, predictions, strict=True):
            row = {
                "id": key,
                "series": data.rows[key]["series"],
                "target_times": data.rows[key]["target_times"],
                "target": target.tolist(),
                "prediction": pred.tolist(),
            }
            packed.write((json.dumps(row, sort_keys=True, allow_nan=False) + "\n").encode())
    return ref(path)



#### train_run

In [ ]:
%%inline_module src.ch04_rnn_lstm.phase29.experiment src/ch04_rnn_lstm/phase29/experiment.py both
def train_run(run):
    release = check_sources()
    entry = next(row for row in release["runs"] if row["run_id"] == run)
    folder = BASE / "runs" / run
    if (folder / "selection.json").exists():
        selected = read_json(folder / "selection.json")
        for value in selected.values():
            if isinstance(value, dict) and set(value) == {"path", "sha256"}:
                checked(value)
        return {"run_id": run, "status": "reused"}
    if shutil.disk_usage(ROOT).free < 5 * 1024**3:
        raise OSError("Host disk below 5 GiB reserve")
    claim = folder / "training_claim.json"
    if (
        claim.exists()
        and read_json(claim)["pid"] != os.getpid()
        and Path(f"/proc/{read_json(claim)['pid']}").exists()
    ):
        raise ValueError("Live worker owns recurrent run")
    atomic_json(claim, {"pid": os.getpid(), "run_id": run})
    config = read_json(checked(entry["config"]))
    method, started = entry["method"], datetime.now(UTC).isoformat()
    backend = configure(method, entry["seed"])
    tick = time.perf_counter()
    data = WindowData(entry["dataset_id"], entry["fold"])
    if data.contract() != read_json(checked(config["data_contract"])):
        raise ValueError("Declared recurrent data contract changed")
    x_demo, y_demo = data.batch(data.train_ids[:4])
    prep_seconds = time.perf_counter() - tick
    monitor = DeviceMonitor(backend, "gpu" if method in {"keras", "pytorch"} else "cpu")
    monitor.start()
    if method in IMPLEMENTATIONS:
        weights = canonical_weights(config["plan"], entry["seed"])
        if (
            canonical_sha256({k: v.tolist() for k, v in weights.items()})
            != config["canonical_initial_weights_sha256"]
        ):
            raise ValueError("Initialization changed")
        trainer = make_trainer(method, config["plan"], weights)
        reference = NumpyNetwork(config["plan"], weights)
        np.testing.assert_allclose(trainer.predict(x_demo), reference.forward(x_demo), atol=1e-5, rtol=1e-4)
        expected = reference.loss_gradients(x_demo, y_demo, "mse")[1]
        actual = expected if method == "scratch" else trainer.model.loss_gradients(x_demo, y_demo, "mse")[1]
        write_once(
            folder / "initial_parity.json",
            {
                "forward": "PASS",
                "gradient": gradient_parity(expected, actual),
                "input_split": "train",
                "ids": data.train_ids[:4],
            },
        )
        result = fit_windows(
            trainer,
            data,
            seed=entry["seed"],
            budget=config["training"],
            resume=load_progress(run, entry["config"]["sha256"]),
            progress=lambda state: save_progress(run, state, entry["config"]["sha256"]),
        )
        weights, history, best_epoch, train_seconds = (
            result["weights"],
            result["history"],
            result["best_epoch"],
            result["train_seconds"],
        )
        output = data.adapter.inverse_target(predict_windows(trainer, data, data.validation_ids))
        native_demo = trainer.predict(x_demo)
        predict_one = lambda: trainer.predict(x_demo[:1])
        write_once(
            folder / "gradient_diagnostics.json",
            {**gate_diagnostics(config["plan"], weights, x_demo, y_demo), "ids": data.train_ids[:4]},
        )
    else:
        tick = time.perf_counter()
        weights = {"baseline_kind": np.asarray([int(method == "seasonal_naive")], dtype="int64")}
        train_seconds = time.perf_counter() - tick
        output = data.baseline(data.validation_ids, method)
        native_demo = data.baseline(data.train_ids[:4], method)
        predict_one = lambda: data.baseline(data.train_ids[:1], method)
        history, best_epoch = [], None
    memory = monitor.finish()
    if method in {"keras", "pytorch"} and not 0 < memory["peak_vram_bytes"] < 8 * 1024**3:
        raise ValueError("Invalid GPU memory measurement")
    timing = inference_timing(predict_one, monitor.synchronize)
    targets = data.targets(data.validation_ids)
    values = regression_metrics(targets, output, data.horizons, data.series(data.validation_ids))
    cp = ArtifactStore(MODELS / run).save_checkpoint("best", weights, provenance(entry, config))
    checkpoint, metadata = ref(MODELS / run / "best.npz"), ref(MODELS / run / "best.json")
    loaded = ArtifactStore(MODELS / run).load_checkpoint(
        "best", provenance(entry, config), expected_manifest_sha256=metadata["sha256"]
    )
    cpu = (
        NumpyNetwork(config["plan"], loaded).forward(x_demo)
        if method in IMPLEMENTATIONS
        else data.baseline(data.train_ids[:4], method)
    )
    comparisons = compare_outputs(cpu, native_cpu(config["plan"], loaded, method, x_demo), native_demo)
    write_once(
        folder / "cpu_demo.json",
        {
            "status": "PASS",
            "input_split": "train",
            "ids": data.train_ids[:4],
            "canonical_x_sha256": array_sha256(x_demo),
            "raw_output": cpu.tolist(),
            "max_absolute_error": comparisons["cpu_reference"]["max_absolute_error"],
            "comparisons": comparisons,
            "export_reference_policy": "native_cpu; GPU comparison retained diagnostic, original tolerance unchanged",
            "checkpoint": checkpoint,
            "serving_backend": "numpy_cpu_portable",
            "training_implementation": method,
        },
    )
    write_once(folder / "history.json", history)
    details = {
        "metrics": values,
        "memory": memory,
        "best_epoch": best_epoch,
        "preprocessing_seconds": prep_seconds,
        "train_count": len(data.train_ids),
        "validation_count": len(data.validation_ids),
        "unit": config["unit"],
        "fold": entry["fold"],
        "platform": platform.platform(),
        "numpy": np.__version__,
        "test_evaluation_executed": False,
    }
    write_once(folder / "validation_details.json", details)
    predictions = save_predictions(run, data, targets, output)
    efficiency = {
        "train_seconds": train_seconds,
        "inference_p50_ms": timing["inference_p50_ms"],
        "inference_p95_ms": timing["inference_p95_ms"],
        "warmup_batches": timing["warmup"],
        "timed_repeats": timing["repeats"],
        "peak_ram_bytes": memory["process_peak_ram_bytes"],
        "peak_vram_bytes": memory["peak_vram_bytes"],
        "checkpoint_bytes": cp["size_bytes"],
    }
    metric_path = METRICS / "validation" / f"{run}.json"
    write_once(
        metric_path,
        record(entry, config, data, values, efficiency, started, [checkpoint, metadata, predictions]),
    )
    write_once(
        folder / "selection.json",
        {
            "run_id": run,
            "status": "succeeded",
            "lifecycle": "trained_validation_selected",
            "selection_split": "validation",
            "inference_ready": True,
            "best_epoch": best_epoch,
            "checkpoint": checkpoint,
            "checkpoint_metadata": metadata,
            "validation_metrics": ref(metric_path),
            "validation_predictions": predictions,
            "history": ref(folder / "history.json"),
            "details": ref(folder / "validation_details.json"),
            "cpu_demo": ref(folder / "cpu_demo.json"),
            "config": entry["config"],
            "model_contract": entry["model_contract"],
            "comparison_group_sha256": entry["comparison_group_sha256"],
            "test_evaluation_executed": False,
        },
    )
    return {
        "run_id": run,
        "status": "PASS",
        "epochs": len(history),
        "best_epoch": best_epoch,
        "rmse": values["rmse"]["value"],
        "train_seconds": train_seconds,
    }



### `src.common.backend_models`

Implementation đầy đủ ở các cell ngay dưới: đọc imports, từng hàm/lớp rồi mới đến phần thực hành. 


#### Thiết lập / hằng số

In [ ]:
%%inline_module src.common.backend_models src/common/backend_models.py both
"""Lazy independent PyTorch and Keras/TensorFlow adapters for canonical plans.

No torch import is needed in the Keras environment (or vice versa). The public
probe methods accept canonical NHWC/NTF NumPy arrays. DatasetView separately
provides native NCHW batches for future PyTorch training runners.
"""

from __future__ import annotations

import numpy as np

from src.common.model_core import parameter_shapes



#### TorchBackend

In [ ]:
%%inline_module src.common.backend_models src/common/backend_models.py both
class TorchBackend:
    def __init__(self, plan, weights, device="cpu"):
        import torch
        from torch import nn

        torch.backends.fp32_precision = "ieee"
        torch.backends.cuda.matmul.fp32_precision = "ieee"
        torch.backends.cudnn.fp32_precision = "ieee"
        torch.backends.cudnn.conv.fp32_precision = "ieee"
        torch.backends.cudnn.rnn.fp32_precision = "ieee"
        if device not in {"cpu", "gpu"}:
            raise ValueError("Unknown device")
        if device == "gpu" and not torch.cuda.is_available():
            raise RuntimeError("Requested GPU is unavailable; no silent fallback")
        self.torch = torch
        self.plan = plan
        self.device = "cuda:0" if device == "gpu" else "cpu"
        self.ops = nn.ModuleDict()
        for step in plan["steps"]:
            kind = step["kind"]
            width = step["input_shape"][-1]
            units = step["output_shape"][-1]
            if kind == "dense":
                layer = nn.Linear(width, units)
            elif kind == "conv2d":
                layer = nn.Conv2d(width, units, 3, padding=1)
            elif kind == "activation":
                layer = nn.ReLU()
            elif kind == "pooling":
                layer = nn.MaxPool2d(2) if step["pool"] == "max2" else nn.AdaptiveAvgPool2d(1)
            else:
                layer = (nn.LSTM if kind == "lstm" else nn.RNN)(width, units, batch_first=True)
                layer.bias_hh_l0.requires_grad_(False)
            self.ops[step["id"]] = layer
        self.ops.to(self.device)
        self.assign(weights)

    @property
    def parameter_count(self):
        return sum(v.numel() for v in self.ops.parameters() if v.requires_grad)

    def parameter_pairs(self):
        pairs = []
        for step in self.plan["steps"]:
            layer = self.ops[step["id"]]
            kind = step["kind"]
            for key in parameter_shapes(step):
                name = (
                    {"w": "weight_ih_l0", "u": "weight_hh_l0", "b": "bias_ih_l0"}[key]
                    if kind in {"rnn", "lstm"}
                    else "weight"
                    if key == "w"
                    else "bias"
                )
                pairs.append((f"{step['id']}/{key}", getattr(layer, name), kind, key))
        return pairs

    def assign(self, weights):
        with self.torch.no_grad():
            for name, param, kind, key in self.parameter_pairs():
                value = weights[name]
                if key in {"w", "u"}:
                    value = value.transpose(3, 2, 0, 1) if kind == "conv2d" else value.T
                param.copy_(self.torch.as_tensor(np.ascontiguousarray(value), device=self.device))
            for step in self.plan["steps"]:
                if step["kind"] in {"rnn", "lstm"}:
                    self.ops[step["id"]].bias_hh_l0.zero_()

    def forward(self, x, trace=False):
        torch = self.torch
        value = (
            torch.as_tensor(np.ascontiguousarray(x), dtype=torch.float32, device=self.device)
            if isinstance(x, np.ndarray)
            else x
        )
        is_image = self.plan["family"] == "cnn"
        if is_image:
            value = value.permute(0, 3, 1, 2).contiguous()
        traces = []
        for step in self.plan["steps"]:
            before = list(value.shape)
            kind = step["kind"]
            value = self.ops[step["id"]](value)
            if kind in {"rnn", "lstm"}:
                value = value[0]
                if not step["return_sequences"]:
                    value = value[:, -1]
            if kind == "pooling" and step["pool"] == "global_average":
                value = value.flatten(1)
            after = list(value.shape)
            if is_image and len(before) == 4:
                before = [before[0], before[2], before[3], before[1]]
            if is_image and len(after) == 4:
                after = [after[0], after[2], after[3], after[1]]
            traces.append({"id": step["id"], "input_shape": before, "output_shape": after})
        return (value, traces) if trace else value

    def predict(self, x):
        with self.torch.no_grad():
            return self.forward(x).cpu().numpy()

    def trace(self, x):
        with self.torch.no_grad():
            return self.forward(x, True)[1]

    def loss_gradients(self, x, y, task):
        self.ops.zero_grad(set_to_none=True)
        logits = self.forward(x)
        target = self.torch.as_tensor(y, device=self.device)
        if task == "mse":
            loss = self.torch.mean((logits - target) ** 2)
        elif task == "binary":
            loss = self.torch.nn.functional.binary_cross_entropy_with_logits(logits, target)
        elif task == "multiclass":
            loss = self.torch.nn.functional.cross_entropy(logits, target.long())
        else:
            raise ValueError("Unknown loss")
        loss.backward()
        gradients = {}
        for name, param, kind, key in self.parameter_pairs():
            value = param.grad.detach().cpu().numpy()
            if key in {"w", "u"}:
                value = value.transpose(2, 3, 1, 0) if kind == "conv2d" else value.T
            gradients[name] = np.ascontiguousarray(value)
        return float(loss.detach().cpu()), gradients



#### KerasBackend

In [ ]:
%%inline_module src.common.backend_models src/common/backend_models.py both
class KerasBackend:
    def __init__(self, plan, weights, device="cpu"):
        import keras
        import tensorflow as tf

        if keras.backend.backend() != "tensorflow":
            raise RuntimeError("Keras must use the independently locked TensorFlow backend")
        if device not in {"cpu", "gpu"}:
            raise ValueError("Unknown device")
        if device == "gpu" and not tf.config.list_physical_devices("GPU"):
            raise RuntimeError("Requested GPU unavailable; no silent fallback")
        self.tf = tf
        tf.config.experimental.enable_tensor_float_32_execution(False)
        self.keras = keras
        self.plan = plan
        self.device = "/GPU:0" if device == "gpu" else "/CPU:0"
        self.layers = {}
        with tf.device(self.device):
            inputs = keras.Input(shape=plan["input_shape"], dtype="float32")
            value = inputs
            for step in plan["steps"]:
                kind = step["kind"]
                units = step["output_shape"][-1]
                name = step["id"]
                if kind == "dense":
                    layer = keras.layers.Dense(units, name=name)
                elif kind == "conv2d":
                    layer = keras.layers.Conv2D(units, 3, padding="same", name=name)
                elif kind == "activation":
                    layer = keras.layers.ReLU(name=name)
                elif kind == "pooling":
                    layer = (
                        keras.layers.MaxPooling2D(2, name=name)
                        if step["pool"] == "max2"
                        else keras.layers.GlobalAveragePooling2D(name=name)
                    )
                elif kind == "rnn":
                    layer = keras.layers.SimpleRNN(
                        units, return_sequences=step["return_sequences"], name=name
                    )
                else:
                    layer = keras.layers.LSTM(
                        units, return_sequences=step["return_sequences"], unit_forget_bias=False, name=name
                    )
                value = layer(value)
                self.layers[name] = layer
            self.model = keras.Model(inputs, value)
            self.trace_model = keras.Model(inputs, [layer.output for layer in self.layers.values()])
            self.assign(weights)

    @property
    def parameter_count(self):
        return sum(int(np.prod(v.shape)) for v in self.model.trainable_weights)

    def parameter_pairs(self):
        return [
            (f"{step['id']}/{key}", variable)
            for step in self.plan["steps"]
            for key, variable in zip(
                parameter_shapes(step), self.layers[step["id"]].trainable_weights, strict=True
            )
        ]

    def assign(self, weights):
        with self.tf.device(self.device):
            for name, variable in self.parameter_pairs():
                variable.assign(weights[name])

    def predict(self, x):
        with self.tf.device(self.device):
            return self.model(x, training=False).numpy()

    def trace(self, x):
        with self.tf.device(self.device):
            values = self.trace_model(x, training=False)
        before = list(x.shape)
        rows = []
        for step, value in zip(self.plan["steps"], values, strict=True):
            rows.append({"id": step["id"], "input_shape": before, "output_shape": list(value.shape)})
            before = list(value.shape)
        return rows

    def loss_gradients(self, x, y, task):
        tf = self.tf
        with tf.device(self.device), tf.GradientTape() as tape:
            logits = self.model(x, training=True)
            target = tf.convert_to_tensor(y)
            if task == "mse":
                loss = tf.reduce_mean(tf.square(logits - tf.cast(target, logits.dtype)))
            elif task == "binary":
                loss = tf.reduce_mean(
                    tf.nn.sigmoid_cross_entropy_with_logits(
                        labels=tf.cast(target, logits.dtype), logits=logits
                    )
                )
            elif task == "multiclass":
                loss = tf.reduce_mean(
                    tf.nn.sparse_softmax_cross_entropy_with_logits(
                        labels=tf.cast(target, tf.int64), logits=logits
                    )
                )
            else:
                raise ValueError("Unknown loss")
        with tf.device(self.device):
            gradients = tape.gradient(loss, self.model.trainable_weights)
        return float(loss.numpy()), {
            name: gradient.numpy()
            for (name, _), gradient in zip(self.parameter_pairs(), gradients, strict=True)
        }



## Chạy thử trực tiếp code vừa khai báo

Ví dụ tensor **synthetic** nhỏ, weights mới khởi tạo: kiểm forward của NumPy và framework trong kernel hiện tại. Không optimizer step, không train, không dùng dataset TEST; các số dưới đây không phải kết quả nghiên cứu.

In [ ]:
import numpy as np
from src.common.model_core import build_plan, canonical_weights
from src.common.numpy_engine import NumpyNetwork
from src.common.backend_models import TorchBackend, KerasBackend
for _family in ['rnn', 'lstm']:
    _shape = {"mlp": (3,), "cnn": (8,8,3), "rnn": (4,3), "lstm": (4,3)}[_family]
    _plan = build_plan(_family, _shape, 2, tiny=True)
    _weights = canonical_weights(_plan, 42)
    _x = np.zeros((2, *_shape), dtype=np.float32)
    _numpy = NumpyNetwork(_plan, _weights).forward(_x)
    _native = (KerasBackend if ACTIVE_FRAMEWORK == "keras" else TorchBackend)(_plan, _weights, "cpu").predict(_x)
    assert np.isfinite(_native).all() and np.allclose(_numpy, _native, atol=1e-4, rtol=1e-4)
    print(_family, "synthetic forward", _native.shape, "max diff", float(np.max(np.abs(_numpy-_native))))


## Thực hành: ch4_bis_house_prices

Code và output lịch sử được giữ để đối chiếu. `RUN_EXPERIMENTS=False` nên không tự tạo split/train/TEST hoặc ghi lại kết quả đã khóa.

# RNN–LSTM / ch4_bis_house_prices

Hai họ ×ba checkpoint train độc lập. Serving thực tế NumPy CPU; không train hoặc mở TEST. Mẫu TRAIN, seed42 cố định; không chọn model bằng TEST. CLI/notebook chưa phải HTTP/React/Render live, không khuyến nghị đầu tư.

In [ ]:
if RUN_EXPERIMENTS:
    from pathlib import Path
    import sys
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'TIEU_LUAN_AI_V3_EXECUTION_PLAN.md').exists())
    if str(root) not in sys.path: sys.path.insert(0, str(root))
    from assignment_runtime.paper_demo import sample_request, infer
    from src.common.contracts import read_json


In [ ]:
if RUN_EXPERIMENTS:
    dataset = 'ch4_bis_house_prices'
    jobs = {}
    for family in ('rnn', 'lstm'):
        request, sample_id = sample_request(dataset, family)
        jobs[family] = infer(request)
        print('TRAIN sample:', sample_id, family)
        for row in jobs[family]['results']:
            print(row['implementation'], row['actual_backend'], row['actual_device'], row['prediction'])
        assert jobs[family]['comparison_valid']
    assert jobs['rnn']['input_sha256'] == jobs['lstm']['input_sha256']

In [ ]:
if RUN_EXPERIMENTS:
    for family, phase in (('rnn',28), ('lstm',29)):
        model_id = jobs[family]['results'][0]['model_id']
        contract = read_json(root / f'artifacts/manifests/phase{phase}/runs' / model_id / 'model_contract.json')
        print(family, contract['composition_plain'], 'parameters=',contract['parameter_count'])
        assert sum(layer['kind'] in {'rnn','lstm','dense'} for layer in contract['layers']) == 6

In [ ]:
if RUN_EXPERIMENTS:
    audit = read_json(root / 'artifacts/manifests/phase29/test_verification.json')
    print('Saved TEST audit:', audit['status'], audit['test_prediction_rows'])
    assert audit['status'] == 'PASS'
    assert not any(name in sys.modules for name in ('torch','tensorflow','keras'))
    print('No TEST forward or training was executed by this notebook.')

## Thực hành: ch4_gold_pink_sheet

Code và output lịch sử được giữ để đối chiếu. `RUN_EXPERIMENTS=False` nên không tự tạo split/train/TEST hoặc ghi lại kết quả đã khóa.

# RNN–LSTM / ch4_gold_pink_sheet

Hai họ ×ba checkpoint train độc lập. Serving thực tế NumPy CPU; không train hoặc mở TEST. Mẫu TRAIN, seed42 cố định; không chọn model bằng TEST. CLI/notebook chưa phải HTTP/React/Render live, không khuyến nghị đầu tư.

In [ ]:
if RUN_EXPERIMENTS:
    dataset = 'ch4_gold_pink_sheet'
    jobs = {}
    for family in ('rnn', 'lstm'):
        request, sample_id = sample_request(dataset, family)
        jobs[family] = infer(request)
        print('TRAIN sample:', sample_id, family)
        for row in jobs[family]['results']:
            print(row['implementation'], row['actual_backend'], row['actual_device'], row['prediction'])
        assert jobs[family]['comparison_valid']
    assert jobs['rnn']['input_sha256'] == jobs['lstm']['input_sha256']

## Thực hành: ch4_household_power

Code và output lịch sử được giữ để đối chiếu. `RUN_EXPERIMENTS=False` nên không tự tạo split/train/TEST hoặc ghi lại kết quả đã khóa.

# RNN–LSTM / ch4_household_power

Hai họ ×ba checkpoint train độc lập. Serving thực tế NumPy CPU; không train hoặc mở TEST. Mẫu TRAIN, seed42 cố định; không chọn model bằng TEST. CLI/notebook chưa phải HTTP/React/Render live, không khuyến nghị đầu tư.

In [ ]:
if RUN_EXPERIMENTS:
    dataset = 'ch4_household_power'
    jobs = {}
    for family in ('rnn', 'lstm'):
        request, sample_id = sample_request(dataset, family)
        jobs[family] = infer(request)
        print('TRAIN sample:', sample_id, family)
        for row in jobs[family]['results']:
            print(row['implementation'], row['actual_backend'], row['actual_device'], row['prediction'])
        assert jobs[family]['comparison_valid']
    assert jobs['rnn']['input_sha256'] == jobs['lstm']['input_sha256']

In [ ]:
import inspect
_notebook_defined = [(name, key) for name, mod in INLINE_MODULES.items() for key, obj in vars(mod).items() if inspect.isfunction(obj) and obj.__module__ == name]
assert all(vars(INLINE_MODULES[name])[key].__code__.co_filename.startswith("<notebook>/") for name,key in _notebook_defined)
print({"module có code ngay trong notebook": sum(bool(getattr(m,"__notebook_source__",False)) for m in INLINE_MODULES.values()), "hàm inline":len(_notebook_defined), "train/TEST đã chạy":False})
